# Synaptic Bouton Analysis Pipeline


This notebook walks through the post-processing workflow used to characterize glutamatergic parallel fiber boutons.
Each chapter tracks a distinct analytical theme, from data curation to mutant comparisons and stability assays.


## Chapter A – Data Foundations

Chapter A assembles the datasets, cleans fluorescence traces, and prepares pooled tables that will feed every later analysis.


### A.1 Library Imports

This cell assembles the analytical toolbox needed for the bouton study. Core scientific libraries such as NumPy, pandas, SciPy, and scikit-learn support numerical modeling, clustering, and statistical testing, while Matplotlib and Seaborn provide the visualization backbone. By centralizing these imports we ensure every downstream analysis stage—trace preprocessing, dimensionality reduction, and classification—can access the same well-defined computational environment.


In [ ]:
## Standard library imports. PCA, Clustering, Ellipse/boundary, Alpha-shape, k-NN boundary, Stability/plasticity

# Standard library imports
import json
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

# Scientific computing and data analysis
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage, set_link_color_palette
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Data visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import ListedColormap, to_hex
from matplotlib.cm import Set1
from matplotlib.patches import Ellipse
from matplotlib.widgets import Button

# Geometric and statistical analysis tools
import alphashape
from shapely.geometry import MultiPolygon, Point, Polygon as ShapelyPolygon
from shapely.affinity import scale as shp_scale
from shapely.prepared import prep
from statannotations.Annotator import Annotator

### A.2 Data Source Configuration

Here we define the file system layout for the complete dataset, including the Excel workbooks that store PCA features, per-trial failure metrics, and target cell annotations. Establishing these paths ensures that subsequent routines retrieve raw traces and metadata consistently, keeping the analysis reproducible regardless of where the notebook is executed.


In [ ]:
## File paths and directory structure. Failures/reliability, PCA, Clustering, Extracellular Ca²⁺, Stability/plasticity, Temporal traces

# File paths and directory structure
BASE_DIR            = Path(r'C:\Users\Anthime.PERROT\PPR_DATA_AND_CODE\Stability_After_temp_t_delete_later')  # Base directory for data and code

PPR_FILENAME        = 'summary.xlsx'           # PCA features file
PPR_TRIALS_FILENAME = 'summary_trials.xlsx'  # Per-trial failure data
TARGET_MAP_FILENAME = 'Target_WT_pooled.xlsx'  # Bouton target identity mapping
OUTPUT_DIR          = BASE_DIR / 'output'

# Data filtering and analysis parameters
EXCEPTIONAL_CONDITIONS = ['Stability_After_05', 'Stability_Before_05', 'Theo_1_5Ca', 'Theo_4Ca', 'WT_Theo', 'Theo_1_5_50Hz', 'Theo_4_50Hz', 'Theo_2_5_50Hz']  # Conditions to exclude from PCA and clustering
PCA_DROP_COLS          = [f'AMP{i}' for i in range(3, 11)] + ['measurement', 'Condition', 'ID', 'Target', '%Fail3']
N_CLUSTERS             = 5                          # Number of clusters for analysis

# Trace processing parameters
STIM_SHIFT  = 0.5                        # Stimulus time offset (seconds)
CROP_END    = 2.0                          # Trace duration to keep (seconds)
SAMPLE_RATE = 1000                      # Target sampling rate (Hz)
N_SAMPLES   = int(CROP_END * SAMPLE_RATE) + 1
COMMON_TIME = np.linspace(0, CROP_END, N_SAMPLES)  # Standardized time vector

### A.3 Output Logistics and Helpers

We next organize the raw recordings and metadata that feed the bouton analysis pipeline, ensuring every condition is accounted for before processing.

This block creates the output directory structure and introduces helper utilities for filtering valid trace files. By curating which spreadsheets qualify as bouton recordings, we avoid ingesting metadata or temporary files and maintain a clean provenance for every trace that enters the processing workflow.


In [ ]:
# Create output directory and define helper functions
OUTPUT_DIR.mkdir(exist_ok=True)

def is_bouton_file(file_path: Path) -> bool:
    """Check if Excel file contains bouton trace data (excludes metadata files)"""
    metadata_files = {PPR_FILENAME.lower(), PPR_TRIALS_FILENAME.lower(), TARGET_MAP_FILENAME.lower()}
    return (file_path.suffix.lower() == '.xlsx' and 
            not file_path.name.startswith('~$') and 
            file_path.name.lower() not in metadata_files)

def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')

### A.4 Experimental Inventory

The loop catalogues each experimental condition present in the raw data repository and enumerates the bouton trace files found within. This systematic survey provides immediate feedback on data availability and builds the foundation for condition-specific preprocessing that follows.


In [ ]:
## Discover experimental conditions and load bouton trace files. Ellipse/boundary, Temporal traces
## Now loading individual trials as in extract_metrics.py

experimental_conditions = [dir_path.name for dir_path in sorted(BASE_DIR.iterdir()) if dir_path.is_dir()]
raw_traces_data         = []

for condition_name in experimental_conditions:
    condition_dir = BASE_DIR / condition_name
    bouton_files  = [file_path for file_path in condition_dir.glob('*.xlsx') if is_bouton_file(file_path)]
    print(f"Processing {condition_name}: {len(bouton_files)} files")
    
    for xlsx_file in sorted(bouton_files):
        bouton_id  = clean_bouton_id(xlsx_file.stem)
        trace_data = pd.read_excel(xlsx_file, sheet_name=0).apply(pd.to_numeric, errors='coerce')
        
        if trace_data.shape[1] >= 2:  # Need at least time and one trial column
            # Time is in the last column (as in extract_metrics.py / demo_single_file.py)
            time_column = trace_data.columns[-1]
            time_array  = trace_data[time_column].to_numpy(float)
            
            # Individual trials: all columns except the last (time)
            # The second-to-last column is typically the average, but we load all
            trial_columns = trace_data.columns[:-1]
            trials_array  = trace_data[trial_columns].to_numpy(float)
            
            # Filter valid time points (finite values)
            valid_mask = np.isfinite(time_array)
            time_array = time_array[valid_mask]
            trials_array = trials_array[valid_mask, :]
            
            # Compute average from individual trials (excluding columns that might be averages)
            # If second-to-last column has a name like 'Average', 'Avg', 'Mean', use it
            # Otherwise compute from all trial columns
            avg_col_name = str(trial_columns[-1]).strip().lower()
            if avg_col_name in ['average', 'avg', 'mean', 'moyenne']:
                avg_trace = trials_array[:, -1]
                individual_trials = trials_array[:, :-1]  # Exclude average column
            else:
                individual_trials = trials_array
                avg_trace = np.nanmean(individual_trials, axis=1)
            
            raw_traces_data.append({
                'ID': bouton_id,
                'Condition': condition_name,
                'Time': time_array.tolist(),
                'Trials': individual_trials.tolist(),  # Individual trials (2D: samples x trials)
                'Avg': avg_trace.tolist(),  # Average trace for backward compatibility
                'n_trials': individual_trials.shape[1],
                'FilePath': str(xlsx_file),
            })

RAW_TRACES_DF = pd.DataFrame(raw_traces_data)
print(f"\n=== Loaded {len(RAW_TRACES_DF)} bouton files ===")
if len(RAW_TRACES_DF) > 0:
    print(f"Trials per bouton: min={RAW_TRACES_DF['n_trials'].min()}, max={RAW_TRACES_DF['n_trials'].max()}, mean={RAW_TRACES_DF['n_trials'].mean():.1f}")

### A.4b Preprocessing Functions (aligned with extract_metrics.py)

These preprocessing functions mirror the approach used in `Feature_extraction/extract_metrics.py`:
1. **NaN interpolation**: Fill missing values using time-wise interpolation
2. **Bleach correction** (optional): Remove slow exponential decay from photobleaching  
3. **ΔF/F₀ normalization**: Convert raw fluorescence to relative change using pre-stimulus baseline
4. **Per-trial processing**: Each trial is preprocessed independently before averaging

In [ ]:
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
import traceback

# Time constant for common time base
SAMPLE_RATE = 1000
N_SAMPLES = int(SAMPLE_RATE * 2.5)  # 2.5 seconds at 1000 Hz
COMMON_TIME = np.linspace(0, 2.5, N_SAMPLES)

# Conditions that need time shift
STIM_SHIFT = 0.5  # 500ms shift for exceptional conditions

# Crop parameters
CROP_END = 2.1  # seconds - aligned with extract_metrics.py FINAL_CROP_S


def fill_nans_timewise(yf, time):
    """Fill NaN values using simple linear interpolation."""
    nans = np.isnan(yf) | ~np.isfinite(yf)
    if np.all(nans):
        return yf
    if not np.any(nans):
        return yf
    valid_idx = np.flatnonzero(~nans)
    nan_idx = np.flatnonzero(nans)
    if valid_idx.size < 2:
        yf[nans] = np.nanmean(yf[~nans])
        return yf
    yf[nan_idx] = np.interp(time[nan_idx], time[valid_idx], yf[valid_idx])
    return yf


def sg_smooth(y, window, poly):
    """Savitzky-Golay smoothing. Mirrors smoothing.py::sg_smooth."""
    w = int(window)
    if w % 2 == 0:
        w += 1
    wmin = poly + 2 if ((poly + 2) % 2 == 1) else poly + 3
    w = max(w, wmin)
    w = min(w, len(y) - 1 if (len(y) % 2 == 0) else len(y))
    if w < 3:
        return y.copy()
    return savgol_filter(y, w, poly)


def _fit_biexp_robust(t, y, mask, n_iter=4, huber_delta=3.0, tau_range_factor=(0.25, 4.0), n_tau=15):
    """Robust bi-exponential fit for bleach correction.
    
    Fits: A + B * exp(-t / tau1) + C * exp(-t / tau2)
    where tau1 < tau2 (fast and slow components).
    """
    tt = t[mask]
    yy = y[mask]
    if tt.size < 5:
        return np.zeros_like(t), 1.0, 1.0
    
    # Estimate tau range from data span
    span = float(tt[-1] - tt[0])
    tau_lo, tau_hi = tau_range_factor
    tau_grid = np.linspace(tau_lo * span, tau_hi * span, int(n_tau))
    
    # Find best tau pair via grid search with Huber weighting
    best_tau1 = tau_grid[0]
    best_tau2 = tau_grid[-1]
    best_cost = np.inf
    w = np.ones(tt.size)
    
    for _ in range(n_iter):
        for i, tau1 in enumerate(tau_grid):
            if tau1 <= 0:
                continue
            for tau2 in tau_grid[i+1:]:  # tau2 > tau1
                if tau2 <= 0:
                    continue
                # Design matrix: [1, exp(-t/tau1), exp(-t/tau2)]
                X = np.column_stack([
                    np.ones_like(tt),
                    np.exp(-tt / tau1),
                    np.exp(-tt / tau2)
                ])
                # Weighted least squares
                Xw = X * w[:, None]
                yw = yy * w
                try:
                    coef = np.linalg.lstsq(Xw, yw, rcond=None)[0]
                    r = yy - X @ coef
                    cost = np.sum(w * r**2)
                    if cost < best_cost:
                        best_cost = cost
                        best_tau1 = tau1
                        best_tau2 = tau2
                except:
                    continue
        
        # Update Huber weights
        X = np.column_stack([
            np.ones_like(tt),
            np.exp(-tt / best_tau1),
            np.exp(-tt / best_tau2)
        ])
        try:
            coef = np.linalg.lstsq(X, yy, rcond=None)[0]
            r = yy - X @ coef
            absr = np.abs(r)
            w = np.where(absr <= huber_delta, 1.0, huber_delta / np.maximum(absr, 1e-12))
        except:
            pass
    
    # Final fit
    X = np.column_stack([
        np.ones_like(tt),
        np.exp(-tt / best_tau1),
        np.exp(-tt / best_tau2)
    ])
    Xw = X * w[:, None]
    yw = yy * w
    coef = np.linalg.lstsq(Xw, yw, rcond=None)[0]
    A, B, C = coef[0], coef[1], coef[2]
    
    # Trend over full time vector
    trend = A + B * np.exp(-t / best_tau1) + C * np.exp(-t / best_tau2)
    return trend, best_tau1, best_tau2


def compute_no_signal_mask(time, y_series, stim_times, post_zoom_s, peak_win_ms=25.0):
    """Mask regions without stimulus-evoked signal.
    
    Simplified version of smoothing.py::compute_no_signal_mask.
    """
    try:
        # Find peak times around each stimulus
        peak_times = []
        for st in np.atleast_1d(stim_times):
            # Simple peak detection in window after stimulus
            win_start = st
            win_end = st + peak_win_ms / 1000.0
            mask = (time >= win_start) & (time <= win_end)
            if np.any(mask):
                idx = np.argmax(y_series[mask])
                peak_times.append(time[mask][idx])
        
        last_peak = np.nanmax(peak_times) if peak_times else np.nan
        pre_mask = time < stim_times[0]
        
        if np.isfinite(last_peak):
            post_mask = time >= (last_peak + post_zoom_s)
        else:
            post_mask = np.zeros_like(time, dtype=bool)
        
        mask = pre_mask | post_mask
        if not np.any(mask):
            mask = pre_mask
        return mask
    except Exception:
        return time < stim_times[0]


def apply_bleach_correction_aligned(
    t: np.ndarray,
    y: np.ndarray,
    train_start_s: float,
    stim_times: np.ndarray,
    *,
    post_zoom_s: float = 0.20,
    peak_window_ms: float = 12.0,
    huber_delta: float = 3.0,
    tau_range_factor: tuple = (0.25, 4.0),
    n_tau: int = 15,
) -> np.ndarray:
    """Correct slow bleaching using robust bi-exponential fit.
    
    Uses pre-train baseline and post-train quiet region for fitting,
    avoiding stimulus-evoked epochs.
    """
    try:
        # Compute quiet mask (no evoked signal)
        mask_quiet = compute_no_signal_mask(
            t, y, stim_times, post_zoom_s, peak_win_ms=peak_window_ms
        )
        
        # Robust bi-exponential fit
        trend, tau1, tau2 = _fit_biexp_robust(
            t, y, mask_quiet, n_iter=4,
            huber_delta=huber_delta,
            tau_range_factor=tau_range_factor,
            n_tau=n_tau
        )
        
        # Baseline reference from pre-train period
        baseline_sel = y[(t < train_start_s) & np.isfinite(y)]
        if baseline_sel.size == 0:
            baseline_sel = y[np.isfinite(y)]
        baseline_ref = float(np.nanmedian(baseline_sel)) if baseline_sel.size else 0.0
        
        # Correct by subtracting trend and restoring baseline
        corrected = y - trend + baseline_ref
        return corrected
    except Exception:
        return y.copy()


print("✓ Preprocessing functions loaded (biexponential bleach correction)")

### A.5 Feature Matrix Assembly

This cell loads the multi-sheet Excel workbook containing precomputed bouton features and aligns them with the discovered experimental conditions. By harmonizing the feature matrices across conditions, we prepare a coherent dataset that can support pooled analyses as well as condition-specific comparisons.


In [ ]:
## Load PCA features from multi-sheet Excel file. AMP1/strength, PCA, Feature correlations

# Load PCA features from multi-sheet Excel file
pca_features_file = BASE_DIR / PPR_FILENAME
excel_data        = pd.ExcelFile(pca_features_file)

# Only process conditions that have corresponding feature sheets
available_conditions = [cond for cond in experimental_conditions if cond in excel_data.sheet_names]
CONDITIONS           = available_conditions

feature_dataframes = []
for condition_name in CONDITIONS:
    condition_features = pd.read_excel(pca_features_file, sheet_name=condition_name)
    
    # Standardize ID column name (handle various naming conventions)
    id_column_names = ['id', 'bouton', 'bouton_id', 'name']
    for column in condition_features.columns:
        if str(column).strip().lower() in id_column_names:
            condition_features = condition_features.rename(columns={column: 'ID'})
            break
    
    # Clean bouton IDs and add condition label
    condition_features['ID'] = condition_features['ID'].apply(
        lambda x: clean_bouton_id(str(x)) if pd.notnull(x) else x
    )
    condition_features['Condition'] = condition_name
    feature_dataframes.append(condition_features)

excel_data.close()
FEATURES_DATAFRAME = pd.concat(feature_dataframes, ignore_index=True)

# Identify and drop boutons with missing critical amplitude/PPR values
def _is_missing_critical(value):
    if isinstance(value, str):
        return value.strip() == ''
    return pd.isna(value)


def _is_critical_column(column_name: str) -> bool:
    column_name = str(column_name)
    if column_name.startswith('AMP') and column_name[3:].isdigit():
        return True
    if column_name.startswith('PPR') and '/1' in column_name:
        numerator = column_name[3:].split('/')[0]
        return numerator.isdigit()
    return False


critical_feature_columns = [
    col for col in FEATURES_DATAFRAME.columns if _is_critical_column(col)
]
invalid_feature_indices = []
invalid_feature_ids = set()

for row_idx, feature_row in FEATURES_DATAFRAME.iterrows():
    missing_columns = [
        col for col in critical_feature_columns
        if _is_missing_critical(feature_row.get(col))
    ]
    if missing_columns:
        raw_id = feature_row.get("ID")
        bouton_id = (
            clean_bouton_id(str(raw_id))
            if pd.notnull(raw_id) else f"row_{row_idx}"
        )
        condition_label = feature_row.get("Condition", "Unknown")
        print(
            f"! Skipping ID {bouton_id} (Condition {condition_label}) "
            f"due to missing values in {', '.join(missing_columns)}"
        )
        invalid_feature_indices.append(row_idx)
        if pd.notnull(raw_id):
            invalid_feature_ids.add(bouton_id)

if invalid_feature_indices:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(
        index=invalid_feature_indices
    ).reset_index(drop=True)

    if invalid_feature_ids and "RAW_TRACES_DF" in globals():
        before_trace_count = len(RAW_TRACES_DF)
        RAW_TRACES_DF = (
            RAW_TRACES_DF[~RAW_TRACES_DF['ID'].isin(invalid_feature_ids)]
            .reset_index(drop=True)
        )
        dropped_traces = before_trace_count - len(RAW_TRACES_DF)
        if dropped_traces:
            print(f"! Ignoring {dropped_traces} raw trace files linked to invalid IDs")

        if "raw_traces_data" in globals():
            kept_ids = set(RAW_TRACES_DF['ID'])
            raw_traces_data = [
                entry for entry in raw_traces_data
                if entry['ID'] in kept_ids
            ]

    elif invalid_feature_ids and "raw_traces_data" in globals():
        before_trace_count = len(raw_traces_data)
        raw_traces_data = [
            entry for entry in raw_traces_data
            if entry['ID'] not in invalid_feature_ids
        ]
        dropped_traces = before_trace_count - len(raw_traces_data)
        if dropped_traces:
            print(f"! Ignoring {dropped_traces} raw trace files linked to invalid IDs")
        RAW_TRACES_DF = pd.DataFrame(raw_traces_data)



# Remove specified amplitude columns from analysis
amplitude_columns_to_drop = [col for col in PCA_DROP_COLS[:8] if col in FEATURES_DATAFRAME.columns]
if amplitude_columns_to_drop:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(columns=amplitude_columns_to_drop)

In [ ]:
# Adjust problematic recordings manually

# Check if manual 50Hz fix file exists
manual_fix_file = BASE_DIR / 'Manual_50Hz_fix_last_event.xlsx'

if manual_fix_file.exists():
    print(f"Loading manual 50Hz fixes from {manual_fix_file.name}...")
    
    # Load the manual fix file
    manual_fixes = pd.read_excel(manual_fix_file)
    
    # Column E contains IDs (0-indexed: column index 4), Column D contains corrected AMP10 (index 3)
    # Assuming column E is the 5th column (index 4) and column D is the 4th column (index 3)
    fixes_df = manual_fixes.iloc[:, [4, 3]].copy()
    fixes_df.columns = ['ID', 'AMP10_corrected']
    
    # Remove any NaN rows
    fixes_df = fixes_df.dropna()
    
    # Convert IDs to string and strip whitespace for matching
    fixes_df['ID'] = fixes_df['ID'].astype(str).str.strip()
    
    # Track updates
    updated_count = 0
    
    # Apply corrections to FEATURES_DATAFRAME
    for idx, fix_row in fixes_df.iterrows():
        fix_id = fix_row['ID']
        new_amp10 = fix_row['AMP10_corrected']
        
        # Find matching rows in FEATURES_DATAFRAME
        mask = FEATURES_DATAFRAME['ID'] == fix_id
        
        if mask.any():
            # Update AMP10
            FEATURES_DATAFRAME.loc[mask, 'AMP10'] = new_amp10
            
            # Recalculate PPR10/1 = AMP10 / AMP1
            amp1_value = FEATURES_DATAFRAME.loc[mask, 'AMP1'].values[0]
            if amp1_value != 0:
                new_ppr10_1 = new_amp10 / amp1_value
                FEATURES_DATAFRAME.loc[mask, 'PPR10/1'] = new_ppr10_1
                
                updated_count += 1
                print(f"  ✓ Updated ID {fix_id}: AMP10={new_amp10:.4f}, PPR10/1={new_ppr10_1:.4f}")
            else:
                print(f"  ⚠ Warning: ID {fix_id} has AMP1=0, cannot calculate PPR10/1")
        else:
            print(f"  ✗ Warning: ID {fix_id} not found in FEATURES_DATAFRAME")
    
    print(f"\nManual corrections applied: {updated_count}/{len(fixes_df)} recordings updated")
else:
    print(f"No manual fix file found at {manual_fix_file.name}, skipping manual corrections")


### A.6 Target Identity Integration

Target identity information (Purkinje cell, interneuron, or unclassified) is imported and standardized in this step. Cleaning identifiers and storing them as categorical labels allows later projections and clustering analyses to be biologically interpretable, connecting statistical patterns back to synaptic targets.


In [ ]:
## Load bouton target identity mapping (PC/IN/UN classification). Extracellular Ca²⁺, Temporal traces

# Load bouton target identity mapping (PC/IN/UN classification)
target_mapping_file          = BASE_DIR / TARGET_MAP_FILENAME
target_identity_data         = pd.read_excel(target_mapping_file).iloc[:, :2].copy()
target_identity_data.columns = ['ID', 'Target']

# Clean and standardize target data
target_identity_data['ID']     = target_identity_data['ID'].apply(lambda x: clean_bouton_id(str(x).strip()))
target_identity_data['Target'] = target_identity_data['Target'].astype(str).str.strip().str.upper()

# Set invalid targets to 'UN' (undefined)
valid_targets = ['PC', 'IN', 'UN']
target_identity_data['Target'] = target_identity_data['Target'].where(
    target_identity_data['Target'].isin(valid_targets), 'UN'
)

# Merge target identities into feature data
FEATURES_DATAFRAME['ID']     = FEATURES_DATAFRAME['ID'].astype(str).str.strip()
FEATURES_DATAFRAME           = FEATURES_DATAFRAME.merge(target_identity_data, on='ID', how='left')
FEATURES_DATAFRAME['Target'] = FEATURES_DATAFRAME['Target'].fillna('UN')  # Missing targets → undefined

# Data loading summary
print(f"\n=== DATA LOADING SUMMARY ===")
print(f"Conditions: {len(CONDITIONS)} ({', '.join(CONDITIONS)})")
print(f"Traces: {len(RAW_TRACES_DF)} | Features: {len(FEATURES_DATAFRAME)} | Target mappings: {len(target_identity_data)}")
print(f"Target distribution: {dict(FEATURES_DATAFRAME['Target'].value_counts())}")
print(f"✓ Successfully processed {len(raw_traces_data)} bouton files")

### A.7 Photobleaching and Normalization Utilities

Before modeling, we align, pad, and normalize fluorescence traces so that recordings collected with different acquisition schemes can be compared on equal footing.

This section defines the signal-processing toolkit for fluorescence traces. Functions handle photobleaching correction, baseline stabilization, exponential fitting, and smoothing, ensuring that raw optical signals are transformed into comparable, biologically meaningful time courses before higher-level analysis.


In [ ]:
## PREPROCESSING FUNCTIONS (Back.ipynb pipeline)

def recenter_baseline(trace: np.ndarray, time: np.ndarray, baseline_end: float = 1.0, baseline_start: float = 0.5) -> np.ndarray:
    """Re-center trace so that the median of the baseline region is exactly 0.
    
    Uses a robust baseline window [baseline_start, baseline_end) to compute
    the median offset and subtracts it from the entire trace.
    
    Args:
        trace: Signal array to re-center
        time: Time array corresponding to trace
        baseline_end: End of baseline window (exclusive, default 1.0s = stim onset)
        baseline_start: Start of baseline window (default 0.5s to skip NaN padding)
    
    Returns:
        Re-centered trace with baseline median = 0
    """
    if trace is None or len(trace) == 0:
        return trace
    baseline_mask = (time >= baseline_start) & (time < baseline_end)
    baseline_values = trace[baseline_mask]
    baseline_values = baseline_values[np.isfinite(baseline_values)]
    if baseline_values.size > 0:
        offset = np.nanmedian(baseline_values)
        return trace - offset
    return trace

# Savitzky-Golay smoothing function
def sg_smooth(y: np.ndarray, window_length: int = 9, polyorder: int = 2) -> np.ndarray:
    """Apply Savitzky-Golay filter, handling NaN gracefully."""
    from scipy.signal import savgol_filter
    if np.all(np.isnan(y)):
        return y.copy()
    valid = np.isfinite(y)
    if valid.sum() < max(window_length, polyorder + 2):
        return y.copy()
    result = y.copy()
    result[valid] = savgol_filter(y[valid], min(window_length, valid.sum() // 2 * 2 - 1), polyorder)
    return result


def fill_nans_timewise(y: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Fill NaN gaps by linear interpolation in time domain."""
    out = y.copy()
    finite_mask = np.isfinite(out)
    if finite_mask.sum() < 2:
        return out
    out[~finite_mask] = np.interp(t[~finite_mask], t[finite_mask], out[finite_mask])
    return out


def correct_photobleaching(
    signal: np.ndarray,
    time: np.ndarray,
    stim_window: tuple = (1.0, 1.5),
    ref_points: int = 20
) -> np.ndarray:
    """Correct for photobleaching using bi-exponential decay model.
    
    This function fits a bi-exponential decay to the signal, excluding
    the stimulus window, then subtracts the fitted trend.
    """
    from scipy.optimize import curve_fit
    
    stim_start, stim_end = stim_window
    
    # Create mask excluding stimulus window
    mask = (time < stim_start) | (time > stim_end)
    t_fit = time[mask]
    y_fit_raw = signal[mask]
    
    # Handle NaN values
    valid = np.isfinite(y_fit_raw)
    if valid.sum() < 5:
        return signal.copy()
    
    t_fit = t_fit[valid]
    y_fit_raw = y_fit_raw[valid]
    
    # Baseline is 10th percentile of ALL valid data
    baseline = np.nanpercentile(y_fit_raw, 10)
    
    # Bi-exponential decay model
    def biexp_decay(t, A1, k1, A2, k2):
        return A1 * np.exp(-k1 * t) + A2 * np.exp(-k2 * t)
    
    # Shift to baseline for fitting
    y_fit = y_fit_raw - baseline
    
    try:
        # Initial guesses based on data range and typical time constants
        y_range = np.ptp(y_fit)
        t_range = np.ptp(t_fit)
        
        p0 = [y_range * 0.5, 0.5 / t_range, y_range * 0.5, 2.0 / t_range]
        
        bounds = ([0, 0.01, 0, 0.01], [np.inf, 10, np.inf, 10])
        
        popt, _ = curve_fit(
            biexp_decay, t_fit, y_fit,
            p0=p0, bounds=bounds, maxfev=5000
        )
        
        A1_fit, k1_fit, A2_fit, k2_fit = popt
        
        # Compute trend over full time vector
        trend = baseline + biexp_decay(time, A1_fit, k1_fit, A2_fit, k2_fit)
        
        # Subtraction correction: subtract decay trend, normalized to stimulus start
        trend_at_stim = baseline + biexp_decay(stim_start, A1_fit, k1_fit, A2_fit, k2_fit)
        corrected = signal - (trend - trend_at_stim)
        
        return corrected
    
    except (RuntimeError, ValueError):
        # If fitting fails, return original signal
        return signal.copy()


# ========================
# ROBUST BI-EXPONENTIAL BLEACH CORRECTION (aligned with iglusnfr_optimized)
# ========================

def _fit_biexp_robust(t, y, mask, n_iter=4, huber_delta=3.0, tau_range_factor=(0.25, 4.0), n_tau=15):
    """Robust bi-exponential fit using Huber weights (simplified).
    
    Based on smoothing.py but streamlined for notebook use.
    """
    tt = t[mask]
    yy = y[mask]
    
    if len(tt) < 5:
        # Fall back to mean-based correction
        return np.full_like(t, np.nanmean(y)), 1.0, 1.0
    
    # Duration-based tau guesses
    duration = tt.max() - tt.min()
    tau_fast = duration * 0.1
    tau_slow = duration * 0.5
    
    # Grid search for tau values
    tau_min = min(tau_fast, tau_slow) * tau_range_factor[0]
    tau_max = max(tau_fast, tau_slow) * tau_range_factor[1]
    tau_candidates = np.logspace(np.log10(tau_min), np.log10(tau_max), n_tau)
    
    best_tau1, best_tau2 = tau_fast, tau_slow
    best_cost = np.inf
    
    # Initialize Huber weights
    w = np.ones_like(yy)
    
    for _ in range(n_iter):
        # Grid search
        for tau1 in tau_candidates:
            for tau2 in tau_candidates:
                if tau2 <= tau1:
                    continue
                X = np.column_stack([
                    np.ones_like(tt),
                    np.exp(-tt / tau1),
                    np.exp(-tt / tau2)
                ])
                Xw = X * w[:, None]
                yw = yy * w
                try:
                    coef = np.linalg.lstsq(Xw, yw, rcond=None)[0]
                    r = yy - X @ coef
                    cost = np.sum(w * r**2)
                    if cost < best_cost:
                        best_cost = cost
                        best_tau1 = tau1
                        best_tau2 = tau2
                except:
                    continue
        
        # Update Huber weights
        X = np.column_stack([
            np.ones_like(tt),
            np.exp(-tt / best_tau1),
            np.exp(-tt / best_tau2)
        ])
        try:
            coef = np.linalg.lstsq(X, yy, rcond=None)[0]
            r = yy - X @ coef
            absr = np.abs(r)
            w = np.where(absr <= huber_delta, 1.0, huber_delta / np.maximum(absr, 1e-12))
        except:
            pass
    
    # Final fit
    X = np.column_stack([
        np.ones_like(tt),
        np.exp(-tt / best_tau1),
        np.exp(-tt / best_tau2)
    ])
    Xw = X * w[:, None]
    yw = yy * w
    coef = np.linalg.lstsq(Xw, yw, rcond=None)[0]
    A, B, C = coef[0], coef[1], coef[2]
    
    # Trend over full time vector
    trend = A + B * np.exp(-t / best_tau1) + C * np.exp(-t / best_tau2)
    return trend, best_tau1, best_tau2


def compute_no_signal_mask(time, y_series, stim_times, post_zoom_s, peak_win_ms=25.0):
    """Mask regions without stimulus-evoked signal.
    
    Simplified version of smoothing.py::compute_no_signal_mask.
    """
    try:
        # Find peak times around each stimulus
        peak_times = []
        for st in np.atleast_1d(stim_times):
            # Simple peak detection in window after stimulus
            win_start = st
            win_end = st + peak_win_ms / 1000.0
            mask = (time >= win_start) & (time <= win_end)
            if np.any(mask):
                idx = np.argmax(y_series[mask])
                peak_times.append(time[mask][idx])
        
        last_peak = np.nanmax(peak_times) if peak_times else np.nan
        pre_mask = time < stim_times[0]
        
        if np.isfinite(last_peak):
            post_mask = time >= (last_peak + post_zoom_s)
        else:
            post_mask = np.zeros_like(time, dtype=bool)
        
        mask = pre_mask | post_mask
        if not np.any(mask):
            mask = pre_mask
        return mask
    except Exception:
        return time < stim_times[0]


def apply_bleach_correction_aligned(
    t: np.ndarray,
    y: np.ndarray,
    train_start_s: float,
    stim_times: np.ndarray,
    *,
    post_zoom_s: float = 0.20,
    peak_window_ms: float = 12.0,
    huber_delta: float = 3.0,
    tau_range_factor: tuple = (0.25, 4.0),
    n_tau: int = 15,
) -> np.ndarray:
    """Correct slow bleaching using robust bi-exponential fit.
    
    Uses pre-train baseline and post-train quiet region for fitting,
    avoiding stimulus-evoked epochs.
    """
    try:
        # Compute quiet mask (no evoked signal)
        mask_quiet = compute_no_signal_mask(
            t, y, stim_times, post_zoom_s, peak_win_ms=peak_window_ms
        )
        
        # Robust bi-exponential fit
        trend, tau1, tau2 = _fit_biexp_robust(
            t, y, mask_quiet, n_iter=4,
            huber_delta=huber_delta,
            tau_range_factor=tau_range_factor,
            n_tau=n_tau
        )
        
        # Baseline reference from pre-train period
        baseline_sel = y[(t < train_start_s) & np.isfinite(y)]
        if baseline_sel.size == 0:
            baseline_sel = y[np.isfinite(y)]
        baseline_ref = float(np.nanmedian(baseline_sel)) if baseline_sel.size else 0.0
        
        # Correct by subtracting trend and restoring baseline
        corrected = y - trend + baseline_ref
        return corrected
    except Exception:
        return y.copy()


# ========================
# PREPROCESSING PARAMETERS (aligned with iglusnfr_optimized preset)
# ========================
PREPROC_PARAMS = {
    'normalize_dff': True,
    'bleach': True,
    'sg_window': 9,
    'sg_poly': 2,
    'f0_eps': 1e-9,
    'post_zoom_s': 0.20,
    'peak_window_ms': 12.0,
    'bleach_huber_delta': 3.0,
    'bleach_tau_range_factor': (0.25, 4.0),
    'bleach_n_tau': 15,
}

# Per-condition ISI mapping (from demo_batch_process.py)
# 50Hz conditions use ISI=0.02, others use ISI=0.05
CONDITION_ISI = {
    'Theo_1_5_50Hz': 0.02,
    'Theo_2_5_50Hz': 0.02,
    'Theo_4_50Hz': 0.02,
}
DEFAULT_ISI = 0.05

# Per-condition train_start (from demo_batch_process.py)
# Exceptional conditions already shifted by 0.5s, so effective train_start is 1.0s
# But for bleach correction we use the actual stim times after shift
CONDITION_TRAIN_START = {}  # All use 1.0s after time alignment
DEFAULT_TRAIN_START = 1.0


def _resample_signal(original_time, original_signal, condition_name):
    """Resample a single signal to COMMON_TIME grid with time shift if needed.
    
    Handles two cases:
    1) EXCEPTIONAL_CONDITIONS: Data recorded with stim at 0.5s needs +0.5s shift
    2) Short baseline data: Data starting after t=0 (e.g., only 0.5s baseline) 
       needs NaN padding at the beginning to align stim to 1.0s
    """
    # Replace infinite values with NaN
    original_signal = np.where(np.isfinite(original_signal), original_signal, np.nan)
    
    # Apply time shift for specific experimental conditions
    if condition_name in EXCEPTIONAL_CONDITIONS:
        adjusted_time = original_time + STIM_SHIFT
        points_before_start = np.sum(COMMON_TIME < adjusted_time[0])
        if points_before_start > 0:
            padded_signal = np.concatenate([np.full(points_before_start, np.nan), original_signal])
            adjusted_time = np.concatenate([COMMON_TIME[:points_before_start], adjusted_time])
        else:
            padded_signal = original_signal
    else:
        adjusted_time = original_time
        padded_signal = original_signal
        
        # Check if data has short baseline (starts after t=0)
        # This handles cases where only 0.5s baseline exists (stim at 0.5s in original)
        # and we need to shift to align stim to 1.0s on COMMON_TIME
        first_time = float(adjusted_time[0]) if len(adjusted_time) > 0 else 0.0
        if first_time > 0.1:  # Data starts significantly after t=0
            # Shift time forward so that stim aligns to 1.0s
            # If data starts at ~0.5s with stim at 0.5s, shift by 0.5s
            adjusted_time = adjusted_time + (1.0 - first_time - 0.5)  # Assume stim was at 0.5s in original
            # Actually simpler: just shift so first_time maps to the missing baseline period
            # If original starts at 0.5s, we need to add 0.5s of NaN before
            adjusted_time = original_time + (1.0 - 0.5)  # Shift by 0.5s to align stim at 1.0s
            
            points_before_start = np.sum(COMMON_TIME < adjusted_time[0])
            if points_before_start > 0:
                padded_signal = np.concatenate([np.full(points_before_start, np.nan), original_signal])
                adjusted_time = np.concatenate([COMMON_TIME[:points_before_start], adjusted_time])
    
    # Resample to common time base
    valid_mask = np.isfinite(padded_signal) & np.isfinite(adjusted_time)
    if np.sum(valid_mask) < 2:
        resampled_signal = np.full(N_SAMPLES, np.nan)
    else:
        try:
            resampled_signal = np.interp(COMMON_TIME, adjusted_time[valid_mask], padded_signal[valid_mask])
        except ValueError:
            resampled_signal = np.full(N_SAMPLES, np.nan)
    
    # Mark points before original data start as NaN (don't extrapolate)
    if len(adjusted_time) > 0:
        first_valid_time = adjusted_time[0]
        resampled_signal[COMMON_TIME < first_valid_time] = np.nan
    
    # Ensure exact length
    if len(resampled_signal) < N_SAMPLES:
        padding_needed = N_SAMPLES - len(resampled_signal)
        resampled_signal = np.concatenate([resampled_signal, np.full(padding_needed, np.nan)])
    else:
        resampled_signal = resampled_signal[:N_SAMPLES]
    
    return resampled_signal


def _preprocess_single_trial(signal, condition_name):
    """Preprocess a single trial following back.ipynb pipeline.
    
    Steps:
      1) NaN interpolation via fill_nans_timewise
      2) Bleach correction via correct_photobleaching (bi-exponential curve_fit)
      3) ΔF/F0 using MEDIAN over full pre-train baseline
      4) Savitzky-Golay smoothing
      5) Baseline re-centering: subtract median of pre-stim baseline so baseline = 0
      
    ALIGNED WITH extract_metrics.py: When F0 is too small (< f0_eps), the trial
    is set to NaN (invalid) and will be excluded from averaging. This prevents
    division by near-zero values that create aberrant ΔF/F0 values.
    """
    # Step 1: Fill NaNs
    signal = fill_nans_timewise(signal, COMMON_TIME)
    
    # Step 2: Bleach correction using the old bi-exponential method (like back.ipynb)
    train_start = CONDITION_TRAIN_START.get(condition_name, DEFAULT_TRAIN_START)
    isi = CONDITION_ISI.get(condition_name, DEFAULT_ISI)
    
    # Compute stim_window end based on ISI and number of pulses
    n_pulses = 10
    stim_window_end = train_start + isi * (n_pulses - 1) + 0.1  # Add small buffer after last pulse
    
    if PREPROC_PARAMS['bleach']:
        signal = correct_photobleaching(
            signal, COMMON_TIME,
            stim_window=(train_start, stim_window_end),
            ref_points=20
        )
    
    # Step 3: ΔF/F0 using MEDIAN (aligned with extract_metrics.py)
    baseline_mask = COMMON_TIME < train_start
    baseline_values = signal[baseline_mask]
    baseline_values = baseline_values[np.isfinite(baseline_values)]
    F0 = np.nanmedian(baseline_values) if baseline_values.size else 0.0
    
    if PREPROC_PARAMS['normalize_dff']:
        # ALIGNED WITH extract_metrics.py: when F0 is too small, set to NaN
        # This mirrors the behavior in extract_metrics.py lines 1199-1206
        if np.abs(F0) < PREPROC_PARAMS['f0_eps']:
            # F0 too small → invalid trial, return NaN array
            delta_f_over_f = np.full_like(signal, np.nan)
        else:
            delta_f_over_f = (signal - F0) / F0
            # Additional safety: clip extreme ΔF/F0 values that could arise from
            # near-zero F0 values that passed the threshold
            # Typical iGluSnFR signals rarely exceed ±10 ΔF/F0
            max_dff = 20.0  # Conservative upper bound
            if np.nanmax(np.abs(delta_f_over_f)) > max_dff:
                # This trial has aberrant values - likely bad F0 estimation
                # Fall back to subtract-only mode (like extract_metrics.py bad_cols fallback)
                delta_f_over_f = signal - F0
    else:
        delta_f_over_f = signal - F0
    
    # Step 4: Savitzky-Golay smoothing (skip if all NaN)
    if np.any(np.isfinite(delta_f_over_f)):
        delta_f_over_f = sg_smooth(delta_f_over_f, PREPROC_PARAMS['sg_window'], PREPROC_PARAMS['sg_poly'])
    
    # Step 5: Re-center baseline to 0
    # After smoothing, the baseline may not be exactly 0. Subtract the median
    # of the pre-stimulus baseline to ensure it's centered at 0.
    if np.any(np.isfinite(delta_f_over_f)):
        baseline_after_smooth = delta_f_over_f[baseline_mask]
        baseline_after_smooth = baseline_after_smooth[np.isfinite(baseline_after_smooth)]
        if baseline_after_smooth.size > 0:
            baseline_offset = np.nanmedian(baseline_after_smooth)
            delta_f_over_f = delta_f_over_f - baseline_offset
    
    return delta_f_over_f, F0


def process_single_trace_aligned(trace_data, use_individual_trials=True) -> dict:
    """
    Process individual bouton trace using EXACT same pipeline as extract_metrics.py.
    
    Pipeline order (matches extract_metrics.py):
      1) Time alignment and resampling (per trial)
      2) NaN interpolation via fill_nans_timewise (per trial)
      3) Bleach correction via robust bi-exponential fit (per trial)
      4) ΔF/F0 using MEDIAN over full pre-train baseline (per trial)
      5) Savitzky-Golay smoothing (sg_window=9, sg_poly=2) (per trial)
      6) Baseline re-centering to 0 (per trial)
      7) Average across preprocessed trials
    
    Args:
        trace_data: Row from RAW_TRACES_DF containing 'Time', 'Trials' (or 'Avg')
        use_individual_trials: If True and 'Trials' is available, process each trial
                               individually then average (like extract_metrics.py).
                               If False, process only the average trace.
    """
    original_time = np.array(trace_data['Time'])
    condition_name = trace_data['Condition']
    
    # Check if individual trials are available
    has_trials = 'Trials' in trace_data and trace_data['Trials'] is not None
    
    if use_individual_trials and has_trials:
        # Process individual trials like extract_metrics.py
        trials_raw = np.array(trace_data['Trials'])
        if trials_raw.ndim == 1:
            trials_raw = trials_raw[:, None]
        
        n_trials = trials_raw.shape[1]
        processed_trials = []
        F0_values = []
        
        for j in range(n_trials):
            trial_signal = trials_raw[:, j]
            # Resample to common time
            resampled = _resample_signal(original_time, trial_signal, condition_name)
            # Preprocess
            processed, f0 = _preprocess_single_trial(resampled, condition_name)
            processed_trials.append(processed)
            F0_values.append(f0)
        
        # Stack and compute average of preprocessed trials
        processed_trials = np.column_stack(processed_trials)
        avg_processed = np.nanmean(processed_trials, axis=1)
        
        # Final baseline re-centering on the averaged trace
        # Even though each trial is baseline-corrected, the average may drift
        # due to NaN handling or trial-to-trial variability
        train_start = CONDITION_TRAIN_START.get(condition_name, DEFAULT_TRAIN_START)
        baseline_mask = COMMON_TIME < train_start
        baseline_avg = avg_processed[baseline_mask]
        baseline_avg = baseline_avg[np.isfinite(baseline_avg)]
        if baseline_avg.size > 0:
            avg_processed = avg_processed - np.nanmedian(baseline_avg)
        
        return {
            'ID': trace_data['ID'],
            'Condition': condition_name,
            'Time': COMMON_TIME,
            'Avg': avg_processed,
            'Trials': processed_trials,
            'F0': np.array(F0_values),
            'n_trials': n_trials,
        }
    else:
        # Fallback: process only the average trace
        original_signal = np.array(trace_data['Avg'])
        
        # Resample and preprocess
        resampled = _resample_signal(original_time, original_signal, condition_name)
        processed, F0 = _preprocess_single_trial(resampled, condition_name)
        
        return {
            'ID': trace_data['ID'],
            'Condition': condition_name,
            'Time': COMMON_TIME,
            'Avg': processed,
            'F0': F0,
        }


def process_single_trace(trace_data) -> dict:
    """Wrapper that redirects to aligned preprocessing."""
    return process_single_trace_aligned(trace_data)

print("✓ Preprocessing functions aligned with extract_metrics.py (iglusnfr_optimized preset)")
print(f"  - Individual trials: Each trial preprocessed separately, then averaged")
print(f"  - Bleach correction: Robust bi-exponential (Huber δ={PREPROC_PARAMS['bleach_huber_delta']})")
print(f"  - Baseline re-centering: Median of pre-stim baseline subtracted → baseline = 0")

### A.8 Trace Processing Pipeline

With the utilities in place, this code iterates through all boutons, applies bleaching correction, normalizes baselines, and stores metadata describing each trace. The result is a harmonized collection of time series that captures synaptic responses across experimental conditions while preserving the contextual information needed for downstream grouping.


In [ ]:
## Process all bouton traces and organize by condition. Temporal traces

all_processed_traces = []
traces_by_condition  = {}

for condition_name in RAW_TRACES_DF['Condition'].unique():
    condition_traces = RAW_TRACES_DF[RAW_TRACES_DF['Condition'] == condition_name]
    
    condition_all_processed_traces = []
    for _, single_trace in condition_traces.iterrows():
        processed_trace = process_single_trace(single_trace)
        all_processed_traces.append(processed_trace)
        condition_all_processed_traces.append(processed_trace['Avg'])
    
    traces_by_condition[condition_name] = condition_all_processed_traces
    print(f"Processed {len(condition_all_processed_traces)} traces for condition: {condition_name}")

print(f"✓ Processed {len(all_processed_traces)} traces across {len(traces_by_condition)} conditions")

### A.9 Condition-Level Trace Visualization

Processed traces are visualized for every condition to sanity-check preprocessing outcomes. Overlaying individual responses reveals whether normalization, alignment, and denoising preserved the stereotyped waveform dynamics expected for each experimental group.


In [ ]:
## Plot processed traces by condition. Extracellular Ca²⁺, Temporal traces

n_conditions = len(traces_by_condition)
fig, axes    = plt.subplots(2, 5, figsize=(20, 8))
axes         = axes.flatten()

for plot_idx, (condition_name, condition_traces) in enumerate(traces_by_condition.items()):
    if plot_idx >= len(axes):
        break
        
    ax = axes[plot_idx]
    
    # Plot individual traces (transparent gray)
    for single_trace in condition_traces:
        ax.plot(COMMON_TIME, single_trace, alpha=0.2, color='gray', linewidth=0.5)
    
    # Plot condition average (bold black)
    condition_average = np.nanmean(condition_traces, axis=0)
    ax.plot(COMMON_TIME, condition_average, color='black', linewidth=2, label='Average')
    
    # Format subplot
    ax.set_title(f'{condition_name}\n(n={len(condition_traces)})', fontsize=10, pad=10)
    ax.set_xlim(0, CROP_END)
    ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)  # Zero line
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, linewidth=1)  # Stimulus onset
    
    # Add axis labels for edge subplots
    if plot_idx >= 5:  # Bottom row
        ax.set_xlabel('Time (s)')
    if plot_idx % 5 == 0:  # Leftmost column
        ax.set_ylabel('ΔF/F')

# Remove unused subplots
for empty_idx in range(n_conditions, len(axes)):
    fig.delaxes(axes[empty_idx])

plt.tight_layout()
plt.suptitle('Processed Bouton Traces: Aligned, Resampled, and Normalized by Condition', 
             y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Create final processed traces dataframe
NORM_TRACES_DATAFRAME = pd.DataFrame(all_processed_traces)

print(f"Time range: {COMMON_TIME[0]:.2f} - {COMMON_TIME[-1]:.2f}s ({len(COMMON_TIME)} points)")
print(f"Exceptional conditions (shifted by {STIM_SHIFT}s): {', '.join(EXCEPTIONAL_CONDITIONS)}")

In [ ]:
# Plot mean traces for WT_Anthime, WT_Theo, WT_Theo_1scd, and SynII conditions

# Build lookup for processed traces
resampled_trace_lookup = {entry['ID']: entry for entry in all_processed_traces}

# Collect traces for each condition
conditions_to_plot = ['WT_Anthime', 'WT_Theo', 'WT_Theo_1scd', 'SynII']
condition_colors = {'WT_Anthime': '#2ca02c', 'WT_Theo': '#1f77b4', 'WT_Theo_1scd': '#ff7f0e', 'SynII': '#d62728'}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

condition_means = {}

# First pass: collect all data to determine shared y-axis limits
all_means = []
all_sems = []

for condition in conditions_to_plot:
    condition_traces = []
    for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == condition].iterrows():
        bouton_id = trace_row['ID']
        if bouton_id in resampled_trace_lookup:
            condition_traces.append(resampled_trace_lookup[bouton_id]['Avg'])
    
    if condition_traces:
        traces_matrix = np.column_stack(condition_traces)
        mean_trace = np.nanmean(traces_matrix, axis=1)
        sem_trace = np.nanstd(traces_matrix, axis=1, ddof=1) / np.sqrt(traces_matrix.shape[1])
        condition_means[condition] = mean_trace
        all_means.append(mean_trace)
        all_sems.append(sem_trace)

# Calculate global y-limits
all_upper = [m + s for m, s in zip(all_means, all_sems)]
all_lower = [m - s for m, s in zip(all_means, all_sems)]
y_min_global = np.nanmin([np.nanmin(l) for l in all_lower])
y_max_global = np.nanmax([np.nanmax(u) for u in all_upper])
y_margin = (y_max_global - y_min_global) * 0.05
y_lim = (y_min_global - y_margin, y_max_global + y_margin)

# Second pass: plot with shared limits
for idx, condition in enumerate(conditions_to_plot):
    ax = axes[idx]
    
    condition_traces = []
    for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == condition].iterrows():
        bouton_id = trace_row['ID']
        if bouton_id in resampled_trace_lookup:
            condition_traces.append(resampled_trace_lookup[bouton_id]['Avg'])
    
    if condition_traces:
        traces_matrix = np.column_stack(condition_traces)
        mean_trace = np.nanmean(traces_matrix, axis=1)
        sem_trace = np.nanstd(traces_matrix, axis=1, ddof=1) / np.sqrt(traces_matrix.shape[1])
        
        ax.plot(COMMON_TIME, mean_trace, color=condition_colors[condition], linewidth=2)
        ax.fill_between(COMMON_TIME, mean_trace - sem_trace, mean_trace + sem_trace,
                        color=condition_colors[condition], alpha=0.25)
    
    ax.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlim(0.5, 2.0)
    ax.set_ylim(y_lim)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ΔF/F')
    ax.set_title(f'{condition} (n={len(condition_traces)})')
    ax.grid(True, alpha=0.3)

# Fifth subplot: grand average of WT conditions only (excluding SynII)
ax_grand = axes[4]
wt_conditions = ['WT_Anthime', 'WT_Theo', 'WT_Theo_1scd']
wt_means = {k: v for k, v in condition_means.items() if k in wt_conditions}

if wt_means:
    all_wt_means = np.column_stack(list(wt_means.values()))
    grand_mean = np.nanmean(all_wt_means, axis=1)
    grand_sem = np.nanstd(all_wt_means, axis=1, ddof=1) / np.sqrt(all_wt_means.shape[1])
    
    ax_grand.plot(COMMON_TIME, grand_mean, color='black', linewidth=2)
    ax_grand.fill_between(COMMON_TIME, grand_mean - grand_sem, grand_mean + grand_sem,
                          color='gray', alpha=0.25)

ax_grand.axhline(0, color='gray', linestyle='dotted', linewidth=1)
ax_grand.axvline(1.0, color='red', linestyle='--', alpha=0.5)
ax_grand.set_xlim(0.5, 2.0)
ax_grand.set_ylim(y_lim)
ax_grand.set_xlabel('Time (s)')
ax_grand.set_ylabel('ΔF/F')
ax_grand.set_title('Grand Average (WT conditions)')
ax_grand.grid(True, alpha=0.3)

# Remove unused subplot
fig.delaxes(axes[5])

plt.tight_layout()
output_file = OUTPUT_DIR / "wt_synii_conditions_mean_traces_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to {output_file}")

### A.10 Condition Pooling

The final phase of Chapter A consolidates cleaned traces and associated metadata into pooled tables that downstream statistical models can consume.

Here we aggregate related experimental conditions into pooled datasets that reflect meaningful biological groupings. This pooling step increases statistical power for later PCA and clustering stages while retaining the ability to trace results back to the original acquisition cohorts.


In [ ]:
## Pool related experimental conditions for analysis. Extracellular Ca²⁺, Stability/plasticity, Synapsin-II / genotype

CONDITION_POOLS = {
    'WT_pooled': ['WT_Theo', 'WT_Anthime', 'WT_Theo_1scd'],
    'stability_before': ['Stability_Before', 'Stability_Before_05'],
    'stability_after': ['Stability_After', 'Stability_After_05']
}

# Add pooled conditions to features dataframe
existing_conditions = set(FEATURES_DATAFRAME['Condition'].unique())
for pool_name, source_conditions in CONDITION_POOLS.items():
    if pool_name not in existing_conditions:
        available_sources = [c for c in source_conditions if c in existing_conditions]
        if available_sources:
            pooled_data              = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'].isin(available_sources)].copy()
            pooled_data['Condition'] = pool_name
            FEATURES_DATAFRAME       = pd.concat([FEATURES_DATAFRAME, pooled_data], ignore_index=True)

# Extract condition-specific dataframes (keeping original names for downstream compatibility)
PCA_Data_WT_Pooled        = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_pooled'].copy()
PCA_Data_WT_Theo          = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_Theo'].copy()
PCA_Data_WT_Anthime       = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'WT_Anthime'].copy()
PCA_Data_SynII            = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'SynII'].copy()
PCA_Data_WT_Low_Ca        = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_1_5Ca'].copy()
PCA_Data_WT_High_Ca       = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_4Ca'].copy()
PCA_Data_Stability_Before = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'stability_before'].copy()
PCA_Data_Stability_After  = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'stability_after'].copy()
PCA_Data_50Hz_1_5_Ca      = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_1_5_50Hz'].copy()
PCA_Data_50Hz_4_Ca        = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_4_50Hz'].copy()
PCA_Data_50Hz_2_5_Ca      = FEATURES_DATAFRAME[FEATURES_DATAFRAME['Condition'] == 'Theo_2_5_50Hz'].copy()    

print(f"✓ {len(FEATURES_DATAFRAME)} boutons across {len(FEATURES_DATAFRAME['Condition'].unique())} conditions")
print(f"Key datasets: WT_pooled({len(PCA_Data_WT_Pooled)}), SynII({len(PCA_Data_SynII)}), 1.5Ca({len(PCA_Data_WT_Low_Ca)}), 4Ca({len(PCA_Data_WT_High_Ca)})")

The PCA will use only the columns AMP1, AMP2, all PPRs and %Fail1 and 2. All the other columns will be un-selected.

## Chapter B – Baseline Comparisons

Chapter B evaluates whether foundational amplitude and plasticity metrics align across experimental cohorts before dimensionality reduction.


### B.1 Distribution Comparison Setup

To test whether key synaptic parameters are comparable across conditions, we assemble a dictionary of metrics—amplitudes, failure rates, and plasticity indices—for visualization. Organizing the data in this way supports consistent histograms and boxplots for each feature.


In [ ]:
# Parameter comparison setup: histograms and boxplots
comparison_datasets = {
    'WT_Theo': PCA_Data_WT_Theo, 
    'WT_Anthime': PCA_Data_WT_Anthime, 
    'SynII': PCA_Data_SynII
}

dataset_colors = ['#1f77b4', '#2ca02c', '#ff7f0e']
color_palette = dict(zip(comparison_datasets.keys(), dataset_colors))

# Parameters to analyze and their reference values
analysis_parameters = ['PPR2/1', 'AMP1', 'AMP2', 'STD_baseline', '%Fail1']
reference_values    = {'PPR2/1': 1.0}  # Theoretical no-facilitation line

# Statistical comparisons to perform
pairwise_comparisons = [("WT_Theo", "SynII"), ("WT_Theo", "WT_Anthime"), ("WT_Anthime", "SynII")]

### B.2 Amplitude and Plasticity Diagnostics

Using the prepared datasets, this cell renders paired histograms and boxplots that contrast amplitude, paired-pulse ratios, and failure fractions across groups. The accompanying statistical annotations help determine whether experimental cohorts can be legitimately compared or require condition-specific treatment.


In [ ]:
# Create parameter comparison plots with statistical analysis (excluding STD_baseline)
parameters_to_plot = [p for p in analysis_parameters if p != 'STD_baseline']
n_params           = len(parameters_to_plot)

fig, axes = plt.subplots(n_params, 2, figsize=(15, 4 * n_params))
# ensure axes is 2D for consistent indexing when n_params == 1
if n_params == 1:
    axes = axes.reshape(1, 2)

statistical_results = []

for param_idx, parameter_name in enumerate(parameters_to_plot):
    # Extract valid data for each dataset
    parameter_data = {}
    for dataset_name, dataset_df in comparison_datasets.items():
        if parameter_name in dataset_df.columns:
            clean_data = dataset_df[parameter_name].dropna()
            if len(clean_data) > 0:
                parameter_data[dataset_name] = clean_data

    # Skip if insufficient data
    if len(parameter_data) < 2 or sum(len(data) for data in parameter_data.values()) < 10:
        statistical_results.append(f"[SKIP] {parameter_name}: insufficient data")
        axes[param_idx, 0].text(0.5, 0.5, f'No data for {parameter_name}',
                               ha='center', va='center', transform=axes[param_idx, 0].transAxes)
        axes[param_idx, 1].axis('off')
        continue

    # Create histogram (left panel)
    ax_histogram         = axes[param_idx, 0]
    all_parameter_values = np.concatenate([data.values for data in parameter_data.values()])
    histogram_bins       = np.linspace(all_parameter_values.min(), all_parameter_values.max(), 21)

    for dataset_name, dataset_values in parameter_data.items():
        transparency = 0.35 if dataset_name == 'SynII' else 0.65
        ax_histogram.hist(dataset_values, bins=histogram_bins, alpha=transparency,
                         label=dataset_name, color=color_palette[dataset_name],
                         density=True, edgecolor='black')

    ax_histogram.set_xlabel(parameter_name)
    ax_histogram.set_ylabel('Probability Density')
    ax_histogram.set_title(f'{parameter_name} Distribution')
    ax_histogram.legend(fontsize=8)

    # Add reference line if specified
    if parameter_name in reference_values:
        ax_histogram.axvline(reference_values[parameter_name], color='gray', linestyle='--', alpha=0.7)

    # Create boxplot with individual points (right panel)
    combined_data_list = []
    for dataset_name, dataset_values in parameter_data.items():
        combined_data_list.append(pd.DataFrame({
            parameter_name: dataset_values,
            'Condition': dataset_name
        }))
    combined_parameter_df = pd.concat(combined_data_list, ignore_index=True)

    ax_boxplot = axes[param_idx, 1]
    sns.boxplot(data=combined_parameter_df, x='Condition', y=parameter_name,
            ax=ax_boxplot, hue='Condition', palette=color_palette, legend=False)
    sns.stripplot(data=combined_parameter_df, x='Condition', y=parameter_name,
                 ax=ax_boxplot, color='black', size=3, alpha=0.6)

    ax_boxplot.set_title(f'{parameter_name} by Condition')
    ax_boxplot.tick_params(axis='x', rotation=30)

    # Add reference line to boxplot
    if parameter_name in reference_values:
        ax_boxplot.axhline(reference_values[parameter_name], color='gray', linestyle='--', alpha=0.7)

    # Statistical annotations and tests
    try:
        # Add pairwise comparison annotations
        stats_annotator = Annotator(ax_boxplot, pairwise_comparisons,
                                   data=combined_parameter_df, x='Condition', y=parameter_name)
        stats_annotator.configure(test='Mann-Whitney', text_format='star', loc='outside',
                                 comparisons_correction='bonferroni', show_test_name=False)
        stats_annotator.apply_and_annotate()

        # Overall group comparison (Kruskal-Wallis)
        group_data           = [parameter_data[name] for name in comparison_datasets.keys() if name in parameter_data]
        kruskal_h, kruskal_p = stats.kruskal(*group_data)
        statistical_results.append(f"{parameter_name}: Kruskal-Wallis H={kruskal_h:.3f}, p={kruskal_p:.4g}")

    except Exception as error:
        statistical_results.append(f"[ERROR] {parameter_name}: {error}")

plt.tight_layout()
plt.show()

# Save results
output_filename_base = OUTPUT_DIR / "01_parameter_comparison"
plt.savefig(f"{output_filename_base}.pdf", dpi=300, bbox_inches='tight')

# Save statistical summary
with open(f"{output_filename_base}_statistics.txt", "w") as stats_file:
    stats_file.write("Parameter Comparison Statistical Results\n")
    stats_file.write("=" * 50 + "\n\n")
    for result in statistical_results:
        stats_file.write(f"{result}\n")
        print(result)

print(f"✓ Saved analysis to {output_filename_base}")

### B.3 Paired-Pulse Response Profiles

Comparing mean PPR waveforms reveals whether facilitation and depression motifs are conserved or condition-specific before clustering.

Here we overlay the averaged PPR trajectories for each condition to inspect facilitation and depression patterns across ten stimulus pulses. This comparison highlights whether short-term plasticity motifs differ significantly between cohorts before entering dimensionality reduction.


In [ ]:
# PPR profile analysis: compare facilitation patterns across conditions

def extract_ppr_profile(condition_dataframe, max_pulse_number=10):
    """Extract PPR profile with means and standard errors for plotting."""
    # Find available PPR columns (PPR2/1, PPR3/1, etc.)
    ppr_column_names = [f'PPR{pulse_num}/1' for pulse_num in range(2, max_pulse_number+1) 
                        if f'PPR{pulse_num}/1' in condition_dataframe.columns]
    
    # PPR1/1 = 1.0 by definition, then calculate means for other ratios
    ppr_means           = [1.0] + condition_dataframe[ppr_column_names].mean().tolist()
    ppr_standard_errors = [0.0] + condition_dataframe[ppr_column_names].sem().tolist()
    total_pulses        = len(ppr_column_names) + 1
    
    return ppr_means, ppr_standard_errors, total_pulses

# Configure conditions for comparison
condition_configs = [
    ('WT_Theo', PCA_Data_WT_Theo, '#1f77b4', 'o'),        # Blue circles
    ('WT_Anthime', PCA_Data_WT_Anthime, '#2ca02c', 's'),  # Green squares  
    ('WT_pooled', PCA_Data_WT_Pooled, '#d62728', 'D'),    # Red diamonds
    ('SynII', PCA_Data_SynII, '#ff7f0e', '^')             # Orange triangles
]

plt.figure(figsize=(10, 6))

# Plot PPR profiles for each condition
for condition_name, condition_data, plot_color, marker_style in condition_configs:
    if condition_data.empty:
        continue
    
    ppr_means, ppr_errors, num_pulses = extract_ppr_profile(condition_data)
    pulse_numbers                     = list(range(1, num_pulses + 1))
    
    # Plot mean trajectory with error bands
    plt.plot(pulse_numbers, ppr_means, marker=marker_style, 
             label = f'{condition_name} (n={len(condition_data)})', 
             color = plot_color, linewidth=2, markersize=6)
    
    # Add standard error shading
    plt.fill_between(pulse_numbers, 
                     np.array(ppr_means) - np.array(ppr_errors),
                     np.array(ppr_means) + np.array(ppr_errors),
                     alpha=0.2, color=plot_color)

# Format plot
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.7, linewidth=1, 
            label='No facilitation')
plt.xlabel('Pulse Number')
plt.ylabel('PPR (A_n/A_1)')
plt.title('Paired-Pulse Ratio Profiles Across Conditions')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "02_ppr_profiles_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print("PPR Profile Summary:")
print("-" * 50)
for condition_name, condition_data, _, _ in condition_configs:
    if condition_data.empty:
        continue
    profile_means, _, num_pulses = extract_ppr_profile(condition_data)
    
    summary_text = f"{condition_name:12} (n={len(condition_data):2d}): PPR2/1={profile_means[1]:.3f}"
    if len(profile_means) >= 10:  # Include PPR10/1 if available
        summary_text += f", PPR10/1={profile_means[9]:.3f}"
    print(summary_text)

print(f"✓ Saved PPR profile comparison to {output_file}")

### B.4 Fig1.d Fiber "20220425_linescan1"

In [ ]:
# Plot traces and PPR profiles for boutons from fiber 20220425_linescan1 in WT_Theo


FIBER_PREFIX = "20220425_linescan1"

# Find matching boutons in WT_Theo
wt_theo_fiber_mask = PCA_Data_WT_Theo['ID'].str.startswith(FIBER_PREFIX)
fiber_boutons = PCA_Data_WT_Theo[wt_theo_fiber_mask].copy()

if len(fiber_boutons) == 0:
    print(f"No boutons found with prefix '{FIBER_PREFIX}' in WT_Theo")
else:
    n_boutons = len(fiber_boutons)
    
    # Collect traces for y-axis scaling
    fiber_traces = []
    for _, bouton in fiber_boutons.iterrows():
        bouton_id = bouton['ID']
        if bouton_id in resampled_trace_lookup:
            fiber_traces.append((bouton_id, resampled_trace_lookup[bouton_id]['Avg']))
    
    # Calculate common y-limits for traces
    all_vals = []
    for _, trace in fiber_traces:
        valid = trace[np.isfinite(trace)]
        if len(valid) > 0:
            all_vals.extend(valid)
    
    if all_vals:
        y_min_plot = np.min(all_vals) * 1.1
        y_max_plot = np.max(all_vals) * 1.1
    else:
        y_min_plot, y_max_plot = -0.5, 0.5
    
    # PPR columns
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in fiber_boutons.columns]
    pulse_numbers = list(range(1, len(ppr_cols) + 2))
    
    # Calculate common y-limits for PPR (starting at 0)
    all_ppr_vals = [1.0]  # PPR1/1 = 1.0
    for _, bouton in fiber_boutons.iterrows():
        all_ppr_vals.extend(bouton[ppr_cols].dropna().tolist())
    ppr_y_max = max(all_ppr_vals) * 1.1
    
    # Create figure with 2 columns: traces (left), PPR profiles (right)
    fig, axes = plt.subplots(n_boutons, 2, figsize=(12, 3 * n_boutons), sharex='col')
    if n_boutons == 1:
        axes = axes.reshape(1, 2)
    
    # Stimulus timing: 20 Hz = 50 ms interval, 10 pulses, starting at 1.0 s
    stim_times = [1.0 + i * 0.05 for i in range(10)]
    
    for idx, (_, bouton) in enumerate(fiber_boutons.iterrows()):
        bouton_id = bouton['ID']
        
        # Left column: trace
        ax_trace = axes[idx, 0]
        if bouton_id in resampled_trace_lookup:
            trace = resampled_trace_lookup[bouton_id]['Avg']
            ax_trace.plot(COMMON_TIME, trace, color='black', linewidth=1)
        ax_trace.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_trace.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_trace.set_xlim(0.5, 1.7)
        ax_trace.set_ylim(y_min_plot, y_max_plot)
        ax_trace.set_ylabel('ΔF/F')
        ax_trace.set_title(f'{bouton_id}', fontsize=9, loc='left')
        
        # Add stimulus ticks above the trace
        for stim_t in stim_times:
            ax_trace.plot(stim_t, y_max_plot * 0.95, marker='|', color='black', markersize=8, markeredgewidth=1.5)
        
        # Right column: PPR profile
        ax_ppr = axes[idx, 1]
        ppr_values = [1.0] + bouton[ppr_cols].tolist()
        ax_ppr.plot(pulse_numbers, ppr_values, marker='o', color='steelblue', linewidth=2, markersize=5)
        ax_ppr.axhline(1.0, color='gray', linestyle='--', linewidth=1)
        ax_ppr.set_xlim(0.5, 10.5)
        ax_ppr.set_ylim(0, ppr_y_max)
        ax_ppr.set_xticks(pulse_numbers)
        ax_ppr.set_ylabel('PPR (A_n/A_1)')
        ax_ppr.grid(True, alpha=0.3)
    
    # X-axis labels on bottom row
    axes[-1, 0].set_xlabel('Time (s)')
    axes[-1, 1].set_xlabel('Pulse Number')
    
    plt.tight_layout()
    plt.suptitle(f'Fiber {FIBER_PREFIX} – WT_Theo (n={n_boutons} boutons)', 
                 fontsize=12, fontweight='bold', y=1.01)
    
    output_file = OUTPUT_DIR / f"Fig1d_fiber_{FIBER_PREFIX}_traces_ppr.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Plotted {n_boutons} boutons from fiber '{FIBER_PREFIX}'")
    print(f"✓ Saved to {output_file}")


## Chapter C – PCA Preparation

Chapter C standardizes features and constructs a unified PCA model that anchors all subsequent visualizations and statistics.


### C.1 Feature Sanitization for PCA

Before dimensionality reduction, we strip away non-numeric identifiers, harmonize column names, and scale each feature to unit variance. This normalization ensures that PCA captures true covariation among synaptic properties rather than artifacts of measurement scale.


In [ ]:
# Prepare datasets for PCA analysis by removing non-feature columns and scaling

def apply_standard_drops(dataframe_dict):
    """Apply standard column drops for PCA analysis to multiple dataframes."""
    return {name: df.drop(columns=[col for col in PCA_DROP_COLS if col in df.columns], errors='ignore') 
            for name, df in dataframe_dict.items() if df is not None and hasattr(df, 'columns')}

# Organize all condition dataframes
condition_dfs = {
    'PCA_Data_WT_Pooled'       : PCA_Data_WT_Pooled, 'PCA_Data_WT_Theo': PCA_Data_WT_Theo, 'PCA_Data_WT_Anthime': PCA_Data_WT_Anthime,
    'PCA_Data_SynII'           : PCA_Data_SynII, 'PCA_Data_WT_Low_Ca': PCA_Data_WT_Low_Ca, 'PCA_Data_WT_High_Ca': PCA_Data_WT_High_Ca,
    'PCA_Data_Stability_Before': PCA_Data_Stability_Before, 'PCA_Data_Stability_After': PCA_Data_Stability_After
}

# Prepare reference dataset for PCA (remove metadata columns)
WT_pooled_for_pca = PCA_Data_WT_Pooled.drop(columns=[col for col in PCA_DROP_COLS if col in PCA_Data_WT_Pooled.columns])

# Fit StandardScaler on WT_pooled reference dataset
scaler                   = StandardScaler()
scaled_data              = {}
scaled_data['WT_pooled'] = scaler.fit_transform(WT_pooled_for_pca)

# Transform all other datasets using same scaling parameters from WT_pooled
datasets_to_scale = {
    'WT_Theo'    : PCA_Data_WT_Theo, 'WT_Anthime': PCA_Data_WT_Anthime, 'SynII': PCA_Data_SynII,
    'WT_1_5Ca'   : PCA_Data_WT_Low_Ca, 'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before, 'stab_after': PCA_Data_Stability_After,
    '50Hz_1_5Ca' : PCA_Data_50Hz_1_5_Ca, '50Hz_4Ca': PCA_Data_50Hz_4_Ca, '50Hz_2_5Ca': PCA_Data_50Hz_2_5_Ca
}

for dataset_name, dataset_df in datasets_to_scale.items():
    pca_features              = dataset_df.drop(columns=[col for col in PCA_DROP_COLS if col in dataset_df.columns])
    scaled_data[dataset_name] = scaler.transform(pca_features)

print(f"✓ Feature columns for PCA: {list(WT_pooled_for_pca.columns)}")
print(f"✓ Removed {len(PCA_DROP_COLS)} metadata columns from each dataset")  
print(f"✓ Scaled {len(scaled_data)} datasets using WT_pooled reference parameters")

### C.2 Reference PCA Model

The WT pooled dataset defines the PCA axes used throughout the study. Fitting the decomposition here and projecting every condition into the shared space creates a consistent coordinate system for cross-condition comparisons and clustering.


In [ ]:
# Fit PCA on WT_pooled reference dataset and transform all conditions

# Fit PCA model using WT_pooled as reference
pca                   = PCA(n_components=2)
pca_data              = {}
pca_data['WT_pooled'] = pca.fit_transform(scaled_data['WT_pooled'])

# Transform all other datasets using same PCA axes from WT_pooled
for dataset_name in datasets_to_scale.keys():
    pca_data[dataset_name] = pca.transform(scaled_data[dataset_name])

# Display PCA results summary
variance_pc1, variance_pc2 = pca.explained_variance_ratio_
print(f"✓ PCA transformation complete")
print(f"✓ PC1 explains {variance_pc1:.1%} of variance, PC2 explains {variance_pc2:.1%}")
print(f"✓ Total variance explained: {variance_pc1 + variance_pc2:.1%}")
print(f"✓ Transformed {len(pca_data)} datasets using WT_pooled PCA axes")

### C.3 PCA DataFrames

Projected coordinates are packaged into tidy DataFrames alongside metadata so that plotting and statistical routines can access principal components with clear provenance. This structure underpins every visualization and classification step built on top of the PCA embedding.


In [ ]:
# Convert PCA coordinates to DataFrames for analysis and plotting

# Create DataFrames for PCA-transformed coordinates
principal_component_columns = ['PC1', 'PC2']
pca_dfs                     = {}

for dataset_name, pca_coordinates in pca_data.items():
    pca_dfs[f'transformed_{dataset_name}'] = pd.DataFrame(pca_coordinates, columns=principal_component_columns)

# Create PCA components table showing feature contributions to each PC
feature_names = list(WT_pooled_for_pca.columns)
df_components = pd.DataFrame(pca.components_, columns=feature_names, index=['PC1', 'PC2'])

# Display top feature contributors for each principal component
print("PCA Components Analysis (top 3 contributors per PC):")
print("-" * 50)
for pc_name in ['PC1', 'PC2']:
    top_contributing_features = df_components.loc[pc_name].abs().nlargest(3)
    feature_contributions     = [f'{feature_name}({contribution:.3f})' 
                           for feature_name, contribution in top_contributing_features.items()]
    print(f"  {pc_name}: {', '.join(feature_contributions)}")

print(f"\n✓ Created {len(pca_dfs)} PCA coordinate DataFrames")
print(f"✓ PCA components table shape: {df_components.shape}")

### C.4 PCA–Feature Correlation Mapping

By concatenating PCA coordinates with the original feature measurements, we compute correlation coefficients that reveal which biophysical parameters drive each principal component. Identifying the strongest contributors translates abstract PCA axes back into mechanistic synaptic descriptors.


In [ ]:
# Combine PCA coordinates with original features for correlation analysis

# Create combined dataset: PCA coordinates + original features
combined_pca_FEATURES_DATAFRAME = pd.concat([
    pca_dfs['transformed_WT_pooled'].reset_index(drop=True),
    PCA_Data_WT_Pooled.reset_index(drop=True)
], axis=1)

# Calculate correlation matrix between principal components and original features
full_correlation_matrix = combined_pca_FEATURES_DATAFRAME.corr(numeric_only=True)
pc_feature_correlations = full_correlation_matrix.iloc[:2, 2:]  # Extract PC1,PC2 vs features

# Display strongest correlations for interpretability
print("Strongest PC-Feature Correlations:")
print("-" * 40)
for pc_name in ['PC1', 'PC2']:
    strongest_correlations = pc_feature_correlations.loc[pc_name].abs().nlargest(3)
    correlation_strings    = [f'{feature_name}({correlation_value:.3f})' 
                          for feature_name, correlation_value in strongest_correlations.items()]
    print(f"  {pc_name}: {', '.join(correlation_strings)}")

# Store comprehensive PCA results for downstream analysis
PCA_RESULTS = {
    'pca_model'         : pca,                           # Fitted PCA transformer
    'scaler'            : scaler,                        # Fitted StandardScaler
    'pca_dataframes'    : pca_dfs,                       # PCA coordinates for all datasets
    'components'        : df_components,                 # Feature contributions to PCs
    'correlations'      : pc_feature_correlations,       # PC-feature correlation matrix
    'explained_variance': pca.explained_variance_ratio_  # Variance explained by each PC
}

print(f"\n✓ Correlation analysis complete")
print(f"✓ PCA results stored in PCA_RESULTS dictionary with {len(PCA_RESULTS)} components")

## Chapter D – Clustering and Release Phenotypes

Chapter D leverages the PCA embedding to interrogate hierarchical clusters, relate them to biological targets, and describe release properties.


### D.1 Cluster Plot Utilities

We import specialized plotting utilities that annotate PCA scatter plots with explained variance and cluster information. These helpers streamline the visualization of complex clustering results throughout the rest of the chapter.


In [ ]:
# Perform hierarchical clustering on PCA-transformed WT_pooled data

# Extract PCA coordinates for clustering
pca_coordinates = pca_data['WT_pooled']  # Shape: (n_samples, 2)

# Perform hierarchical clustering using Ward linkage method
linkage_matrix      = linkage(pca_coordinates, method='ward')
cluster_assignments_raw = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')

# Compute mean AMP1 for each cluster and reorder labels accordingly
amp1_values = PCA_Data_WT_Pooled['AMP1'].values
cluster_amp1_means = {}
for cid in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments_raw == cid
    cluster_amp1_means[cid] = np.nanmean(amp1_values[mask])

# Sort clusters by ascending mean AMP1
sorted_clusters = sorted(cluster_amp1_means.keys(), key=lambda c: cluster_amp1_means[c])
old_to_new = {old: new for new, old in enumerate(sorted_clusters, start=1)}
cluster_assignments = np.array([old_to_new[c] for c in cluster_assignments_raw])

# Add cluster labels to original dataframe
PCA_Data_WT_Pooled_clustered               = PCA_Data_WT_Pooled.copy()
PCA_Data_WT_Pooled_clustered['HC_Cluster'] = cluster_assignments

# Visualize clusters in PCA space using plot_pca_nice
pc1_variance = PCA_RESULTS["explained_variance"][0]
pc2_variance = PCA_RESULTS["explained_variance"][1]

# Generate consistent Set1 colors for clusters
set1_colors          = Set1(np.linspace(0, 1, N_CLUSTERS))
cluster_rgba_colors  = [tuple(color) for color in set1_colors]
cluster_hex_colors   = [to_hex(color) for color in cluster_rgba_colors]
cluster_palette      = ListedColormap(cluster_rgba_colors, name='cluster_palette')
cluster_color_lookup = {cid: cluster_rgba_colors[cid - 1] for cid in range(1, N_CLUSTERS + 1)}

def get_cluster_color(cluster_id):
    cluster_id = int(cluster_id)
    return cluster_color_lookup[cluster_id]

def get_cluster_colors(labels):
    return [get_cluster_color(cid) for cid in labels]

def get_cluster_hex_color(cluster_id):
    cluster_id = int(cluster_id)
    return cluster_hex_colors[cluster_id - 1]

# Plot each cluster separately for legend
for cluster_id in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments == cluster_id
    plt.scatter(pca_coordinates[mask, 0], pca_coordinates[mask, 1],
                c=[get_cluster_color(cluster_id)],
                s=50, marker='o', edgecolors='black', linewidths=0.6,
                label=f'Cluster {cluster_id}', alpha=0.9)

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('Hierarchical Clustering in PCA Space (WT pooled)')
plt.legend()
plt.grid(True, alpha=0.3)
output_file = OUTPUT_DIR / "Fig3a_pca_wt_pooled_clustered.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Display cluster distribution statistics
print("Hierarchical Clustering Results (ordered by mean AMP1):")
print("-" * 50)
total_samples = len(cluster_assignments)

for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_size       = np.sum(cluster_assignments == cluster_id)
    cluster_percentage = (cluster_size / total_samples) * 100
    mean_amp1          = np.nanmean(amp1_values[cluster_assignments == cluster_id])
    print(f"Cluster {cluster_id}: {cluster_size:2d} samples ({cluster_percentage:4.1f}%), mean AMP1={mean_amp1:.4f}")



print(f"Total: {total_samples} samples distributed across {N_CLUSTERS} clusters")
print("✓ Saved clustering visualization using consistent cluster colors")


In [ ]:
# Plot PCA without cluster colors - all points in gray

plt.figure(figsize=(8, 6))

# Plot all WT pooled points in gray
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c='gray', alpha=1, s=50, edgecolors='black')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA Projection: WT Pooled (unclustered)')
plt.grid(True, alpha=0.3)
plt.tight_layout()



# Save and display
output_file = OUTPUT_DIR / "04_pca_wt_pooled_unclustered.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved unclustered PCA plot to {output_file}")
print(f"✓ Total samples: {len(pca_coordinates)}")

#### D.1.a Control boutons from same fibers

In [ ]:
# Extract fiber IDs from WT pooled data (first 22 characters)
fiber_ids = PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22]).unique()

# Filter fibers with more than 4 boutons
fiber_bouton_counts = {fiber: PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22] == fiber).sum() 
                       for fiber in fiber_ids}
eligible_fibers = [fiber for fiber, count in fiber_bouton_counts.items() if count > 2]

print(f"Fibers with >4 boutons: {len(eligible_fibers)} out of {len(fiber_ids)} total fibers")

# Calculate grid dimensions for all eligible fibers
n_fibers = len(eligible_fibers)
n_cols = 3
n_rows = int(np.ceil(n_fibers / n_cols))

# Create figure with subplots for all eligible fibers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
axes = axes.flatten()

# Color palette for fibers
set1_cmap = plt.get_cmap('Set1')

# Plot each eligible fiber in its own subplot
for idx, fiber_prefix in enumerate(eligible_fibers):
    ax = axes[idx]
    
    # Plot WT pooled with cluster colors (background) - lighter
    ax.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
               c=get_cluster_colors(cluster_assignments), alpha=0.15, s=20)
    
    # Find boutons belonging to this fiber
    fiber_mask = PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22] == fiber_prefix)
    fiber_indices = np.where(fiber_mask)[0]
    
    if len(fiber_indices) > 0:
        # Use consistent color for this fiber
        fiber_color = set1_cmap(idx % 9)  # Cycle through Set1 colors
        
        # Plot all boutons from this fiber with the same color
        ax.scatter(pca_coordinates[fiber_indices, 0], pca_coordinates[fiber_indices, 1],
                   color=fiber_color, s=150, marker='o', 
                   edgecolors='black', linewidths=1.5, alpha=0.9,
                   label=f'{fiber_prefix}\n(n={len(fiber_indices)} boutons)')
    
    # Format subplot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    ax.set_xlim(-7, 10)
    ax.set_ylim(-6, 6)
    ax.set_title(f'Fiber: {fiber_prefix}', fontsize=10, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(n_fibers, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'PCA: All Fibers from WT Pooled with >4 boutons ({n_fibers} fibers)', 
             fontsize=16, fontweight='bold', y=1.002)
plt.show()

# Print fiber statistics
print(f"\n=== ALL FIBERS STATISTICS (>4 boutons) ===")
print(f"Total unique fibers in WT pooled: {len(fiber_ids)}")
print(f"Fibers with >4 boutons: {len(eligible_fibers)}")
print("-" * 60)
for fiber_prefix in eligible_fibers:
    fiber_mask = PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22] == fiber_prefix)
    n_boutons = fiber_mask.sum()
    
    # Get cluster distribution for this fiber
    fiber_clusters = cluster_assignments[fiber_mask]
    cluster_counts = {i: np.sum(fiber_clusters == i) for i in range(1, N_CLUSTERS + 1) if np.sum(fiber_clusters == i) > 0}
    
    print(f"  {fiber_prefix}: {n_boutons} boutons")
    print(f"    Cluster distribution: {cluster_counts}")


#### D.1.b Fiber Area Distribution

In [ ]:
from scipy.spatial import ConvexHull
from scipy.stats import mannwhitneyu
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

# Analyze PCA area coverage for individual fibers with 4+ boutons


def calculate_convex_hull_area(points):
    """Calculate area of convex hull for a set of 2D points."""
    if len(points) < 3:
        return 0.0
    try:
        hull = ConvexHull(points)
        return hull.volume  # In 2D, volume is area
    except Exception:
        return 0.0

# Extract fiber IDs from WT pooled data (first 22 characters)
fiber_ids = PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22]).unique()

# Filter fibers with 4+ boutons
fiber_bouton_counts = {fiber: PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22] == fiber).sum() 
                       for fiber in fiber_ids}
eligible_fibers = {fiber: count for fiber, count in fiber_bouton_counts.items() if count >= 4}

print(f"=== FIBER AREA ANALYSIS ===")
print(f"Total unique fibers in WT pooled: {len(fiber_ids)}")
print(f"Fibers with 4+ boutons: {len(eligible_fibers)}")

# Calculate real fiber areas
real_fiber_areas = []
fiber_size_distribution = {}  # Track how many boutons per fiber

for fiber_prefix, n_boutons in eligible_fibers.items():
    fiber_mask = PCA_Data_WT_Pooled['ID'].apply(lambda x: str(x)[:22] == fiber_prefix)
    fiber_indices = np.where(fiber_mask)[0]
    fiber_coords = pca_coordinates[fiber_indices]
    
    area = calculate_convex_hull_area(fiber_coords)
    real_fiber_areas.append(area)
    
    # Track size distribution
    if n_boutons not in fiber_size_distribution:
        fiber_size_distribution[n_boutons] = 0
    fiber_size_distribution[n_boutons] += 1

print(f"\nFiber size distribution:")
for size in sorted(fiber_size_distribution.keys()):
    count = fiber_size_distribution[size]
    print(f"  {size} boutons: {count} fibers ({100*count/len(eligible_fibers):.1f}%)")

# Generate null distribution by random sampling
n_permutations = 10000
null_areas = []

print(f"\nGenerating null distribution with {n_permutations} permutations...")

# Get all WT pooled coordinates
all_wt_coords = pca_coordinates.copy()
n_total_boutons = len(all_wt_coords)

def generate_null_areas_for_permutation(args):
    """Generate null areas for a single permutation."""
    fiber_size_distribution, all_wt_coords, seed = args
    np.random.seed(seed)
    n_total_boutons = len(all_wt_coords)
    perm_areas = []
    
    for fiber_size, n_fibers in fiber_size_distribution.items():
        for _ in range(n_fibers):
            random_indices = np.random.choice(n_total_boutons, size=fiber_size, replace=False)
            random_coords = all_wt_coords[random_indices]
            area = calculate_convex_hull_area(random_coords)
            perm_areas.append(area)
    
    return perm_areas

# Prepare arguments for parallel execution
seeds = np.random.randint(0, 2**31, size=n_permutations)
args_list = [(fiber_size_distribution, all_wt_coords, seed) for seed in seeds]

# Use multiprocessing to parallelize
# Use threading to parallelize (ProcessPoolExecutor fails in notebooks due to pickling issues)
n_workers = min(multiprocessing.cpu_count(), 8)
print(f"Using {n_workers} workers for parallel computation...")

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    results = list(executor.map(generate_null_areas_for_permutation, args_list))
for perm_areas in results:
    null_areas.extend(perm_areas)

print(f"  Completed {n_permutations} permutations")

# Statistical comparison
real_areas_array = np.array(real_fiber_areas)
null_areas_array = np.array(null_areas)

# Mann-Whitney U test
u_stat, p_value = mannwhitneyu(real_areas_array, null_areas_array, alternative='two-sided')

# Calculate summary statistics
real_median = np.median(real_areas_array)
real_mean = np.mean(real_areas_array)
null_median = np.median(null_areas_array)
null_mean = np.mean(null_areas_array)

print(f"\n=== STATISTICAL RESULTS ===")
print(f"Real fibers (n={len(real_areas_array)}):")
print(f"  Mean area: {real_mean:.3f}")
print(f"  Median area: {real_median:.3f}")
print(f"  Range: [{np.min(real_areas_array):.3f}, {np.max(real_areas_array):.3f}]")

print(f"\nNull distribution (n={len(null_areas_array)}):")
print(f"  Mean area: {null_mean:.3f}")
print(f"  Median area: {null_median:.3f}")
print(f"  Range: [{np.min(null_areas_array):.3f}, {np.max(null_areas_array):.3f}]")

print(f"\nMann-Whitney U test:")
print(f"  U-statistic: {u_stat:.2f}")
print(f"  p-value: {p_value:.6g}")
print(f"  Effect: Real fibers {'smaller' if real_median < null_median else 'larger'} than random")

# Visualization: side-by-side histograms
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Determine common bins for fair comparison
all_areas = np.concatenate([real_areas_array, null_areas_array])
common_bins = np.linspace(0, np.percentile(all_areas, 99), 30)

# Left panel: Real fiber areas
ax1.hist(real_areas_array, bins=common_bins, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(real_median, color='red', linestyle='--', linewidth=2, label=f'Median: {real_median:.3f}')
ax1.set_xlabel('Convex Hull Area (PCA units²)')
ax1.set_ylabel('Frequency')
ax1.set_title(f'Real Fibers (n={len(real_areas_array)})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right panel: Null distribution
ax2.hist(null_areas_array, bins=common_bins, alpha=0.7, color='lightcoral', edgecolor='black')
ax2.axvline(null_median, color='darkred', linestyle='--', linewidth=2, label=f'Median: {null_median:.3f}')
ax2.axvline(real_median, color='steelblue', linestyle=':', linewidth=2, label=f'Real median (ref)')
ax2.set_xlabel('Convex Hull Area (PCA units²)')
ax2.set_ylabel('Frequency')
ax2.set_title(f'Null Distribution (n={len(null_areas_array)})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Fiber Area Distribution: Real vs Random\nMann-Whitney p={p_value:.6g}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

output_file = OUTPUT_DIR / "05_fiber_area_distribution_real_vs_null.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Additional visualization: Overlay with transparency
plt.figure(figsize=(10, 6))

plt.hist(null_areas_array, bins=common_bins, alpha=0.5, color='lightcoral', 
         edgecolor='black', label=f'Null (n={len(null_areas_array)})', density=True)
plt.hist(real_areas_array, bins=common_bins, alpha=0.7, color='steelblue', 
         edgecolor='black', label=f'Real fibers (n={len(real_areas_array)})', density=True)

plt.axvline(null_median, color='darkred', linestyle='--', linewidth=2, 
            label=f'Null median: {null_median:.3f}')
plt.axvline(real_median, color='darkblue', linestyle='--', linewidth=2, 
            label=f'Real median: {real_median:.3f}')

plt.xlabel('Convex Hull Area (PCA units²)')
plt.ylabel('Probability Density')
plt.title(f'Fiber Area Distribution Comparison\nMann-Whitney U={u_stat:.0f}, p={p_value:.6g}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

output_file_overlay = OUTPUT_DIR / "06_fiber_area_overlay_comparison.pdf"
plt.savefig(output_file_overlay, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved area distribution plots:")
print(f"  {output_file}")
print(f"  {output_file_overlay}")

# Save detailed results to text file
stats_output = OUTPUT_DIR / "fiber_area_statistics.txt"
with open(stats_output, 'w') as f:
    f.write("FIBER AREA ANALYSIS: REAL vs NULL DISTRIBUTION\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Dataset: WT pooled (fibers with 4+ boutons)\n")
    f.write(f"Total fibers analyzed: {len(eligible_fibers)}\n\n")
    
    f.write("FIBER SIZE DISTRIBUTION:\n")
    for size in sorted(fiber_size_distribution.keys()):
        count = fiber_size_distribution[size]
        f.write(f"  {size} boutons: {count} fibers ({100*count/len(eligible_fibers):.1f}%)\n")
    
    f.write(f"\nREAL FIBER AREAS (n={len(real_areas_array)}):\n")
    f.write(f"  Mean:   {real_mean:.6f}\n")
    f.write(f"  Median: {real_median:.6f}\n")
    f.write(f"  SD:     {np.std(real_areas_array):.6f}\n")
    f.write(f"  Min:    {np.min(real_areas_array):.6f}\n")
    f.write(f"  Max:    {np.max(real_areas_array):.6f}\n")
    
    f.write(f"\nNULL DISTRIBUTION (n={len(null_areas_array)}):\n")
    f.write(f"  Permutations: {n_permutations}\n")
    f.write(f"  Mean:   {null_mean:.6f}\n")
    f.write(f"  Median: {null_median:.6f}\n")
    f.write(f"  SD:     {np.std(null_areas_array):.6f}\n")
    f.write(f"  Min:    {np.min(null_areas_array):.6f}\n")
    f.write(f"  Max:    {np.max(null_areas_array):.6f}\n")
    
    f.write(f"\nSTATISTICAL TEST:\n")
    f.write(f"  Test: Mann-Whitney U (two-sided)\n")
    f.write(f"  U-statistic: {u_stat:.6f}\n")
    f.write(f"  p-value: {p_value:.10f}\n")
    f.write(f"  Effect: Real fibers are {'smaller' if real_median < null_median else 'larger'} than random\n")
    f.write(f"  Median difference: {real_median - null_median:.6f}\n")

print(f"✓ Saved detailed statistics to {stats_output}")

### D.2 Hierarchical Dendrogram

Using the PCA coordinates, this code reconstructs the hierarchical clustering tree to visualize how boutons group across linkage distances. The dendrogram exposes nested relationships among boutons that complement the 2D PCA projection.


In [ ]:
# Create dendrogram visualization of hierarchical clustering

# Reuse consistent cluster colors for dendrogram branches
if 'cluster_hex_colors' not in globals():
    dendrogram_colormap = plt.get_cmap('Set1')
    cluster_hex_colors  = [to_hex(dendrogram_colormap(i)) for i in range(N_CLUSTERS)]
set_link_color_palette(cluster_hex_colors)

# Create sample labels for dendrogram leaves
try:
    sample_labels = PCA_Data_WT_Pooled.index.tolist()
except AttributeError:
    sample_labels = [f'Sample_{i+1}' for i in range(len(pca_coordinates))]

# Calculate clustering threshold for specified number of clusters
clustering_threshold = linkage_matrix[-N_CLUSTERS+1, 2]

# Generate dendrogram plot
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix,
          color_threshold=clustering_threshold,
          labels=sample_labels,
          leaf_rotation=90,
          leaf_font_size=8)

plt.title(f'Hierarchical Clustering Dendrogram (k={N_CLUSTERS})')
plt.xlabel('Samples')
plt.ylabel('Ward Distance')
plt.tight_layout()

# Save dendrogram
output_file = OUTPUT_DIR / "07_hierarchical_clustering_dendrogram.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display clustering information
print(f"Clustering threshold for {N_CLUSTERS} clusters: {clustering_threshold:.3f}")
print(f"Dendrogram branches colored by cluster membership")
print(f"✓ Saved dendrogram to {output_file}")


### D.3 PCA Correlation Circle

A correlation circle is generated to display how each feature loads onto the first two principal components. This biplot view clarifies which synaptic properties pull samples along specific PCA axes and aids in interpreting cluster separation.


In [ ]:
# Create PCA correlation circle (biplot) showing feature contributions to principal components

# Extract correlation coefficients between PCs and original features
feature_pc_correlations = PCA_RESULTS['correlations'].T.values  # Shape: (n_features, 2)
scaling_factor          = 1.0
correlation_vectors     = feature_pc_correlations * scaling_factor

# Create correlation circle visualization
fig, ax = plt.subplots(figsize=(8, 8))

# Add reference elements: axes and unit circle
ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
unit_circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='-', alpha=0.5)
ax.add_patch(unit_circle)

# Plot correlation vectors for each feature
feature_names = list(WT_pooled_for_pca.columns)
for feature_idx, feature_name in enumerate(feature_names):
    pc1_correlation, pc2_correlation = correlation_vectors[feature_idx, 0], correlation_vectors[feature_idx, 1]
    
    # Draw correlation vector as arrow
    ax.arrow(0, 0, pc1_correlation, pc2_correlation, 
             color='darkred', alpha=0.8, 
             head_width=0.03, head_length=0.05, 
             length_includes_head=True, linewidth=1.5)
    
    # Position feature label outside the arrow tip
    label_x_position = pc1_correlation * 1.1
    label_y_position = pc2_correlation * 1.1
    ax.text(label_x_position, label_y_position, feature_name, 
            ha='center', va='center', fontsize=10, weight='bold', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Format plot with variance information
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
total_variance             = pc1_variance + pc2_variance

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.set_title(f'PCA Correlation Circle\n({total_variance:.1%} total variance explained)')
ax.grid(True, alpha=0.2)

plt.tight_layout()

# Save correlation circle
output_file = OUTPUT_DIR / "Fig3a_pca_correlation_circle.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display strongest feature correlations for biological interpretation
print("Strongest Feature-PC Correlations:")
print("=" * 45)
pc_feature_correlations = PCA_RESULTS['correlations']

for pc_name in ['PC1', 'PC2']:
    strongest_features = pc_feature_correlations.loc[pc_name].abs().nlargest(5)
    print(f"\n{pc_name} (strongest contributors):")
    
    for feature_name, correlation_magnitude in strongest_features.items():
        correlation_value     = pc_feature_correlations.loc[pc_name, feature_name]
        correlation_direction = "+" if correlation_value > 0 else "-"
        print(f"  {correlation_direction} {feature_name}: {correlation_magnitude:.3f}")

print(f"\n✓ Saved correlation circle to {output_file}")

### D.4 WT Reference Check

To validate that Anthime's WT dataset aligns with the pooled WT reference, we overlay both cohorts in PCA space. This control confirms that lab-to-lab differences do not distort the shared coordinate system.


In [ ]:
# Compare WT Pooled and WT Anthime datasets in PCA space

# Extract PCA coordinates for comparison datasets
wt_pooled_coordinates  = np.asarray(pca_data['WT_pooled'])
wt_anthime_coordinates = np.asarray(pca_data['WT_Anthime'])

# Create PCA comparison plot
plt.figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')


# Plot WT Anthime dataset  
plt.scatter(wt_anthime_coordinates[:, 0], wt_anthime_coordinates[:, 1], 
           marker='d', edgecolors='black', linewidths=0.6, s=50, c='magenta',
           label=f'WT Anthime (n={len(wt_anthime_coordinates)})')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA Projection: WT Pooled vs WT Anthime')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "09_pca_wt_pooled_vs_anthime.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ PCA comparison plot saved to {output_file}")
print(f"✓ WT Pooled: {len(wt_pooled_coordinates)} samples")
print(f"✓ WT Anthime: {len(wt_anthime_coordinates)} samples")

### D.5 Release Property Survey

Beyond traces, we examine amplitude and plasticity metrics across clusters to understand how release phenotypes differ among bouton classes.


#### D.5.a Cluster PPR Trajectories

Paired-pulse response curves are assembled per cluster to evaluate how facilitation or depression patterns differ among bouton classes. Linking temporal plasticity to cluster identity refines our interpretation of each group.


In [ ]:
# PPR profiles by hierarchical cluster
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
x_pulses = list(range(1, len(ppr_cols)+2))
clusters = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].unique())

plt.figure(figsize=(10, 6))
for cluster in clusters:
    cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster]
    means        = [1] + cluster_data[ppr_cols].mean().tolist()
    sems         = [0] + cluster_data[ppr_cols].sem().tolist()
    color        = get_cluster_color(cluster)

    plt.plot(x_pulses, means, marker='o', label=f'Cluster {cluster} (n={len(cluster_data)})', color=color, linewidth=2)
    plt.fill_between(x_pulses, np.array(means)-np.array(sems), np.array(means)+np.array(sems), alpha=0.2, color=color)

plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.ylabel('Mean PPR (A_n/A_1)')
plt.xlabel('Pulse Number')
plt.xticks(x_pulses)
plt.title('PPR Profiles by Hierarchical Cluster')
plt.legend(loc='upper right')
plt.tight_layout()

output_file = OUTPUT_DIR / "Fig3h_ppr_profiles_by_cluster.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()


#### D.5.b Cluster Metric Utility

Reusable helper functions are defined to produce boxplots and statistical annotations for any scalar feature across clusters. This modularity supports consistent reporting of amplitude and plasticity differences.


In [ ]:
# Reusable function for cluster boxplot analysis
def cluster_boxplot_analysis(data, column, title, output_prefix):
    'Create boxplot by cluster with statistical analysis.'
    from scipy.stats import kruskal, mannwhitneyu
    from itertools import combinations

    plt.figure(figsize=(8, 5))
    clusters = sorted(data['HC_Cluster'].unique())
    ax = sns.boxplot(x='HC_Cluster', y=column, hue='HC_Cluster', data=data, palette=cluster_hex_colors, legend=False)
    sns.stripplot(x='HC_Cluster', y=column, data=data, color='k', size=3, alpha=0.5, ax=ax)

    if 'PPR' in column:
        plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)

    plt.ylabel(column)
    plt.title(title)

    # Clean styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.get_xaxis().set_visible(False)

    # Create legend
    handles = [plt.Line2D([0], [0], color=get_cluster_color(cluster_id), lw=4) for cluster_id in clusters]
    labels = [f'Cluster {cluster_id}' for cluster_id in clusters]
    plt.legend(handles, labels, loc='upper right')

    # Statistical tests
    data_per_cluster = {c: data.loc[data['HC_Cluster'] == c, column].dropna().values for c in clusters}

    try:
        kw_stat, kw_p = kruskal(*data_per_cluster.values())
    except ValueError:
        kw_stat, kw_p = float('nan'), float('nan')

    pairs = list(combinations(clusters, 2))
    results = []
    for a, b in pairs:
        x, y = data_per_cluster[a], data_per_cluster[b]
        try:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), float('nan')
        results.append({
            'C_A': a, 'C_B': b, 'n_A': len(x), 'n_B': len(y),
            'U_stat': stat, 'p_raw': p, 'p_corr': min(p * len(pairs), 1.0) if not np.isnan(p) else np.nan
        })

    results.sort(key=lambda d: d['p_corr'] if not np.isnan(d['p_corr']) else 1)

    # Save results
    stats_file = OUTPUT_DIR / f"{output_prefix}_statistics.txt"
    with open(stats_file, "w") as f:
        f.write(f"{column} Statistical Analysis by Cluster")
        f.write(f"Kruskal-Wallis: H = {kw_stat:.4f}, p = {kw_p:.6g}")
        f.write(f"Bonferroni correction: {len(pairs)}")
        f.write("C_A	C_B	nA	nB	U_stat	p_raw	p_corr")
        for r in results:
            f.write(f"{r['C_A']}	{r['C_B']}	{r['n_A']}	{r['n_B']}	"
                    f"{r['U_stat']:.4f}	{r['p_raw']:.6g}	{r['p_corr']:.6g}")

    plt.tight_layout()
    output_file = OUTPUT_DIR / f"{output_prefix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✓ Saved {column} analysis to {output_file} and {stats_file}")


#### D.5.c Cluster-Level components Analysis

Applying the reusable plotting routine, we examine how components varies across clusters. This metric probes whether early facilitation distinguishes bouton groups discovered by hierarchical clustering.


In [ ]:
# AMP1 analysis  
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'AMP1', 'AMP1 by Cluster', 'Fig3e_amp1_cluster')

# PPR2/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR2/1', 'PPR2/1 by Cluster', 'Fig3f_ppr2_1_cluster')

# PPR3/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR3/1', 'PPR3/1 by Cluster', 'FigS3b_ppr3_1_cluster')

# %Fail1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, '%Fail1', '%Fail1 by Cluster', 'Fig3g_fail1_cluster')

#### D.5.d Mean trace for each cluster

In [ ]:
# Plot mean traces for each cluster

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Collect traces by cluster
cluster_traces = {cid: [] for cid in range(1, N_CLUSTERS + 1)}

for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    bouton_id = row['ID']
    cluster_id = row['HC_Cluster']
    if bouton_id in resampled_trace_lookup:
        cluster_traces[cluster_id].append(resampled_trace_lookup[bouton_id]['Avg'])

# Calculate global y-limits
all_means = []
all_sems = []
for cid in range(1, N_CLUSTERS + 1):
    if cluster_traces[cid]:
        traces_matrix = np.column_stack(cluster_traces[cid])
        mean_trace = np.nanmean(traces_matrix, axis=1)
        sem_trace = np.nanstd(traces_matrix, axis=1, ddof=1) / np.sqrt(traces_matrix.shape[1])
        all_means.append(mean_trace)
        all_sems.append(sem_trace)

all_upper = [m + s for m, s in zip(all_means, all_sems)]
all_lower = [m - s for m, s in zip(all_means, all_sems)]
y_min_global = np.nanmin([np.nanmin(l) for l in all_lower])
y_max_global = np.nanmax([np.nanmax(u) for u in all_upper])
y_margin = (y_max_global - y_min_global) * 0.05
y_lim = (y_min_global - y_margin, y_max_global + y_margin)

# Plot each cluster
for idx, cid in enumerate(range(1, N_CLUSTERS + 1)):
    ax = axes[idx]
    color = get_cluster_color(cid)
    
    if cluster_traces[cid]:
        traces_matrix = np.column_stack(cluster_traces[cid])
        mean_trace = np.nanmean(traces_matrix, axis=1)
        sem_trace = np.nanstd(traces_matrix, axis=1, ddof=1) / np.sqrt(traces_matrix.shape[1])
        
        ax.plot(COMMON_TIME, mean_trace, color=color, linewidth=2)
        ax.fill_between(COMMON_TIME, mean_trace - sem_trace, mean_trace + sem_trace,
                        color=color, alpha=0.25)
    
    ax.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlim(0.5, 2.0)
    ax.set_ylim(y_lim)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ΔF/F')
    ax.set_title(f'Cluster {cid} (n={len(cluster_traces[cid])})')
    ax.grid(True, alpha=0.3)

# Remove unused subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.suptitle('Mean Traces by Cluster', fontsize=14, fontweight='bold', y=1.02)

output_file = OUTPUT_DIR / "Fig3d_cluster_mean_traces.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster mean traces to {output_file}")

#### D.5.e Superposition of cluster with same PPR but different amplitude values (Cluster 3 and 5)

In [ ]:
# Superpose mean traces for Cluster 3 and Cluster 5

# Get traces for each cluster
cluster3_traces = []
cluster5_traces = []

for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    bouton_id = row['ID']
    cluster_id = row['HC_Cluster']
    if bouton_id in resampled_trace_lookup:
        if cluster_id == 3:
            cluster3_traces.append(resampled_trace_lookup[bouton_id]['Avg'])
        elif cluster_id == 5:
            cluster5_traces.append(resampled_trace_lookup[bouton_id]['Avg'])

# Calculate means and SEMs
cluster3_matrix = np.column_stack(cluster3_traces)
cluster5_matrix = np.column_stack(cluster5_traces)

cluster3_mean = np.nanmean(cluster3_matrix, axis=1)
cluster3_sem = np.nanstd(cluster3_matrix, axis=1, ddof=1) / np.sqrt(cluster3_matrix.shape[1])

cluster5_mean = np.nanmean(cluster5_matrix, axis=1)
cluster5_sem = np.nanstd(cluster5_matrix, axis=1, ddof=1) / np.sqrt(cluster5_matrix.shape[1])

# Plot superposition
fig, ax = plt.subplots(figsize=(10, 6))

color3 = get_cluster_color(3)
color5 = get_cluster_color(5)

ax.plot(COMMON_TIME, cluster3_mean, color=color3, linewidth=2, 
        label=f'Cluster 3 (n={len(cluster3_traces)})')
ax.fill_between(COMMON_TIME, cluster3_mean - cluster3_sem, cluster3_mean + cluster3_sem,
                color=color3, alpha=0.25)

ax.plot(COMMON_TIME, cluster5_mean, color=color5, linewidth=2, 
        label=f'Cluster 5 (n={len(cluster5_traces)})')
ax.fill_between(COMMON_TIME, cluster5_mean - cluster5_sem, cluster5_mean + cluster5_sem,
                color=color5, alpha=0.25)

ax.axhline(0, color='gray', linestyle='dotted', linewidth=1)
ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
ax.set_xlim(0.5, 2.0)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔF/F')
ax.set_title('Mean Traces: Cluster 3 vs Cluster 5')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
output_file = OUTPUT_DIR / "Fig3i_cluster3_vs_cluster5_mean_traces.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to {output_file}")
print(f"  Cluster 3: {len(cluster3_traces)} traces")
print(f"  Cluster 5: {len(cluster5_traces)} traces")

## Chapter E - Diveristy along a fiber

Chapter E analyses the axonal organization, examining how bouton classes distribute along individual fibers.


### E.1 Fiber-Level Diversity

Boutons are regrouped by axonal fiber to quantify how many distinct clusters appear along individual fibers. Assessing intra-fiber heterogeneity sheds light on whether structural units host multiple functional bouton classes.


In [ ]:
# Analyze cluster diversity within individual fibers (boutons from same axon)

def extract_fiber_id(bouton_id):
    """Extract fiber ID from bouton ID (first 22 characters)."""
    return str(bouton_id)[:22]

def analyze_fiber_diversity(min_boutons_per_fiber=4):
    """Analyze how clusters are distributed within individual fibers."""
    
    # Extract fiber IDs and count boutons per fiber
    fiber_data             = PCA_Data_WT_Pooled_clustered.copy()
    fiber_data['Fiber_ID'] = fiber_data['ID'].apply(extract_fiber_id)
    
    # Count boutons per fiber
    fiber_bouton_counts = fiber_data.groupby('Fiber_ID').size()
    
    # Filter fibers with sufficient boutons
    valid_fibers  = fiber_bouton_counts[fiber_bouton_counts >= min_boutons_per_fiber].index
    filtered_data = fiber_data[fiber_data['Fiber_ID'].isin(valid_fibers)]
    
    print(f"Fiber analysis: {len(valid_fibers)} fibers with {min_boutons_per_fiber}+ boutons")
    print(f"Total boutons analyzed: {len(filtered_data)}")
    
    return filtered_data, valid_fibers

def calculate_cluster_diversity(filtered_data):
    """Calculate number of different clusters per fiber."""
    fiber_diversity = {}
    
    for fiber_id in filtered_data['Fiber_ID'].unique():
        fiber_boutons        = filtered_data[filtered_data['Fiber_ID'] == fiber_id]
        unique_clusters      = fiber_boutons['HC_Cluster'].nunique()
        total_boutons        = len(fiber_boutons)
        cluster_distribution = fiber_boutons['HC_Cluster'].value_counts(normalize=True)
        
        fiber_diversity[fiber_id] = {
            'num_cluster_types': unique_clusters,
            'total_boutons': total_boutons,
            'cluster_props': cluster_distribution.to_dict()
        }
    
    return fiber_diversity

# Main analysis
filtered_data, valid_fibers = analyze_fiber_diversity(min_boutons_per_fiber=4)
fiber_diversity             = calculate_cluster_diversity(filtered_data)

# Organize data by diversity level
diversity_categories = {'1': [], '2': [], '3': [], '4+': []}
for fiber_id, info in fiber_diversity.items():
    num_types = info['num_cluster_types']
    category  = str(num_types) if num_types <= 3 else '4+'
    diversity_categories[category].append(info)

# Calculate average cluster proportions for each diversity category
diversity_means  = {}
diversity_counts = {}

for category, fiber_list in diversity_categories.items():
    diversity_counts[category] = len(fiber_list)
    
    if len(fiber_list) > 0:
        # Calculate mean proportion for each cluster
        cluster_means = {}
        for cluster_id in range(1, N_CLUSTERS + 1):
            proportions               = [fiber['cluster_props'].get(cluster_id, 0) for fiber in fiber_list]
            cluster_means[cluster_id] = np.mean(proportions)
        diversity_means[category] = cluster_means
    else:
        diversity_means[category] = {i: 0 for i in range(1, N_CLUSTERS + 1)}

# Create stacked bar plot
fig, ax = plt.subplots(figsize=(10, 6))

diversity_order = ['1', '2', '3', '4+']
x_positions     = np.arange(len(diversity_order))
bottom_values   = np.zeros(len(diversity_order))

total_fibers    = len(valid_fibers)

# Plot stacked bars
for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_heights = []
    for category in diversity_order:
        # Height = (fibers in category / total fibers) * 100 * average cluster proportion
        fiber_percentage   = (diversity_counts[category] / total_fibers) * 100
        cluster_proportion = diversity_means[category][cluster_id]
        height             = fiber_percentage * cluster_proportion
        cluster_heights.append(height)
    
    ax.bar(x_positions, cluster_heights, bottom=bottom_values, 
           color=get_cluster_color(cluster_id), label=f'Cluster {cluster_id}', alpha=0.8)
    bottom_values += cluster_heights

# Add fiber count labels
for i, category in enumerate(diversity_order):
    count      = diversity_counts[category]
    percentage = (count / total_fibers) * 100
    ax.text(i, percentage + 1, f'n={count}\n({percentage:.1f}%)', 
            ha='center', va='bottom', fontweight='bold', fontsize=9)

# Format plot
ax.set_xlabel('Number of Cluster Types per Fiber')
ax.set_ylabel('Percentage of Fibers (%)')
ax.set_title(f'Fiber Cluster Diversity (Fibers with 4+ Boutons)\nTotal: {total_fibers} fibers')
ax.set_xticks(x_positions)
ax.set_xticklabels(diversity_order)
ax.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()

output_file = OUTPUT_DIR / "15_fiber_cluster_diversity_analysis.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics
print(f"\n=== FIBER CLUSTER DIVERSITY SUMMARY ===")
for category in diversity_order:
    count = diversity_counts[category]
    if count > 0:
        percentage = (count / total_fibers) * 100
        print(f"\nFibers with {category} cluster type(s): {count} ({percentage:.1f}%)")
        
        # Show cluster composition
        for cluster_id in range(1, N_CLUSTERS + 1):
            prop = diversity_means[category][cluster_id]
            if prop > 0.05:  # Only show clusters with >5% average proportion
                print(f"  Cluster {cluster_id}: {prop:.1%} average proportion")

# Identify most diverse fibers
diverse_fibers = [fid for fid, info in fiber_diversity.items() if info['num_cluster_types'] >= 3]
if diverse_fibers:
    print(f"\nMost diverse fibers (3+ cluster types): {len(diverse_fibers)} fibers")
    for fiber_id in diverse_fibers[:5]:  # Show first 5
        info = fiber_diversity[fiber_id]
        clusters = list(info['cluster_props'].keys())
        print(f"  {fiber_id}: {info['num_cluster_types']} clusters ({clusters})")

print(f"\n✓ Saved fiber diversity analysis to {output_file}")


### E.2 Representative Fiber Dynamics

For a selected fiber, raw and smoothed traces are visualized to showcase how preprocessing captures the essential synaptic waveform while reducing noise. This example grounds the fiber-level analysis in concrete data.


In [ ]:
# Create trace lookup from resampled data
resampled_trace_lookup = {}
for _, trace_row in NORM_TRACES_DATAFRAME.iterrows():
    resampled_trace_lookup[trace_row['ID']] = {
        'Time': trace_row['Time'],
        'Avg': trace_row['Avg']
    }

# Analyze traces from a specific fiber (raw vs smoothed)
from scipy.signal import savgol_filter

def analyze_single_fiber(fiber_prefix, window_length=9, poly_order=2):
    """Analyze all boutons from a specific fiber."""
    
    # Find boutons from this fiber
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(fiber_prefix)]
    
    if len(fiber_boutons) == 0:
        print(f"No boutons found with prefix '{fiber_prefix}'")
        return
    
    print(f"Found {len(fiber_boutons)} boutons from fiber '{fiber_prefix}'")
    
    # Get trace data for these boutons
    fiber_traces = {}
    for _, bouton in fiber_boutons.iterrows():
        bouton_id = bouton['ID']
        if bouton_id in resampled_trace_lookup:
            trace_data = resampled_trace_lookup[bouton_id]['Avg']
            cluster_id = bouton['HC_Cluster']
            fiber_traces[bouton_id] = {
                'raw': trace_data,
                'cluster': cluster_id
            }
    
    if not fiber_traces:
        print(f"No trace data found for fiber '{fiber_prefix}'")
        return
    
    # Apply smoothing
    for bouton_id in fiber_traces:
        raw_trace = fiber_traces[bouton_id]['raw']
        
        # Handle NaN values - interpolate or skip
        if np.all(np.isnan(raw_trace)):
            # All NaN, just copy
            fiber_traces[bouton_id]['smoothed'] = raw_trace.copy()
            continue
        
        # Create a copy for smoothing
        trace_for_smoothing = raw_trace.copy()
        
        # Interpolate NaN values for smoothing
        nan_mask = np.isnan(trace_for_smoothing)
        if np.any(nan_mask):
            # Get valid indices and values
            valid_indices = np.where(~nan_mask)[0]
            valid_values = trace_for_smoothing[~nan_mask]
            
            if len(valid_indices) < 3:
                # Not enough valid points, just copy
                fiber_traces[bouton_id]['smoothed'] = raw_trace.copy()
                continue
            
            # Interpolate NaN values
            nan_indices = np.where(nan_mask)[0]
            trace_for_smoothing[nan_mask] = np.interp(nan_indices, valid_indices, valid_values)
        
        # Adjust window length if needed
        win_len = min(window_length, len(trace_for_smoothing))
        if win_len % 2 == 0:  # Must be odd
            win_len -= 1
        if win_len < 3:
            smoothed_trace = trace_for_smoothing.copy()
        else:
            smoothed_trace = savgol_filter(trace_for_smoothing, window_length=win_len, 
                                         polyorder=min(poly_order, win_len-1), mode='interp')
        
        # Restore NaN values in the smoothed output where original had NaN
        if np.any(nan_mask):
            smoothed_trace[nan_mask] = np.nan
        
        fiber_traces[bouton_id]['smoothed'] = smoothed_trace
    
    return fiber_traces

def plot_fiber_traces(fiber_traces, fiber_prefix):
    """Plot raw vs smoothed traces for a fiber."""
    if not fiber_traces:
        return
    
    n_boutons = len(fiber_traces)
    
    # Create subplot layout (2 columns: raw, smoothed)
    fig, axes = plt.subplots(n_boutons, 2, figsize=(10, min(20, 2*n_boutons)), 
                            sharex=True, sharey=True)
    
    if n_boutons == 1:
        axes = axes.reshape(1, -1)
    
    # Calculate common y-limits
    all_values = []
    for data in fiber_traces.values():
        all_values.extend(data['raw'][np.isfinite(data['raw'])])
        all_values.extend(data['smoothed'][np.isfinite(data['smoothed'])])
    
    if all_values:
        y_min, y_max = np.min(all_values), np.max(all_values)
        y_padding = (y_max - y_min) * 0.05
        y_lims = (y_min - y_padding, y_max + y_padding)
    else:
        y_lims = (-0.5, 0.5)
    
    # Plot each bouton
    for idx, (bouton_id, data) in enumerate(fiber_traces.items()):
        cluster_id = data['cluster']
        color = get_cluster_color(cluster_id)
        
        # Raw trace (left)
        ax_raw = axes[idx, 0]
        ax_raw.plot(COMMON_TIME, data['raw'], color=color, linewidth=1.2)
        ax_raw.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_raw.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_raw.set_xlim(0.5, 2.0)
        ax_raw.set_ylim(y_lims)
        ax_raw.set_ylabel('ΔF/F', fontsize=8)
        ax_raw.set_title(f'{bouton_id} (Cluster {cluster_id})\nRaw', fontsize=9, loc='left')
        ax_raw.tick_params(labelsize=7)
        
        # Smoothed trace (right)
        ax_smooth = axes[idx, 1]
        ax_smooth.plot(COMMON_TIME, data['smoothed'], color=color, linewidth=1.2)
        ax_smooth.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_smooth.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        ax_smooth.set_xlim(0.5, 2.0)
        ax_smooth.set_ylim(y_lims)
        ax_smooth.set_title('Savitzky-Golay Smoothed', fontsize=9, loc='left')
        ax_smooth.tick_params(labelsize=7)
    
    # Add x-axis labels to bottom row
    axes[-1, 0].set_xlabel('Time (s)', fontsize=8)
    axes[-1, 1].set_xlabel('Time (s)', fontsize=8)
    
    fig.suptitle(f'Single Fiber Analysis: {fiber_prefix} (n={n_boutons})', fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"16_fiber_traces_{safe_prefix}_raw_vs_smoothed.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

def plot_fiber_ppr_profiles(fiber_boutons, fiber_prefix):
    """Plot PPR profiles for boutons from the same fiber."""
    
    # Get PPR data
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in fiber_boutons.columns]
    if not ppr_cols:
        print("No PPR columns found")
        return
    
    pulse_numbers = list(range(1, len(ppr_cols) + 2))  # Include pulse 1
    
    plt.figure(figsize=(8, 5))
    
    # Plot each bouton's PPR profile
    for _, bouton in fiber_boutons.iterrows():
        cluster_id = bouton['HC_Cluster']
        color = get_cluster_color(cluster_id)
        
        # Get PPR values (start with 1.0 for pulse 1)
        ppr_values = [1.0] + bouton[ppr_cols].tolist()
        
        plt.plot(pulse_numbers, ppr_values, color=color, alpha=0.8, linewidth=2, 
                marker='o', markersize=4, label=f'Cluster {cluster_id}')
    
    plt.axhline(1.0, color='gray', linestyle='--', linewidth=1)
    plt.xlabel('Pulse Number')
    plt.ylabel('PPR (A_n/A_1)')
    plt.title(f'PPR Profiles: {fiber_prefix} (n={len(fiber_boutons)})')
    plt.grid(True, alpha=0.3)
    plt.xticks(pulse_numbers)
    
    # Only show legend if multiple clusters
    if fiber_boutons['HC_Cluster'].nunique() > 1:
        plt.legend()
    
    plt.tight_layout()
    
    # Save figure
    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"17_fiber_ppr_{safe_prefix}_profiles.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return output_file

# Example usage - change the prefix to analyze different fibers
#FIBER_PREFIX = "241212_Fibre2_PortionA_"  # Change this to your fiber of interest
FIBER_PREFIX = "20220425_linescan1_20Hz_"  # Change this to your fiber of interest

# Run analysis
fiber_traces = analyze_single_fiber(FIBER_PREFIX)

if fiber_traces:
    # Plot traces
    trace_output = plot_fiber_traces(fiber_traces, FIBER_PREFIX)
    
    # Plot PPR profiles
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(FIBER_PREFIX)]
    ppr_output    = plot_fiber_ppr_profiles(fiber_boutons, FIBER_PREFIX)
    
    print(f"✓ Fiber analysis complete:")
    print(f"  Traces: {trace_output}")
    print(f"  PPR profiles: {ppr_output}")
    
    # Summary statistics
    cluster_distribution = fiber_boutons['HC_Cluster'].value_counts().sort_index()
    print(f"\nFiber cluster composition:")
    for cluster_id, count in cluster_distribution.items():
        print(f"  Cluster {cluster_id}: {count} boutons")

else:
    print(f"No analysis possible for fiber '{FIBER_PREFIX}'")
    
    # Show available fiber prefixes
    available_prefixes = PCA_Data_WT_Pooled_clustered['ID'].str[:25].value_counts()
    print("\nAvailable fiber prefixes (showing top 10):")
    for prefix, count in available_prefixes.head(10).items():
        if count >= 3:  # Only show fibers with multiple boutons
            print(f"  '{prefix}': {count} boutons")


## Chapter F - Post-synaptic element identification

Chapter F associates datapoints to their target, either Purkinje, Interneurons or Undefined.

### F.1 Target Projection

By coloring the PCA scatter with Purkinje versus interneuron labels, we inspect whether synaptic target identity explains variance captured by the first components. This biological overlay links statistical clusters back to anatomical classes.


In [ ]:
# Scatter plot: WT pooled points with PC (red) and IN (blue) overlays
# Assumes pca_coordinates (np.ndarray), PCA_Data_WT_Pooled (DataFrame) and PCA_RESULTS are available

plt.figure(figsize=(9, 7))

# Plot original PCA
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
            c=get_cluster_colors(cluster_assignments),
            s=50, marker='o', linewidths=0.6,
            label='WT pooled (2.5mM Ca)', alpha=0.4)


# Target masks
targets = PCA_Data_WT_Pooled['Target'].values
pc_mask = targets == 'PC'
in_mask = targets == 'IN'

# Calculate medians for PC and IN groups
pc_coordinates = pca_coordinates[pc_mask]
in_coordinates = pca_coordinates[in_mask]

pc_median = np.median(pc_coordinates, axis=0) if len(pc_coordinates) > 0 else None
in_median = np.median(in_coordinates, axis=0) if len(in_coordinates) > 0 else None

# Calculate distance between medians
if pc_median is not None and in_median is not None:
    median_distance = np.linalg.norm(pc_median - in_median)
else:
    median_distance = None

# Plot Purkinje Cells (PC)
if np.any(pc_mask):
    plt.scatter(pca_coordinates[pc_mask, 0], pca_coordinates[pc_mask, 1],
                c='red', marker='o', s=100, 
                edgecolors='black', linewidths=0.6,
                label=f'Purkinje Cells (PC, n={np.sum(pc_mask)})')

# Plot Interneurons (IN)
if np.any(in_mask):
    plt.scatter(pca_coordinates[in_mask, 0], pca_coordinates[in_mask, 1],
                c='blue', marker='^', s=100, 
                edgecolors='black', linewidths=0.6,
                label=f'Interneurons (IN, n={np.sum(in_mask)})')
    
# Plot medians and distance line
if pc_median is not None and in_median is not None:
    # Plot median points
    plt.scatter(pc_median[0], pc_median[1], c='darkred', marker='X', s=200, 
               label='PC Median', edgecolors='black', linewidth=2)
    plt.scatter(in_median[0], in_median[1], c='darkblue', marker='X', s=200, 
               label='IN Median', edgecolors='black', linewidth=2)
    
    # Draw line between medians
    plt.plot([pc_median[0], in_median[0]], [pc_median[1], in_median[1]], 
             color='black', linestyle='--', linewidth=2, alpha=0.8, 
             label=f'Median Distance: {median_distance:.3f}')
    
    # Annotate distance
    midpoint_x = (pc_median[0] + in_median[0]) / 2
    midpoint_y = (pc_median[1] + in_median[1]) / 2
    plt.annotate(f'd = {median_distance:.3f}', 
                xy=(midpoint_x, midpoint_y), 
                xytext=(midpoint_x + 0.5, midpoint_y + 0.5),
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))


# Axis labels with explained variance
pc1_var, pc2_var = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_var:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_var:.1%} variance)')
plt.xlim(-7, 10)
plt.ylim(-6, 6)
plt.title('PCA: WT pooled colored by Target (PC vs IN)')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and show
output_path = OUTPUT_DIR / "18_pca_pc_in_scatter.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved PCA PC/IN scatter to {output_path}")

# Summary statistics
print("=== PC vs IN SEPARATION ANALYSIS ===")
print(f"Purkinje Cells (PC): {np.sum(pc_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_mask):2d} samples")




if pc_median is not None and in_median is not None:
    print(f"PC median coordinates:     ({pc_median[0]:.3f}, {pc_median[1]:.3f})")
    print(f"IN median coordinates:     ({in_median[0]:.3f}, {in_median[1]:.3f})")
    print(f"Median-to-median distance: {median_distance:.3f}")
else:
    print("Cannot calculate median distance - insufficient data")



### F.2 Target-Aligned Trace Summaries A FAIRE MOYENNE VS TOUT

Average traces for Purkinje and interneuron boutons are contrasted here with consistent color assignments. Visualizing response dynamics per target type reveals how physiology underpins spatial separation in PCA space.


In [ ]:
# Extract PC and IN data
pc_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'PC'].copy()
in_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['Target'] == 'IN'].copy()

# Extract target coordinates and IDs
target_cell_types = PCA_Data_WT_Pooled['Target'].values
in_cell_mask      = target_cell_types == 'IN'
pc_cell_mask      = target_cell_types == 'PC'

in_coordinates = pca_coordinates[in_cell_mask]
in_ids         = PCA_Data_WT_Pooled.loc[in_cell_mask, 'ID'].tolist()

pc_coordinates = pca_coordinates[pc_cell_mask]
pc_ids         = PCA_Data_WT_Pooled.loc[pc_cell_mask, 'ID'].tolist()

# Ellipse tightness control (0.50 = loose, 0.95 = tight, 0.99 = very tight)
ELLIPSE_CONFIDENCE = 0.5

def fit_confidence_ellipse(points, confidence=0.95):
    """Fit confidence ellipse around points and return parameters."""
    from scipy.stats import chi2
    center = points.mean(axis=0)
    cov = np.cov(points.T)
    chi2_val = chi2.ppf(confidence, df=2)
    
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0]))
    width, height = 2 * np.sqrt(chi2_val * eigenvals)
    cov_inv = np.linalg.inv(cov)
    
    return center, cov_inv, chi2_val, (width, height, angle)

# Fit ellipse around SynII points
synii_points              = pca_data['SynII']
ellipse_params            = fit_confidence_ellipse(synii_points, ELLIPSE_CONFIDENCE)
center, cov_inv, chi2_val = ellipse_params[:3]

def classify_points_in_ellipse(points, center, cov_inv, chi2_threshold):
    """Return boolean mask for points inside ellipse."""
    inside_mask = []
    for pt in points:
        diff          = pt - center
        mahal_dist_sq = diff @ cov_inv @ diff.T
        inside_mask.append(mahal_dist_sq <= chi2_threshold)
    return np.array(inside_mask)

def compare_traces_inside_outside_improved(target_ids, inside_mask, target_name, target_color):
    """Compare mean traces for inside vs outside ellipse groups with better visibility."""
    inside_ids  = [target_ids[i] for i in range(len(target_ids)) if inside_mask[i]]
    outside_ids = [target_ids[i] for i in range(len(target_ids)) if not inside_mask[i]]
    
    # Get traces for each group
    inside_traces  = [resampled_trace_lookup[bid]['Avg'] for bid in inside_ids if bid in resampled_trace_lookup]
    outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in outside_ids if bid in resampled_trace_lookup]
    synii_traces   = [resampled_trace_lookup[bid]['Avg'] for bid in PCA_Data_SynII['ID'] if bid in resampled_trace_lookup]
    
    if not (inside_traces and outside_traces and synii_traces):
        print(f"Insufficient trace data for {target_name} comparison")
        return
    
    # Calculate means and SEMs
    inside_mean = np.nanmean(inside_traces, axis=0)
    inside_sem  = np.nanstd(inside_traces, axis=0, ddof=1) / np.sqrt(len(inside_traces))
    
    outside_mean = np.nanmean(outside_traces, axis=0)
    outside_sem  = np.nanstd(outside_traces, axis=0, ddof=1) / np.sqrt(len(outside_traces))
    
    synii_mean   = np.nanmean(synii_traces, axis=0)
    synii_sem    = np.nanstd(synii_traces, axis=0, ddof=1) / np.sqrt(len(synii_traces))
    
    # Create 3-panel figure
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # Panel 1: Inside ellipse with SEM
    inside_color = 'red' if target_name == 'IN' else 'green'
    ax1.plot(COMMON_TIME, inside_mean, color=inside_color, linewidth=1, 
             label=f'{target_name} inside ellipse (n={len(inside_traces)})')
    ax1.fill_between(COMMON_TIME, inside_mean-inside_sem, inside_mean+inside_sem, color=inside_color, alpha=0.3)
    ax1.axhline(0, color='black', linestyle='dotted', linewidth=1)
    ax1.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax1.set_xlim(0.5, 2.0)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('ΔF/F')
    ax1.set_title(f'{target_name} Inside Ellipse')
    ax1.legend()
    
    # Panel 2: Outside ellipse with SEM
    ax2.plot(COMMON_TIME, outside_mean, color='black', linewidth=1,
             label=f'{target_name} outside ellipse (n={len(outside_traces)})')
    ax2.fill_between(COMMON_TIME, outside_mean-outside_sem, outside_mean+outside_sem, color='black', alpha=0.3)
    ax2.axhline(0, color='black', linestyle='dotted', linewidth=1)
    ax2.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax2.set_xlim(0.5, 2.0)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ΔF/F')
    ax2.set_title(f'{target_name} Outside Ellipse')
    ax2.legend()
    
    # Panel 3: Overlay comparison
    ax3.plot(COMMON_TIME, inside_mean, color=inside_color, linewidth=1, 
             label=f'Inside (n={len(inside_traces)})')
    ax3.plot(COMMON_TIME, outside_mean, color='black', linewidth=1,
             label=f'Outside (n={len(outside_traces)})')
    ax3.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax3.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_xlim(0.5, 2.0)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('ΔF/F')
    ax3.set_title(f'{target_name} Inside vs Outside')
    ax3.legend()
    
    # Remove top and right spines, remove grids
    for ax in [ax1, ax2, ax3]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / f"{target_name.lower()}_inside_outside_ellipse_traces_improved.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ {target_name} ellipse analysis:")
    print(f"  Inside: {len(inside_traces)} traces ({len(inside_traces)/len(target_ids)*100:.1f}%)")
    print(f"  Outside: {len(outside_traces)} traces ({len(outside_traces)/len(target_ids)*100:.1f}%)")
    
    return output_file

# Run improved analysis for both cell types
if len(in_ids) > 0:
    in_inside_mask = classify_points_in_ellipse(in_coordinates, center, cov_inv, chi2_val)
    compare_traces_inside_outside_improved(in_ids, in_inside_mask, '19_IN', 'red')

if len(pc_ids) > 0:
    pc_inside_mask = classify_points_in_ellipse(pc_coordinates, center, cov_inv, chi2_val)
    compare_traces_inside_outside_improved(pc_ids, pc_inside_mask, '20_PC', 'mediumseagreen')

### F.3 Cluster Composition Comparison

In [ ]:
# Compare cluster distributions between Purkinje Cells, Interneurons, and overall WT
target_cell_types = PCA_Data_WT_Pooled_clustered['Target'].values
pc_mask = target_cell_types == 'PC'
in_mask = target_cell_types == 'IN'

# Get cluster assignments for each target type
pc_cluster_assignments = cluster_assignments[pc_mask]
in_cluster_assignments = cluster_assignments[in_mask]
wt_cluster_assignments = cluster_assignments  # All WT pooled

# Count clusters for each group
pc_cluster_counts = pd.Series(pc_cluster_assignments).value_counts().sort_index()
in_cluster_counts = pd.Series(in_cluster_assignments).value_counts().sort_index()
wt_cluster_counts = pd.Series(wt_cluster_assignments).value_counts().sort_index()

# Ensure all clusters represented
all_clusters = sorted(range(1, N_CLUSTERS + 1))
pc_complete = pd.Series([pc_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
in_complete = pd.Series([in_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
wt_complete = pd.Series([wt_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
pc_percentages = 100 * pc_complete / len(pc_cluster_assignments)
in_percentages = 100 * in_complete / len(in_cluster_assignments)
wt_percentages = 100 * wt_complete / len(wt_cluster_assignments)

# Stacked bar plot with 3 bars
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_pc = bottom_in = bottom_wt = 0
for cluster_id in all_clusters:
    color = get_cluster_color(cluster_id)
    pc_pct = pc_percentages.iloc[cluster_id - 1]
    in_pct = in_percentages.iloc[cluster_id - 1]
    wt_pct = wt_percentages.iloc[cluster_id - 1]

    # PC bar
    ax.bar(0, pc_pct, bar_width, bottom=bottom_pc, color=color,
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    if pc_pct > 3:
        ax.text(0, bottom_pc + pc_pct/2, f"{pc_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9, fontweight='bold')
    bottom_pc += pc_pct

    # IN bar
    ax.bar(1, in_pct, bar_width, bottom=bottom_in, color=color)
    if in_pct > 3:
        ax.text(1, bottom_in + in_pct/2, f"{in_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9, fontweight='bold')
    bottom_in += in_pct
    
    # WT bar
    ax.bar(2, wt_pct, bar_width, bottom=bottom_wt, color=color)
    if wt_pct > 3:
        ax.text(2, bottom_wt + wt_pct/2, f"{wt_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9, fontweight='bold')
    bottom_wt += wt_pct

# Add sample counts
ax.text(0, 102, f"n={len(pc_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(1, 102, f"n={len(in_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(2, 102, f"n={len(wt_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: PC vs IN vs WT Pooled')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Purkinje Cells', 'Interneurons', 'WT Pooled'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
ax.legend(handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "21_pc_vs_in_vs_wt_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")
print(f"PC distribution: {dict(pc_cluster_counts)}")
print(f"IN distribution: {dict(in_cluster_counts)}")
print(f"WT distribution: {dict(wt_cluster_counts)}")


## Chapter G – SynII 

Chapter G projects SynII boutons into the WT-defined manifold, contrasts their properties.


### G.1 SynII Projection

SynII bouton features are projected into the WT-derived PCA space to assess how the mutant dataset occupies the existing manifold. This shared embedding enables direct comparison between SynII and WT boutons.


In [ ]:
# Project SynII data onto WT-trained PCA space
plt.figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors
plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')

# Plot SynII projected data
synii_pca_coords = pca_data['SynII']
plt.scatter(synii_pca_coords[:, 0], synii_pca_coords[:, 1],
           marker='*', s=100, c='darkred', alpha=0.8,
           edgecolors='white', linewidth=1, label=f'SynII KO (n={len(synii_pca_coords)})')

# Calculate and plot SynII centroid
synii_centroid = np.mean(synii_pca_coords, axis=0)
plt.scatter(synii_centroid[0], synii_centroid[1],
           marker='X', s=100, c='red', linewidth=1,
           label=f'SynII Centroid ({synii_centroid[0]:.2f}, {synii_centroid[1]:.2f})', zorder=10)

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)                    
plt.ylim(-6, 6)
plt.title('SynII KO Projection onto WT PCA Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "Fig6e_synii_pca_projection.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ SynII centroid: PC1={synii_centroid[0]:.3f}, PC2={synii_centroid[1]:.3f}")
print(f"✓ Saved to {output_file}")


#### G.1.a Control SynII per fiber

In [ ]:
# Project SynII data onto WT-trained PCA space with ID-based coloring
plt.figure(figsize=(8, 6))

# # Plot WT pooled data with cluster colors
# plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
#            c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')

# Get SynII bouton IDs and extract first 14 characters
synii_ids = PCA_Data_SynII['ID'].values
synii_prefixes = [str(bouton_id)[:14] for bouton_id in synii_ids]

# Create unique prefix-to-color mapping with Set2 colormap
unique_prefixes = sorted(set(synii_prefixes))
set2_colormap = plt.get_cmap('Set1')
specified_colors = [set2_colormap(i) for i in range(len(unique_prefixes))]
prefix_color_map = dict(zip(unique_prefixes, specified_colors))

# Plot SynII points grouped by prefix
synii_pca_coords = pca_data['SynII']
for prefix in unique_prefixes:
    mask = np.array([p == prefix for p in synii_prefixes])
    plt.scatter(synii_pca_coords[mask, 0], synii_pca_coords[mask, 1],
               marker='*', s=100, c=[prefix_color_map[prefix]], 
               linewidth=1, label=f'SynII {prefix}')

# Format plot
pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
plt.xlim(-7, 10)                    
plt.ylim(-6, 6)
plt.title('SynII KO Projection onto WT PCA Space (colored by ID prefix)')
plt.legend(loc='best', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "23_synii_pca_projection_by_prefix.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ Found {len(unique_prefixes)} unique ID prefixes: {unique_prefixes}")
print(f"✓ Saved to {output_file}")


### G.2 Cluster Attribution for SynII

Using distances in PCA space, SynII boutons are assigned to the nearest WT-derived clusters. This provides a principled way to translate the WT clustering schema onto the mutant data.


In [ ]:
# Assign SynII samples to WT-derived clusters using PCA space
def assign_clusters_robust(X_existing, labels_existing, X_new, method='centroid'):
    """Assign new samples to existing clusters using specified linkage method."""
    unique_labels = np.unique(labels_existing)
    
    if method == 'centroid':
        centroids = np.vstack([X_existing[labels_existing == lbl].mean(axis=0) for lbl in unique_labels])
        distances = cdist(X_new, centroids)
        return unique_labels[np.argmin(distances, axis=1)]
    
    elif method == 'single':
        dist_matrix = np.empty((X_new.shape[0], len(unique_labels)))
        for j, lbl in enumerate(unique_labels):
            cluster_points = X_existing[labels_existing == lbl]
            dist_matrix[:, j] = np.min(cdist(X_new, cluster_points), axis=1)
        return unique_labels[np.argmin(dist_matrix, axis=1)]
    
    else:
        raise ValueError("method must be 'centroid' or 'single'")

# Assign SynII samples to WT clusters
synii_cluster_assignments = assign_clusters_robust(pca_coordinates, cluster_assignments, 
                                                   pca_data['SynII'], method='single')

print(f"✓ Assigned {len(synii_cluster_assignments)} SynII samples to WT clusters")
print(f"SynII cluster distribution: {dict(pd.Series(synii_cluster_assignments).value_counts().sort_index())}")

### G.3 Cluster Composition Comparison

Cluster membership counts for WT and SynII boutons are contrasted to reveal which bouton phenotypes expand or diminish in the mutant. The resulting bar plots offer a population-level view of SynII remodeling.


In [ ]:
# Compare cluster distributions between WT and SynII
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
synii_cluster_counts = pd.Series(synii_cluster_assignments).value_counts()

# Ensure all clusters represented
all_clusters = sorted(wt_cluster_counts.index)
synii_complete = pd.Series([synii_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)
synii_percentages = 100 * synii_complete / len(synii_cluster_assignments)

# Stacked bar plot
fig, ax = plt.subplots(figsize=(8, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_wt = bottom_synii = 0
for cluster_id in all_clusters:
    color = get_cluster_color(cluster_id)
    wt_pct = wt_percentages.iloc[cluster_id - 1]
    synii_pct = synii_percentages.iloc[cluster_id - 1]

    # WT bar
    ax.bar(0, wt_pct, bar_width, bottom=bottom_wt, color=color,
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    if wt_pct > 3:  # Only label if segment is large enough
        ax.text(0, bottom_wt + wt_pct/2, f"{wt_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9)
    bottom_wt += wt_pct

    # SynII bar
    ax.bar(1, synii_pct, bar_width, bottom=bottom_synii, color=color, alpha=0.7)
    if synii_pct > 3:
        ax.text(1, bottom_synii + synii_pct/2, f"{synii_pct:.1f}%", ha='center', va='center',
                color=get_text_color(color), fontsize=9)
    bottom_synii += synii_pct

# Add sample counts
ax.text(0, 102, f"n={len(cluster_assignments)}", ha='center', va='bottom', fontweight='bold')
ax.text(1, 102, f"n={len(synii_cluster_assignments)}", ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: WT vs SynII')
ax.set_xticks([0, 1])
ax.set_xticklabels(['WT', 'SynII'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
ax.legend(handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "24_wt_vs_synii_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")


### G.4 SynII Descriptive Statistics

Summary statistics for SynII amplitudes and failure rates are computed to contextualize the mutant population. Reporting mean and standard deviation for key metrics helps quantify how SynII boutons diverge from WT benchmarks.


In [ ]:
# Calculate mean ± SD for Amp1, Amp2, and %Fail1 in SynII data
print("Mean ± SD for SynII data:")
print(f"AMP1: {PCA_Data_SynII['AMP1'].mean():.3f} ± {PCA_Data_SynII['AMP1'].std():.3f}")
print(f"AMP2: {PCA_Data_SynII['AMP2'].mean():.3f} ± {PCA_Data_SynII['AMP2'].std():.3f}")
print(f"%Fail1: {PCA_Data_SynII['%Fail1'].mean():.3f} ± {PCA_Data_SynII['%Fail1'].std():.3f}")

### G.5 SynII-Enriched Clusters

This analysis identifies clusters disproportionately populated by SynII boutons and extracts the associated traces. Spotlighting these groups isolates the synaptic phenotypes most affected by the SynII mutation.


In [ ]:
# Identify and analyze SynII-enriched clusters
enriched_clusters = []
for cluster_id in all_clusters:
    wt_pct    = wt_percentages[cluster_id]
    synii_pct = synii_percentages[cluster_id]
    if synii_pct > wt_pct:
        enriched_clusters.append(cluster_id)
        print(f"Cluster {cluster_id}: WT={wt_pct:.1f}%, SynII={synii_pct:.1f}% (enriched)")

if enriched_clusters:
    # PPR profile comparison for enriched clusters
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
    x_pulses = list(range(1, len(ppr_cols)+2))
    
    plt.figure(figsize=(10, 6))
    
    # Plot enriched WT clusters
    for cluster_id in enriched_clusters:
        cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_id]
        means        = [1] + cluster_data[ppr_cols].mean().tolist()
        sems         = [0] + cluster_data[ppr_cols].sem().tolist()
        color        = get_cluster_color(cluster_id)
        
        plt.plot(x_pulses, means, marker='o', label=f'WT Cluster {cluster_id} (n={len(cluster_data)})', 
                color=color, linewidth=2)
        plt.fill_between(x_pulses, np.array(means)-np.array(sems), np.array(means)+np.array(sems), 
                        alpha=0.2, color=color)
    
    # Add SynII profile
    synii_means = [1] + PCA_Data_SynII[ppr_cols].mean().tolist()
    synii_sems  = [0] + PCA_Data_SynII[ppr_cols].sem().tolist()
    plt.plot(x_pulses, synii_means, marker='s', label=f'SynII KO (n={len(PCA_Data_SynII)})', 
            color='red', linewidth=1)
    plt.fill_between(x_pulses, np.array(synii_means)-np.array(synii_sems), 
                    np.array(synii_means)+np.array(synii_sems), alpha=0.2, color='red')
    
    plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    plt.ylabel('Mean PPR (A_n/A_1)')
    plt.xlabel('Pulse Number')
    plt.xticks(x_pulses)
    plt.title(f'PPR Profiles: SynII-Enriched WT Clusters vs SynII KO')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "25_synii_enriched_clusters_ppr_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Saved enriched clusters comparison to {output_file}")
else:
    print("No SynII-enriched clusters found")

# SynII summary statistics
print(f"\nSynII Summary Statistics:")
for param in ['AMP1', 'AMP2', '%Fail1']:
    if param in PCA_Data_SynII.columns:
        mean_val = PCA_Data_SynII[param].mean()
        std_val  = PCA_Data_SynII[param].std()
        print(f"{param}: {mean_val:.3f} ± {std_val:.3f}")

### G.6 SynII vs WT Trace Overlays

Mean traces from SynII-enriched clusters are compared against their WT counterparts to visualize how response kinetics shift in the mutant. These overlays tie population statistics back to time-domain dynamics.


In [ ]:
# Compare mean traces between SynII and WT with cluster assignments

# Ensure SynII has cluster assignments
if 'cluster_synII' not in PCA_Data_SynII.columns:
    PCA_Data_SynII['cluster_synII'] = synii_cluster_assignments

# Extract SynII and WT traces from resampled data
synii_traces = [resampled_trace_lookup[bid]['Avg'] for bid in PCA_Data_SynII['ID'] if bid in resampled_trace_lookup]
wt_traces    = []

for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    bouton_id = row['ID']
    if bouton_id in resampled_trace_lookup:
        wt_traces.append(resampled_trace_lookup[bouton_id]['Avg'])

if len(synii_traces) > 0 and len(wt_traces) > 0:
    # Calculate mean and SEM for each group
    synii_matrix = np.column_stack(synii_traces)
    wt_matrix    = np.column_stack(wt_traces)
    
    synii_mean   = np.nanmean(synii_matrix, axis=1)
    synii_sem    = np.nanstd(synii_matrix, axis=1, ddof=1) / np.sqrt(synii_matrix.shape[1])
    
    wt_mean      = np.nanmean(wt_matrix, axis=1)
    wt_sem       = np.nanstd(wt_matrix, axis=1, ddof=1) / np.sqrt(wt_matrix.shape[1])
    
    # Calculate y-limits for consistent scaling
    all_values = np.concatenate([
        synii_mean - synii_sem, synii_mean + synii_sem,
        wt_mean - wt_sem, wt_mean + wt_sem
    ])
    y_min, y_max = np.nanmin(all_values), np.nanmax(all_values)
    y_padding    = (y_max - y_min) * 0.05
    
    # Create comparison plot with cropped time axis (3 subplots)
    fig, (ax_synii, ax_wt, ax_overlay) = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    
    # SynII panel
    ax_synii.plot(COMMON_TIME, synii_mean, color='red', linewidth=2, 
                  label=f'SynII KO (n={len(synii_traces)})')
    ax_synii.fill_between(COMMON_TIME, synii_mean - synii_sem, synii_mean + synii_sem, 
                          color='red', alpha=0.25)
    
    ax_synii.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_synii.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_synii.set_xlim(0.5, 2.0)
    ax_synii.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_synii.set_title('SynII KO - Average Response')
    ax_synii.set_ylabel('ΔF/F')
    ax_synii.set_xlabel('Time (s)')
    ax_synii.legend()
    ax_synii.grid(True, alpha=0.3)
    
    # WT panel
    ax_wt.plot(COMMON_TIME, wt_mean, color='black', linewidth=2, 
               label=f'WT (n={len(wt_traces)})')
    ax_wt.fill_between(COMMON_TIME, wt_mean - wt_sem, wt_mean + wt_sem, 
                       color='black', alpha=0.25)
    ax_wt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_wt.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_wt.set_xlim(0.5, 2.0)
    ax_wt.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_wt.set_title('WT - Average Response')
    ax_wt.set_xlabel('Time (s)')
    ax_wt.legend()
    ax_wt.grid(True, alpha=0.3)
    
    # Overlay panel
    ax_overlay.plot(COMMON_TIME, wt_mean, color='black', linewidth=2, 
                    label=f'WT (n={len(wt_traces)})')
    ax_overlay.fill_between(COMMON_TIME, wt_mean - wt_sem, wt_mean + wt_sem, 
                            color='black', alpha=0.2)
    ax_overlay.plot(COMMON_TIME, synii_mean, color='red', linewidth=2, 
                    label=f'SynII KO (n={len(synii_traces)})')
    ax_overlay.fill_between(COMMON_TIME, synii_mean - synii_sem, synii_mean + synii_sem, 
                            color='red', alpha=0.2)
    ax_overlay.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax_overlay.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Stimulus')
    ax_overlay.set_xlim(0.5, 2.0)
    ax_overlay.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_overlay.set_title('Overlay - SynII vs WT')
    ax_overlay.set_xlabel('Time (s)')
    ax_overlay.legend()
    ax_overlay.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "26_synii_vs_wt_mean_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Mean trace comparison: SynII (n={len(synii_traces)}) vs WT (n={len(wt_traces)})")
    print(f"✓ Time axis focused on stimulus response period (0.5-2.0s)")
    
    # Cluster distribution summary
    if 'enriched_clusters' in locals() and enriched_clusters:
        print(f"\nSynII distribution in enriched clusters {enriched_clusters}:")
        for cluster_id in enriched_clusters:
            count   = np.sum(synii_cluster_assignments == cluster_id)
            percent = 100 * count / len(synii_cluster_assignments)
            print(f"  Cluster {cluster_id}: {count} samples ({percent:.1f}%)")
    
    print(f"✓ Saved to {output_file}")
    
else:
    print("No trace data available for comparison")

### G.7 Cluster-level components analyses

In [ ]:
# Create combined WT pooled + SynII dataset with cluster labels

# Start with WT pooled clustered data
PCA_Data_WT_pooled_SynII_Clust = PCA_Data_WT_Pooled_clustered.copy()

# Add SynII data with 'SynII KO' as cluster label
synii_data_with_cluster = PCA_Data_SynII.copy()
synii_data_with_cluster['HC_Cluster'] = 6

# Concatenate both datasets
PCA_Data_WT_pooled_SynII_Clust = pd.concat([
    PCA_Data_WT_pooled_SynII_Clust,
    synii_data_with_cluster
], ignore_index=True)

print(f"✓ Created PCA_Data_WT_pooled_SynII_Clust dataset")
print(f"  WT clustered samples: {len(PCA_Data_WT_Pooled_clustered)}")
print(f"  SynII samples: {len(PCA_Data_SynII)}")
print(f"  Total samples: {len(PCA_Data_WT_pooled_SynII_Clust)}")
print(f"\nCluster distribution:")
print(PCA_Data_WT_pooled_SynII_Clust['HC_Cluster'].value_counts().sort_index())

In [ ]:
def cluster_boxplot_analysis(data, column, title, output_prefix):
    """Create boxplot by cluster with statistical analysis."""
    from scipy.stats import kruskal, mannwhitneyu
    from itertools import combinations

    plt.figure(figsize=(8, 5))
    clusters = sorted(data['HC_Cluster'].unique())
    
    # Define color palette matching the cluster colors used throughout
    cluster_hex_colors = {
        1: "#d62728",  #  - Cluster 1
        2: "#2ca02c",  #  - Cluster 2
        3: "#ff7f0e",  #  - Cluster 3
        4: "#8c564b",  #  - Cluster 4
        5: "#a1a1a1",  #  - Cluster 5
        6: "#9467bd",  # Brown - SynII KO
    }
    
    # Create palette list matching the clusters present in data
    palette = [cluster_hex_colors[c] for c in clusters]
    
    ax = sns.boxplot(x='HC_Cluster', y=column, hue='HC_Cluster', data=data, palette=palette, legend=False)
    sns.stripplot(x='HC_Cluster', y=column, data=data, color='k', size=3, alpha=0.5, ax=ax)

    if 'PPR' in column:
        plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)

    plt.ylabel(column)
    plt.title(title)

    # Clean styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.get_xaxis().set_visible(False)

    # Create legend with custom label for cluster 6
    handles = [plt.Line2D([0], [0], color=cluster_hex_colors[cluster_id], lw=4) for cluster_id in clusters]
    labels = [f'SynII KO' if cluster_id == 6 else f'Cluster {cluster_id}' for cluster_id in clusters]
    plt.legend(handles, labels, loc='upper right')

    # Statistical tests
    data_per_cluster = {c: data.loc[data['HC_Cluster'] == c, column].dropna().values for c in clusters}

    try:
        kw_stat, kw_p = kruskal(*data_per_cluster.values())
    except ValueError:
        kw_stat, kw_p = float('nan'), float('nan')

    pairs = list(combinations(clusters, 2))
    results = []
    for a, b in pairs:
        x, y = data_per_cluster[a], data_per_cluster[b]
        try:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), float('nan')
        results.append({
            'C_A': a, 'C_B': b, 'n_A': len(x), 'n_B': len(y),
            'U_stat': stat, 'p_raw': p, 'p_corr': min(p * len(pairs), 1.0) if not np.isnan(p) else np.nan
        })

    results.sort(key=lambda d: d['p_corr'] if not np.isnan(d['p_corr']) else 1)

    # Save results
    stats_file = OUTPUT_DIR / f"{output_prefix}_statistics.txt"
    with open(stats_file, "w") as f:
        f.write(f"{column} Statistical Analysis by Cluster\n")
        f.write(f"Kruskal-Wallis: H = {kw_stat:.4f}, p = {kw_p:.6g}\n")
        f.write(f"Bonferroni correction: {len(pairs)}\n")
        f.write("C_A\tC_B\tnA\tnB\tU_stat\tp_raw\tp_corr\n")
        for r in results:
            f.write(f"{r['C_A']}\t{r['C_B']}\t{r['n_A']}\t{r['n_B']}\t"
                    f"{r['U_stat']:.4f}\t{r['p_raw']:.6g}\t{r['p_corr']:.6g}\n")

    plt.tight_layout()
    output_file = OUTPUT_DIR / f"{output_prefix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✓ Saved {column} analysis to {output_file} and {stats_file}")

# AMP1 analysis
cluster_boxplot_analysis(PCA_Data_WT_pooled_SynII_Clust, 'AMP1', 'AMP1 by Cluster', 'amp1_cluster')

# PPR2/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_pooled_SynII_Clust, 'PPR2/1', 'PPR2/1 by Cluster', 'ppr2_1_cluster')

# PPR3/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_pooled_SynII_Clust, 'PPR3/1', 'PPR3/1 by Cluster', 'ppr3_1_cluster')

# %Fail1 analysis
cluster_boxplot_analysis(PCA_Data_WT_pooled_SynII_Clust, '%Fail1', '%Fail1 by Cluster', 'fail1_cluster')


### G.8 High-Amplitude Bouton VS Syn II

#### G.8.a High-Amplitude Bouton Identification VS SYN II

This cell pinpoints WT boutons whose initial EPSC amplitudes exceed the largest SynII response. Flagging these outliers enables targeted inspection of whether unusually strong WT boutons cluster together or remain dispersed.


In [ ]:
# Extract SynII amplitude data from AMP1-AMP10 fields in FEATURES_DATAFRAME
synii_trace_data = []
synii_max_amplitudes = []

# Loop through SynII boutons in NORM_TRACES_DATAFRAME
for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == 'SynII'].iterrows():
    bouton_id = trace_row['ID']
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        synii_trace_data.append((bouton_id, trace_values))
        
        # Find matching row in FEATURES_DATAFRAME
        matching_feature = FEATURES_DATAFRAME[
            (FEATURES_DATAFRAME['ID'] == bouton_id) & 
            (FEATURES_DATAFRAME['Condition'] == 'SynII')
        ]
        
        if not matching_feature.empty:
            # Extract AMP1-AMP10 values
            amp_columns = [f'AMP{i}' for i in range(1, 11)]
            amp_values = []
            for col in amp_columns:
                if col in matching_feature.columns:
                    val = matching_feature[col].iloc[0]
                    if pd.notna(val):
                        amp_values.append(abs(val))
            
            # Find maximum absolute amplitude across all AMP fields
            if len(amp_values) > 0:
                max_amp = np.max(amp_values)
                synii_max_amplitudes.append(max_amp)

# Calculate SynII amplitude threshold (95th percentile)
if len(synii_max_amplitudes) > 0:
    synii_amplitude_threshold = np.percentile(synii_max_amplitudes, 95)
else:
    synii_amplitude_threshold = 0

# Extract WT pooled trace data and identify high-amplitude traces
wt_trace_data = []
wt_high_amplitude_traces = []
wt_regular_traces = []

# Loop through WT boutons in NORM_TRACES_DATAFRAME
for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'].isin(['WT_Anthime', 'WT_Theo', 'WT_Theo_1scd'])].iterrows():
    bouton_id = trace_row['ID']
    condition = trace_row['Condition']
    
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        wt_trace_data.append((bouton_id, trace_values))
        
        # Find matching row in FEATURES_DATAFRAME
        matching_feature = FEATURES_DATAFRAME[
            (FEATURES_DATAFRAME['ID'] == bouton_id) & 
            (FEATURES_DATAFRAME['Condition'] == condition)
        ]
        
        if not matching_feature.empty:
            # Extract AMP1-AMP10 values
            amp_columns = [f'AMP{i}' for i in range(1, 11)]
            amp_values = []
            for col in amp_columns:
                if col in matching_feature.columns:
                    val = matching_feature[col].iloc[0]
                    if pd.notna(val):
                        amp_values.append(abs(val))
            
            # Check if any amplitude exceeds SynII threshold
            if len(amp_values) > 0:
                max_amp = np.max(amp_values)
                if max_amp > synii_amplitude_threshold:
                    wt_high_amplitude_traces.append((bouton_id, trace_values, max_amp))
                else:
                    wt_regular_traces.append((bouton_id, trace_values))

# Calculate common y-axis limits for both panels
all_trace_values = []

# Collect all SynII trace values
for _, trace_values in synii_trace_data:
    all_trace_values.extend(trace_values[~np.isnan(trace_values)])

# Collect all WT trace values
for _, trace_values in wt_trace_data:
    all_trace_values.extend(trace_values[~np.isnan(trace_values)])

# Set common y-limits with some padding
y_min = np.min(all_trace_values) * 1.1
y_max = np.max(all_trace_values) * 1.1

# Create the dual-panel plot with clipped x-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: SynII traces
ax1.set_title(f'SynII Traces (n={len(synii_trace_data)})', fontweight='bold')

synii_traces_for_avg = []
for bouton_id, trace_values in synii_trace_data:
    ax1.plot(COMMON_TIME, trace_values, color='orange', alpha=0.1, linewidth=0.3)
    synii_traces_for_avg.append(trace_values)

# SynII average
if synii_traces_for_avg:
    synii_average = np.nanmean(synii_traces_for_avg, axis=0)
    ax1.plot(COMMON_TIME, synii_average, color='darkorange', linewidth=1, label='SynII Average')

# Threshold lines
ax1.axhline(synii_amplitude_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'Threshold: {synii_amplitude_threshold:.3f}')
ax1.axhline(-synii_amplitude_threshold, color='red', linestyle='--', linewidth=2)

ax1.set_xlabel('Time (s)')
ax1.set_ylabel('ΔF/F')
ax1.set_xlim(0.5, 2.0)
ax1.set_ylim(y_min, y_max)
ax1.legend()
ax1.set_title('SynII Traces (Individual + Average)')

# Panel 2: WT traces with improved readability
ax2.set_title(f'WT Traces: {len(wt_high_amplitude_traces)}/{len(wt_trace_data)} above threshold', fontweight='bold')

# Plot WT traces below threshold with very low alpha
for bouton_id, trace_values in wt_regular_traces:
    ax2.plot(COMMON_TIME, trace_values, color='black', alpha=0.05, linewidth=0.25)

# Plot WT traces above threshold with moderate alpha
for bouton_id, trace_values, max_amp in wt_high_amplitude_traces:
    ax2.plot(COMMON_TIME, trace_values, color='red', alpha=0.25, linewidth=0.3)

# WT average
if wt_trace_data:
    wt_traces_for_avg = [trace_values for _, trace_values in wt_trace_data]
    wt_average = np.nanmean(wt_traces_for_avg, axis=0)
    ax2.plot(COMMON_TIME, wt_average, color='darkblue', linewidth=1, label='WT Average')

# Threshold lines
ax2.axhline(synii_amplitude_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'SynII Threshold')
ax2.axhline(-synii_amplitude_threshold, color='red', linestyle='--', linewidth=2)

# Add legend entries for trace types
ax2.plot([], [], color='black', alpha=0.6, linewidth=2, label=f'Below threshold (n={len(wt_regular_traces)})')
ax2.plot([], [], color='red', alpha=0.7, linewidth=2, label=f'Above threshold (n={len(wt_high_amplitude_traces)})')

ax2.set_xlabel('Time (s)')
ax2.set_ylabel('ΔF/F')
ax2.set_xlim(0.5, 2.0)
ax2.set_ylim(y_min, y_max)
ax2.legend()
ax2.axvline(1.0, color='blue', linestyle=':', alpha=0.7, linewidth=2)

plt.tight_layout()

output_file = OUTPUT_DIR / "27_synii_vs_wt_amplitude_comparison_improved.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

#### G.8.b Mean traces threshold

In [ ]:
# Extract WT pooled trace data and identify high-amplitude traces
wt_trace_data = []
wt_high_amplitude_traces = []
wt_regular_traces = []

# Loop through WT boutons in NORM_TRACES_DATAFRAME
for _, trace_row in NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'].isin(['WT_Anthime', 'WT_Theo', 'WT_Theo_1scd'])].iterrows():
    bouton_id = trace_row['ID']
    condition = trace_row['Condition']
    
    if bouton_id in resampled_trace_lookup:
        trace_values = resampled_trace_lookup[bouton_id]['Avg']
        wt_trace_data.append((bouton_id, trace_values))
        
        # Find matching row in FEATURES_DATAFRAME
        matching_feature = FEATURES_DATAFRAME[
            (FEATURES_DATAFRAME['ID'] == bouton_id) & 
            (FEATURES_DATAFRAME['Condition'] == condition)
        ]
        
        if not matching_feature.empty:
            # Extract AMP1-AMP10 values
            amp_columns = [f'AMP{i}' for i in range(1, 11)]
            amp_values = []
            for col in amp_columns:
                if col in matching_feature.columns:
                    val = matching_feature[col].iloc[0]
                    if pd.notna(val):
                        amp_values.append(abs(val))
            
            # Check if any amplitude exceeds SynII threshold
            if len(amp_values) > 0:
                max_amp = np.max(amp_values)
                if max_amp > synii_amplitude_threshold:
                    wt_high_amplitude_traces.append((bouton_id, trace_values, max_amp))
                else:
                    wt_regular_traces.append((bouton_id, trace_values))

# Calculate mean wt_regular_traces and mean wt_high_amplitude_traces
wt_regular_matrix = np.column_stack([trace_values for _, trace_values in wt_regular_traces])
wt_high_amp_matrix = np.column_stack([trace_values for _, trace_values, _ in wt_high_amplitude_traces])
wt_regular_mean = np.nanmean(wt_regular_matrix, axis=1)
wt_high_amp_mean = np.nanmean(wt_high_amp_matrix, axis=1)

# stack synii_trace_data for mean calculation
synii_matrix = np.column_stack([trace_values for _, trace_values in synii_trace_data])
synii_mean = np.nanmean(synii_matrix, axis=1)

# Plot mean traces
plt.figure(figsize=(10, 6))
plt.plot(COMMON_TIME, wt_regular_mean, color='black', linewidth=1, label='WT Below Threshold Mean')
plt.plot(COMMON_TIME, wt_high_amp_mean, color='red', linewidth=1, label='WT Above Threshold Mean')
plt.plot(COMMON_TIME, synii_mean, color='orange', linewidth=1, label='SynII Mean')
plt.xlabel('Time (s)')
plt.ylabel('ΔF/F')
plt.title('Mean WT Traces (Below and Above Threshold)')
plt.xlim(0.5, 2.0)
plt.legend()
plt.tight_layout()  
output_file = OUTPUT_DIR / "28_Mean_WT_Traces_Threshold.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

#### G.8.c High-Amplitude Bouton Mapping

The previously identified high-amplitude WT boutons are highlighted within PCA space to see whether they define a coherent region or scatter across clusters. Their locations inform hypotheses about the mechanisms supporting exceptionally strong synapses.


In [ ]:
# Show PCA locations of high-amplitude WT traces identified in the previous cell (ignoring target type)
if wt_high_amplitude_traces:
    high_amp_ids = [bouton_id for bouton_id, _, _ in wt_high_amplitude_traces]
    high_amp_mask = PCA_Data_WT_Pooled_clustered['ID'].isin(high_amp_ids)
    high_amp_coordinates = pca_coordinates[high_amp_mask]   

    plt.figure(figsize=(8, 6)) 
# Plot WT pooled data with cluster colors
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1],
           c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')
    
# Plot high-amplitude WT traces
    plt.scatter(high_amp_coordinates[:, 0], high_amp_coordinates[:, 1], 
               marker='p', c='red', edgecolors='black', linewidths=0.6, s=70, label=f'High-Amplitude WT (n={len(high_amp_coordinates)})')
    
# Format plot
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('PCA Locations of High-Amplitude WT Traces')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    output_file = OUTPUT_DIR / "29_pca_high_amplitude_wt_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

### G.9 – SynII and Target Identity

We investigate how SynII bouton positions relate to Purkinje and interneuron territories within PCA space.


#### G.9.a SynII Confidence Ellipses

We construct ellipses around SynII boutons in PCA space to delineate the region occupied by the mutant population. This geometric boundary provides an interpretable measure of SynII variability relative to WT clusters.


In [ ]:
def compare_inside_outside_simplified(in_ids, in_inside_mask, pc_ids, pc_inside_mask):
    """Create clean 2-panel comparison of inside vs outside ellipse traces."""
    
    # Prepare IN data
    in_inside_ids     = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
    in_outside_ids    = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
    
    in_inside_traces  = [resampled_trace_lookup[bid]['Avg'] for bid in in_inside_ids if bid in resampled_trace_lookup]
    in_outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in in_outside_ids if bid in resampled_trace_lookup]
    
    # Prepare PC data
    pc_inside_ids     = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
    pc_outside_ids    = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
    
    pc_inside_traces  = [resampled_trace_lookup[bid]['Avg'] for bid in pc_inside_ids if bid in resampled_trace_lookup]
    pc_outside_traces = [resampled_trace_lookup[bid]['Avg'] for bid in pc_outside_ids if bid in resampled_trace_lookup]
    
    # Create 2-panel figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: IN cells
    if in_inside_traces and in_outside_traces:
        in_inside_mean  = np.nanmean(in_inside_traces, axis=0)
        in_inside_sem   = np.nanstd(in_inside_traces, axis=0, ddof=1) / np.sqrt(len(in_inside_traces))
        
        in_outside_mean = np.nanmean(in_outside_traces, axis=0)
        in_outside_sem  = np.nanstd(in_outside_traces, axis=0, ddof=1) / np.sqrt(len(in_outside_traces))
        
        ax1.plot(COMMON_TIME, in_inside_mean, color='red', linewidth=1, 
                label=f'Inside ellipse (n={len(in_inside_traces)})')
        ax1.fill_between(COMMON_TIME, in_inside_mean-in_inside_sem, in_inside_mean+in_inside_sem, 
                        color='red', alpha=0.3)
        
        ax1.plot(COMMON_TIME, in_outside_mean, color='black', linewidth=1,
                label=f'Outside ellipse (n={len(in_outside_traces)})')
        ax1.fill_between(COMMON_TIME, in_outside_mean-in_outside_sem, in_outside_mean+in_outside_sem, 
                        color='black', alpha=0.3)
    
    ax1.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax1.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlim(0.5, 2.0)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('ΔF/F')
    ax1.set_title('IN Cells: Inside vs Outside SynII Ellipse')
    ax1.legend()
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # Panel 2: PC cells
    if pc_inside_traces and pc_outside_traces:
        pc_inside_mean  = np.nanmean(pc_inside_traces, axis=0)
        pc_inside_sem   = np.nanstd(pc_inside_traces, axis=0, ddof=1) / np.sqrt(len(pc_inside_traces))
        
        pc_outside_mean = np.nanmean(pc_outside_traces, axis=0)
        pc_outside_sem  = np.nanstd(pc_outside_traces, axis=0, ddof=1) / np.sqrt(len(pc_outside_traces))
        
        ax2.plot(COMMON_TIME, pc_inside_mean, color='green', linewidth=1, 
                label=f'Inside ellipse (n={len(pc_inside_traces)})')
        ax2.fill_between(COMMON_TIME, pc_inside_mean-pc_inside_sem, pc_inside_mean+pc_inside_sem, 
                        color='green', alpha=0.3)
        
        ax2.plot(COMMON_TIME, pc_outside_mean, color='black', linewidth=1,
                label=f'Outside ellipse (n={len(pc_outside_traces)})')
        ax2.fill_between(COMMON_TIME, pc_outside_mean-pc_outside_sem, pc_outside_mean+pc_outside_sem, 
                        color='black', alpha=0.3)
    
    ax2.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax2.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlim(0.5, 2.0)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ΔF/F')
    ax2.set_title('PC Cells: Inside vs Outside SynII Ellipse')
    ax2.legend()
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "30_ellipse_inside_outside_comparison_simplified.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ IN: Inside n={len(in_inside_traces)}, Outside n={len(in_outside_traces)}")
    print(f"✓ PC: Inside n={len(pc_inside_traces)}, Outside n={len(pc_outside_traces)}")

# Run simplified comparison
compare_inside_outside_simplified(in_ids, in_inside_mask, pc_ids, pc_inside_mask)

#### G.9.b Target Composition Within SynII Space

By examining which Purkinje and interneuron boutons fall inside or outside the SynII ellipse, we evaluate whether the mutant phenotype preferentially overlaps with specific target identities.


In [ ]:
# Four-panel comparison: IN/PC inside/outside SynII ellipse traces

def plot_group_traces(ax, trace_ids, group_name, color, show_individuals=True):
    """Plot individual traces + average for a group."""
    # Get traces for this group
    group_traces = [resampled_trace_lookup[bid]['Avg'] for bid in trace_ids if bid in resampled_trace_lookup]
    
    if not group_traces:
        ax.text(0.5, 0.5, f'No traces\navailable', ha='center', va='center', 
                transform=ax.transAxes, fontsize=12, color='gray')
        ax.set_title(f'{group_name}\n(n=0)')
        return
    
    # Plot individual traces
    if show_individuals:
        for trace in group_traces:
            ax.plot(COMMON_TIME, trace, color=color, alpha=0.2, linewidth=0.5)
    
    # Calculate and plot average
    group_mean = np.nanmean(group_traces, axis=0)
    group_sem  = np.nanstd(group_traces, axis=0, ddof=1) / np.sqrt(len(group_traces))
    
    ax.plot(COMMON_TIME, group_mean, color=color, linewidth=1, 
            label=f'{group_name} avg')
    ax.fill_between(COMMON_TIME, group_mean-group_sem, group_mean+group_sem, 
                    color=color, alpha=0.3)
    
    # Format subplot
    ax.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax.axvline(1.0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlim(0.5, 2.0)
    ax.set_title(f'{group_name}\n(n={len(group_traces)})')
    ax.grid(True, alpha=0.3)
    
    return group_mean, group_sem

# Prepare trace groups based on ellipse classification
if 'in_inside_mask' in locals() and 'pc_inside_mask' in locals():
    # Get IDs for each group
    in_inside_ids  = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
    in_outside_ids = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
    pc_inside_ids  = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
    pc_outside_ids = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
    
    # Create 2x2 subplot layout
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
    
    # Plot each group
    plot_group_traces(axes[0,0], in_inside_ids, 'IN Inside Ellipse', 'red')
    plot_group_traces(axes[0,1], in_outside_ids, 'IN Outside Ellipse', 'darkred')
    plot_group_traces(axes[1,0], pc_inside_ids, 'PC Inside Ellipse', 'mediumseagreen')
    plot_group_traces(axes[1,1], pc_outside_ids, 'PC Outside Ellipse', 'darkgreen')
    
    # Add common labels
    for ax in axes[-1, :]:  # Bottom row
        ax.set_xlabel('Time (s)')
    for ax in axes[:, 0]:   # Left column
        ax.set_ylabel('ΔF/F')
    
    # Add overall title
    fig.suptitle(f'Trace Analysis: Inside vs Outside SynII {ELLIPSE_CONFIDENCE:.0%} Ellipse', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "31_four_panel_ellipse_trace_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Summary statistics
    print(f"=== ELLIPSE TRACE ANALYSIS SUMMARY ===")
    print(f"Ellipse confidence level: {ELLIPSE_CONFIDENCE:.0%}")
    print(f"IN inside ellipse:  {len(in_inside_ids):2d}  traces ({len(in_inside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
    print(f"IN outside ellipse: {len(in_outside_ids):2d} traces ({len(in_outside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
    print(f"PC inside ellipse:  {len(pc_inside_ids):2d}  traces ({len(pc_inside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
    print(f"PC outside ellipse: {len(pc_outside_ids):2d} traces ({len(pc_outside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
    
    print(f"\n✓ Saved four-panel comparison to {output_file}")
    
else:
    print("Run ellipse analysis first to generate inside/outside classifications")

## Chapter H – Calcium Perturbations and Stability Experiments

Chapter H explores how extracellular calcium and longitudinal manipulations reshape bouton phenotypes within the PCA framework.


### H.1 Calcium Modulation in PCA Space

High- and low-calcium conditions are projected into the WT PCA embedding to observe how extracellular calcium reshapes bouton distributions. Visualizing these shifts indicates whether calcium availability drives distinct synaptic states.


In [ ]:
# Analyze calcium concentration effects on bouton properties in PCA space

def plot_calcium_trajectories():
    """Plot how calcium concentration changes affect PCA positioning."""
    
    plt.figure(figsize=(10, 8))
    
    # Background: WT pooled (2.5mM Ca standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.4, s=30, label='WT pooled (2.5mM Ca)')
    
    # Low calcium (1.5mM) - blue triangles pointing down
    low_ca_coords = pca_data['WT_1_5Ca']
    plt.scatter(low_ca_coords[:, 0], low_ca_coords[:, 1], 
               marker='v', s=60, c='blue', alpha=0.8, edgecolors='darkblue', linewidth=0.5,
               label=f'1.5mM Ca (n={len(low_ca_coords)})')
    
    # High calcium (4mM) - red triangles pointing up  
    high_ca_coords = pca_data['WT_4Ca']
    plt.scatter(high_ca_coords[:, 0], high_ca_coords[:, 1], 
               marker='^', s=60, c='red', alpha=0.8, edgecolors='darkred', linewidth=0.5,
               label=f'4mM Ca (n={len(high_ca_coords)})')
    
    # Calculate centroids
    center_pooled = np.mean(pca_coordinates, axis=0)
    center_low_ca = np.mean(low_ca_coords, axis=0)
    center_high_ca = np.mean(high_ca_coords, axis=0)
    
    # Plot centroids
    plt.scatter(center_low_ca[0], center_low_ca[1], marker='X', s=180, c='blue', 
               edgecolor='black', linewidth=2, label='1.5mM centroid')
    plt.scatter(center_high_ca[0], center_high_ca[1], marker='X', s=180, c='red', 
               edgecolor='black', linewidth=2, label='4mM centroid')
    
    # Draw single arrow from low calcium to high calcium centroid
    plt.arrow(center_low_ca[0], center_low_ca[1],
              center_high_ca[0] - center_low_ca[0], center_high_ca[1] - center_low_ca[1],
              color='purple', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
    
    # Annotate centroid coordinates
    plt.text(center_low_ca[0], center_low_ca[1], f'  1.5mM\n({center_low_ca[0]:.2f},{center_low_ca[1]:.2f})', 
             color='blue', fontsize=9, ha='left', va='center', fontweight='bold')
    plt.text(center_high_ca[0], center_high_ca[1], f'  4mM\n({center_high_ca[0]:.2f},{center_high_ca[1]:.2f})', 
             color='red', fontsize=9, ha='left', va='center', fontweight='bold')
    
    # Connect paired boutons between conditions (assuming matched order)
    n_pairs = min(len(low_ca_coords), len(high_ca_coords))
    if n_pairs > 0:
        for i in range(n_pairs):
            plt.plot([low_ca_coords[i, 0], high_ca_coords[i, 0]],
                     [low_ca_coords[i, 1], high_ca_coords[i, 1]],
                     color='gray', alpha=0.4, linewidth=1)
        print(f"Connected {n_pairs} bouton pairs between calcium conditions")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('Calcium Concentration Effects on Bouton Properties')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "32_calcium_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_low_ca, center_high_ca, n_pairs

def calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs):
    """Calculate movement statistics for calcium concentration changes."""
    
    # Distances from standard condition to each calcium level
    dist_to_low  = np.linalg.norm(center_low_ca - center_pooled)
    dist_to_high = np.linalg.norm(center_high_ca - center_pooled)
    
    # Individual bouton movements
    low_ca_coords  = pca_data['WT_1_5Ca']
    high_ca_coords = pca_data['WT_4Ca']
    
    # Movement from standard to low calcium
    movements_to_low  = []
    n_low_comparisons = min(len(pca_coordinates), len(low_ca_coords))
    for i in range(n_low_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - low_ca_coords[i])
        movements_to_low.append(dist)
    
    # Movement from standard to high calcium
    movements_to_high  = []
    n_high_comparisons = min(len(pca_coordinates), len(high_ca_coords))
    for i in range(n_high_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - high_ca_coords[i])
        movements_to_high.append(dist)
    
    # Movement between calcium conditions (paired boutons)
    calcium_range_movements = []
    for i in range(n_pairs):
        dist = np.linalg.norm(low_ca_coords[i] - high_ca_coords[i])
        calcium_range_movements.append(dist)
    
    return {
        'centroid_distances': {'low': dist_to_low, 'high': dist_to_high},
        'individual_movements': {
            'to_low': movements_to_low,
            'to_high': movements_to_high,
            'between_ca': calcium_range_movements
        }
    }

# Run analysis
center_pooled, center_low_ca, center_high_ca, n_pairs = plot_calcium_trajectories()
movement_stats = calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs)

# Display results
print(f"\n=== CALCIUM CONCENTRATION ANALYSIS ===")
print(f"Centroid coordinates:")
print(f"  WT pooled (2.5mM): ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  1.5mM Ca:          ({center_low_ca[0]:.3f}, {center_low_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['low']:.3f}")
print(f"  4mM Ca:            ({center_high_ca[0]:.3f}, {center_high_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['high']:.3f}")

print(f"\nIndividual bouton movements in PCA space:")
if movement_stats['individual_movements']['to_low']:
    low_moves = movement_stats['individual_movements']['to_low']
    print(f"2.5mM → 1.5mM Ca (n={len(low_moves)}): {np.mean(low_moves):.3f} ± {np.std(low_moves):.3f}")

if movement_stats['individual_movements']['to_high']:
    high_moves = movement_stats['individual_movements']['to_high']
    print(f"2.5mM → 4mM Ca (n={len(high_moves)}): {np.mean(high_moves):.3f} ± {np.std(high_moves):.3f}")

if movement_stats['individual_movements']['between_ca']:
    range_moves = movement_stats['individual_movements']['between_ca']
    print(f"1.5mM ↔ 4mM Ca (n={len(range_moves)}): {np.mean(range_moves):.3f} ± {np.std(range_moves):.3f}")

print(f"\n✓ Calcium trajectory analysis complete")


### H.2 Calcium-Dependent Amp1 Distributions

Amplitude distributions for the first stimulus are contrasted between calcium conditions to test how release probability responds to extracellular calcium changes.


In [ ]:
# Compare AMP1 distributions between calcium concentrations

def plot_calcium_amp1_comparison():
    """Compare AMP1 distributions between 2.5mM and 1.5mM calcium."""
    
    # Get AMP1 data for both conditions
    amp1_standard = PCA_Data_WT_Pooled['AMP1'].dropna()
    amp1_low_ca   = PCA_Data_WT_Low_Ca['AMP1'].dropna()
    
    # Calculate common bins for fair comparison
    all_amp1_values = pd.concat([amp1_standard, amp1_low_ca])
    bin_edges       = np.linspace(all_amp1_values.min(), all_amp1_values.max(), 61)
    
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Calculate weights for percentage display
    weights_standard = np.ones(len(amp1_standard)) * (100.0 / len(amp1_standard))
    weights_low_ca   = np.ones(len(amp1_low_ca)) * (100.0 / len(amp1_low_ca))
    
    # Plot histograms
    plt.hist(amp1_standard, bins=bin_edges, alpha=0.7, color='gray', 
             weights=weights_standard, edgecolor='black', linewidth=0.5,
             label=f'WT 2.5mM Ca (n={len(amp1_standard)})')
    plt.hist(amp1_low_ca, bins=bin_edges, alpha=0.7, color='blue', 
             weights=weights_low_ca, edgecolor='darkblue', linewidth=0.5,
             label=f'WT 1.5mM Ca (n={len(amp1_low_ca)})')
    
    # Add vertical lines for means
    mean_standard = amp1_standard.mean()
    mean_low_ca = amp1_low_ca.mean()
    
    plt.axvline(mean_standard, color='black', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Mean 2.5mM: {mean_standard:.3f}')
    plt.axvline(mean_low_ca, color='blue', linestyle='--', linewidth=2, alpha=0.8,
                label=f'Mean 1.5mM: {mean_low_ca:.3f}')
    
    # Format plot
    plt.xlabel('AMP1 (Amplitude)')
    plt.ylabel('Proportion (%)')
    plt.title('AMP1 Distribution: 2.5mM vs 1.5mM Calcium')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "33_amp1_histogram_calcium_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return amp1_standard, amp1_low_ca, output_file

# Run analysis
amp1_standard, amp1_low_ca, output_file = plot_calcium_amp1_comparison()

# Statistical comparison
from scipy.stats import mannwhitneyu, ttest_ind

# Perform statistical tests
mw_stat, mw_p = mannwhitneyu(amp1_standard, amp1_low_ca, alternative='two-sided')
t_stat, t_p   = ttest_ind(amp1_standard, amp1_low_ca)

# Summary statistics
print(f"=== AMP1 CALCIUM COMPARISON ===")
print(f"2.5mM Ca (standard): {amp1_standard.mean():.3f} ± {amp1_standard.std():.3f} (n={len(amp1_standard)})")
print(f"1.5mM Ca (low):      {amp1_low_ca.mean():.3f} ± {amp1_low_ca.std():.3f} (n={len(amp1_low_ca)})")

print(f"\nStatistical tests:")
print(f"Mann-Whitney U test: U={mw_stat:.1f}, p={mw_p:.4g}")
print(f"T-test: t={t_stat:.3f}, p={t_p:.4g}")

# Effect size (Cohen's d)
pooled_std = np.sqrt(((len(amp1_standard)-1)*amp1_standard.var() + (len(amp1_low_ca)-1)*amp1_low_ca.var()) / 
                     (len(amp1_standard) + len(amp1_low_ca) - 2))
cohens_d   = (amp1_standard.mean() - amp1_low_ca.mean()) / pooled_std
print(f"Cohen's d (effect size): {cohens_d:.3f}")

# Percentage change
pct_change = ((amp1_low_ca.mean() - amp1_standard.mean()) / amp1_standard.mean()) * 100
print(f"Percentage change (1.5mM vs 2.5mM): {pct_change:+.1f}%")

print(f"\n✓ Saved comparison to {output_file}")

### H.3 Calcium-Dependent Failure Rates

Failure percentages are compared between calcium levels, revealing whether reduced calcium disproportionately increases synaptic failures.


In [ ]:
# Compare failure rates between calcium concentrations
fail1_standard = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_low_ca   = PCA_Data_WT_Low_Ca['%Fail1'].dropna()

# Plot histograms
all_fail1 = pd.concat([fail1_standard, fail1_low_ca])
bins      = np.linspace(all_fail1.min(), all_fail1.max(), 31)

plt.figure(figsize=(10, 6))
weights_standard = np.ones(len(fail1_standard)) / len(fail1_standard) * 100
weights_low_ca   = np.ones(len(fail1_low_ca)) / len(fail1_low_ca) * 100

plt.hist(fail1_standard, bins=bins, alpha=0.7, color='gray', weights=weights_standard, 
         edgecolor='black', label=f'WT 2.5mM Ca (n={len(fail1_standard)})')
plt.hist(fail1_low_ca, bins=bins, alpha=0.7, color='blue', weights=weights_low_ca, 
         edgecolor='darkblue', label=f'WT 1.5mM Ca (n={len(fail1_low_ca)})')

plt.axvline(fail1_standard.mean(), color='black', linestyle='--', linewidth=2)
plt.axvline(fail1_low_ca.mean(), color='blue', linestyle='--', linewidth=2)

plt.xlabel('%Fail1')
plt.ylabel('Proportion (%)')
plt.title('Failure Rate: 2.5mM vs 1.5mM Calcium')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

output_file = OUTPUT_DIR / "34_fail1_calcium_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Stats
from scipy.stats import mannwhitneyu
_, p_value = mannwhitneyu(fail1_standard, fail1_low_ca)
print(f"2.5mM Ca: {fail1_standard.mean():.1f}% ± {fail1_standard.std():.1f}%")
print(f"1.5mM Ca: {fail1_low_ca.mean():.1f}% ± {fail1_low_ca.std():.1f}%")
print(f"Mann-Whitney p = {p_value:.4g}")

### H.4 Calcium Impact on Summary Metrics

Non-parametric tests and paired boxplots quantify how calcium concentration affects key amplitudes and plasticity measures, providing statistical backing for observed shifts.


In [ ]:
# Direct comparison between low and high calcium conditions
from scipy.stats import mannwhitneyu

# Prepare data for plotting
amp1_comparison = pd.DataFrame({
    'AMP1': pd.concat([PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

fail1_comparison = pd.DataFrame({
    '%Fail1': pd.concat([PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

# Create side-by-side boxplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# AMP1 comparison
sns.boxplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1, 
           palette=['blue', 'red'], showcaps=True, fliersize=0)
sns.stripplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1,
             color='black', size=3, alpha=0.6)
ax1.set_title('AMP1: Low vs High Calcium')

# %Fail1 comparison  
sns.boxplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
           palette=['blue', 'red'], showcaps=True, fliersize=0)
sns.stripplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
             color='black', size=3, alpha=0.6)
ax2.set_title('%Fail1: Low vs High Calcium')

plt.tight_layout()

output_file = OUTPUT_DIR / "35_calcium_direct_comparison_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Statistical tests
amp1_u, amp1_p = mannwhitneyu(PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1'])
fail1_u, fail1_p = mannwhitneyu(PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1'])

# Results
print("1.5mM vs 4mM Calcium Comparison:")
print("-" * 40)
print(f"AMP1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_Low_Ca['AMP1'].std():.3f}")
print(f"  4mM:   {PCA_Data_WT_High_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_High_Ca['AMP1'].std():.3f}")
print(f"  p = {amp1_p:.4g}")

print(f"%Fail1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_Low_Ca['%Fail1'].std():.1f}%")
print(f"  4mM:   {PCA_Data_WT_High_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_High_Ca['%Fail1'].std():.1f}%")
print(f"  p = {fail1_p:.4g}")

# Save stats
stats_file = OUTPUT_DIR / "calcium_comparison_statistics.txt"
with open(stats_file, 'w') as f:
    f.write("1.5mM vs 4mM Calcium Statistical Comparison\n")
    f.write("=" * 45 + "\n\n")
    f.write(f"AMP1: Mann-Whitney U={amp1_u:.1f}, p={amp1_p:.6g}\n")
    f.write(f"%Fail1: Mann-Whitney U={fail1_u:.1f}, p={fail1_p:.6g}\n")

print(f"✓ Saved to {output_file} and {stats_file}")

### H.5 Calcium PPR 

Average paired-pulse profiles are contrasted across calcium conditions to determine whether facilitation dynamics are calcium-sensitive.


In [ ]:
# Compare PPR profiles across calcium concentrations
ppr_cols      = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Low_Ca.columns]
pulse_numbers = list(range(1, len(ppr_cols) + 2))

# Calculate PPR profiles for each condition
def get_ppr_profile(data, condition_name):
    means = [1.0] + data[ppr_cols].mean().tolist()
    sems  = [0.0] + data[ppr_cols].sem().tolist()
    return means, sems

# Get profiles for each calcium condition
means_low_ca, sems_low_ca     = get_ppr_profile(PCA_Data_WT_Low_Ca, '1.5mM Ca')
means_high_ca, sems_high_ca   = get_ppr_profile(PCA_Data_WT_High_Ca, '4mM Ca') 
means_standard, sems_standard = get_ppr_profile(PCA_Data_WT_Pooled, '2.5mM Ca')

# Plot PPR profiles
plt.figure(figsize=(8, 5))

# Low calcium (blue)
plt.plot(pulse_numbers, means_low_ca, marker='o', color='blue', linewidth=2,
         label=f'1.5mM Ca (n={len(PCA_Data_WT_Low_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_low_ca) - np.array(sems_low_ca),
                 np.array(means_low_ca) + np.array(sems_low_ca), color='blue', alpha=0.2)

# High calcium (red)
plt.plot(pulse_numbers, means_high_ca, marker='s', color='red', linewidth=2,
         label=f'4mM Ca (n={len(PCA_Data_WT_High_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_high_ca) - np.array(sems_high_ca),
                 np.array(means_high_ca) + np.array(sems_high_ca), color='red', alpha=0.2)

# Standard calcium (black)
plt.plot(pulse_numbers, means_standard, marker='D', color='black', linewidth=2,
         label=f'2.5mM Ca (n={len(PCA_Data_WT_Pooled)})')
plt.fill_between(pulse_numbers, np.array(means_standard) - np.array(sems_standard),
                 np.array(means_standard) + np.array(sems_standard), color='black', alpha=0.15)

# Format plot
plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.xlabel('Pulse Number')
plt.ylabel('PPR (A_n/A_1)')
plt.title('PPR Profiles: Calcium Concentration Effects')
plt.xticks(pulse_numbers)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save figure and data
output_fig = OUTPUT_DIR / "36_ppr_profiles_calcium_comparison.pdf"
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

# Save numerical data
output_data = OUTPUT_DIR / "36_ppr_profiles_calcium_data.txt"
with open(output_data, "w") as f:
    f.write("Pulse\t1.5mM_Mean\t1.5mM_SEM\t4mM_Mean\t4mM_SEM\t2.5mM_Mean\t2.5mM_SEM\n")
    for i, pulse in enumerate(pulse_numbers):
        f.write(f"{pulse}\t{means_low_ca[i]:.4f}\t{sems_low_ca[i]:.4f}\t"
                f"{means_high_ca[i]:.4f}\t{sems_high_ca[i]:.4f}\t"
                f"{means_standard[i]:.4f}\t{sems_standard[i]:.4f}\n")

print(f"✓ Saved PPR profiles to {output_fig}")
print(f"✓ Saved numerical data to {output_data}")

### H.6 Calcium Trace Morphology

Mean traces from high- and low-calcium experiments are compared over the response window, highlighting kinetic differences attributable to calcium availability.


In [ ]:
# Compare mean traces between calcium concentrations (0.5-2.0s window)

# Extract traces for calcium conditions
low_ca_traces  = []
high_ca_traces = []

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    if row['Condition'] == 'Theo_1_5Ca':
        low_ca_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_4Ca':
        high_ca_traces.append(row['Avg'])

if not low_ca_traces or not high_ca_traces:
    print("Calcium trace conditions not found in resampled data")
    available_conditions = NORM_TRACES_DATAFRAME['Condition'].unique()
    print(f"Available conditions: {list(available_conditions)}")
else:
    # Calculate means and SEMs
    low_ca_mean = np.nanmean(low_ca_traces, axis=0)
    low_ca_sem = np.nanstd(low_ca_traces, axis=0, ddof=1) / np.sqrt(len(low_ca_traces))
    
    high_ca_mean = np.nanmean(high_ca_traces, axis=0)
    high_ca_sem = np.nanstd(high_ca_traces, axis=0, ddof=1) / np.sqrt(len(high_ca_traces))
    
    # Plot comparison (0.5-2.0s window)
    plt.figure(figsize=(10, 5))
    
    plt.plot(COMMON_TIME, low_ca_mean, color='blue', linewidth=2, 
             label=f'1.5mM Ca (n={len(low_ca_traces)})')
    plt.fill_between(COMMON_TIME, low_ca_mean - low_ca_sem, low_ca_mean + low_ca_sem, 
                     color='blue', alpha=0.25)
    
    plt.plot(COMMON_TIME, high_ca_mean, color='red', linewidth=2,
             label=f'4mM Ca (n={len(high_ca_traces)})')
    plt.fill_between(COMMON_TIME, high_ca_mean - high_ca_sem, high_ca_mean + high_ca_sem, 
                     color='red', alpha=0.25)

    # Add stimulus markers (every 100ms from 1.0s)
    stim_times = [1.0 + 0.1*i for i in range(10)]
    for stim_time in stim_times:
        if stim_time <= 2.0:
            plt.axvline(stim_time, color='gray', linestyle='--', alpha=0.4, linewidth=1)
    
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.xlim(0.5, 2.0)
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.title('Mean Traces: 1.5mM vs 4mM Calcium')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "37_calcium_mean_traces_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Calcium trace comparison: 1.5mM (n={len(low_ca_traces)}) vs 4mM (n={len(high_ca_traces)})")
    print(f"✓ Saved to {output_file}")

In [ ]:
# Compare mean traces between calcium concentrations, normalized by mean AMP1

# Extract traces and AMP1 values for calcium conditions
low_ca_traces  = []
high_ca_traces = []
low_ca_amp1_values = []
high_ca_amp1_values = []

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    bouton_id = row['ID']
    condition = row['Condition']
    
    if condition == 'Theo_1_5Ca':
        low_ca_traces.append(row['Avg'])
        # Get AMP1 from PCA_Data_WT_Low_Ca
        matching = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'] == bouton_id]
        if not matching.empty and pd.notna(matching['AMP1'].iloc[0]):
            low_ca_amp1_values.append(matching['AMP1'].iloc[0])
    elif condition == 'Theo_4Ca':
        high_ca_traces.append(row['Avg'])
        # Get AMP1 from PCA_Data_WT_High_Ca
        matching = PCA_Data_WT_High_Ca[PCA_Data_WT_High_Ca['ID'] == bouton_id]
        if not matching.empty and pd.notna(matching['AMP1'].iloc[0]):
            high_ca_amp1_values.append(matching['AMP1'].iloc[0])

if not low_ca_traces or not high_ca_traces:
    print("Calcium trace conditions not found in resampled data")
    available_conditions = NORM_TRACES_DATAFRAME['Condition'].unique()
    print(f"Available conditions: {list(available_conditions)}")
else:
    # Calculate mean traces first
    low_ca_mean_raw = np.nanmean(low_ca_traces, axis=0)
    high_ca_mean_raw = np.nanmean(high_ca_traces, axis=0)
    
    # Find the value at 1s (first stimulus) for normalization
    # COMMON_TIME should contain the time axis; find index closest to 1.0s
    idx_1s = np.argmin(np.abs(COMMON_TIME - 1.0))
    
    # Get the peak value around 1s (search in a small window after stimulus)
    window_samples = int(0.05 * len(COMMON_TIME) / (COMMON_TIME[-1] - COMMON_TIME[0]))  # ~50ms window
    
    low_ca_peak_1s = np.nanmax(low_ca_mean_raw[idx_1s:idx_1s + window_samples])
    high_ca_peak_1s = np.nanmax(high_ca_mean_raw[idx_1s:idx_1s + window_samples])
    
    # Normalize traces so that first event at 1s equals 1
    low_ca_traces_norm = [trace / low_ca_peak_1s for trace in low_ca_traces]
    high_ca_traces_norm = [trace / high_ca_peak_1s for trace in high_ca_traces]
    
    # Calculate means and SEMs of normalized traces
    low_ca_mean = np.nanmean(low_ca_traces_norm, axis=0)
    low_ca_sem = np.nanstd(low_ca_traces_norm, axis=0, ddof=1) / np.sqrt(len(low_ca_traces_norm))
    
    high_ca_mean = np.nanmean(high_ca_traces_norm, axis=0)
    high_ca_sem = np.nanstd(high_ca_traces_norm, axis=0, ddof=1) / np.sqrt(len(high_ca_traces_norm))
    
    # Plot comparison (0.5-2.0s window)
    plt.figure(figsize=(10, 5))
    
    plt.plot(COMMON_TIME, low_ca_mean, color='blue', linewidth=2, 
             label=f'1.5mM Ca (n={len(low_ca_traces)})')
    plt.fill_between(COMMON_TIME, low_ca_mean - low_ca_sem, low_ca_mean + low_ca_sem, 
                     color='blue', alpha=0.25)
    
    plt.plot(COMMON_TIME, high_ca_mean, color='red', linewidth=2,
             label=f'4mM Ca (n={len(high_ca_traces)})')
    plt.fill_between(COMMON_TIME, high_ca_mean - high_ca_sem, high_ca_mean + high_ca_sem, 
                     color='red', alpha=0.25)

    # Add stimulus markers (every 100ms from 1.0s)
    stim_times = [1.0 + 0.1*i for i in range(10)]
    for stim_time in stim_times:
        if stim_time <= 2.0:
            plt.axvline(stim_time, color='gray', linestyle='--', alpha=0.4, linewidth=1)
    
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.axhline(1, color='black', linestyle=':', linewidth=1, alpha=0.5)  # Reference line at 1
    plt.xlim(0.5, 2.0)
    plt.xlabel('Time (s)')
    plt.ylabel('ΔF/F')
    plt.title('Mean Traces Normalized to First Event: 1.5mM vs 4mM Calcium')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "38_calcium_mean_traces_amp1_normalized.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Calcium trace comparison (normalized to 1st peak = 1):")
    print(f"  1.5mM Ca: n={len(low_ca_traces)}, peak at 1s={low_ca_peak_1s:.3f}")
    print(f"  4mM Ca:   n={len(high_ca_traces)}, peak at 1s={high_ca_peak_1s:.3f}")
    print(f"✓ Saved to {output_file}")

### H.7 Calcium Correlation Structure

We correlate PPR ratios with amplitude and failure metrics under each calcium condition to see how release probability and synaptic reliability interact with plasticity when calcium is limiting.


In [ ]:
# Correlation analysis: AMP1 and %Fail1 vs PPR2/1 across calcium conditions
from scipy.stats import pearsonr, t

def calcium_scatter_analysis(x_param, nbr, y_param='PPR2/1'):
    """Create scatter plot with regression analysis for calcium conditions."""
    
    # Extract data for both conditions
    low_ca_data  = PCA_Data_WT_Low_Ca[[x_param, y_param]].dropna()
    high_ca_data = PCA_Data_WT_High_Ca[[x_param, y_param]].dropna()
    
    x_low, y_low   = low_ca_data[x_param].values, low_ca_data[y_param].values
    x_high, y_high = high_ca_data[x_param].values, high_ca_data[y_param].values
    
    # Calculate separate correlations
    r_low, p_low   = pearsonr(x_low, y_low) if len(x_low) > 1 else (float('nan'), float('nan'))
    r_high, p_high = pearsonr(x_high, y_high) if len(x_high) > 1 else (float('nan'), float('nan'))
    
    # Pooled analysis
    x_pool = np.concatenate([x_low, x_high])
    y_pool = np.concatenate([y_low, y_high])
    
    if len(x_pool) > 2:
        slope, intercept = np.polyfit(x_pool, y_pool, 1)
        r_pool, p_pool   = pearsonr(x_pool, y_pool)
        
        # Calculate confidence intervals
        x_grid = np.linspace(x_pool.min(), x_pool.max(), 100)
        y_fit  = intercept + slope * x_grid
        
        # Simplified CI calculation
        residuals = y_pool - (intercept + slope * x_pool)
        mse       = np.sum(residuals**2) / (len(x_pool) - 2)
        se        = np.sqrt(mse)
        
        t_crit = t.ppf(0.975, len(x_pool) - 2)
        margin = t_crit * se
        
    else:
        r_pool = p_pool = float('nan')
        x_grid = y_fit = margin = None
    
    # Create plot
    plt.figure(figsize=(7, 5))
    plt.scatter(x_low, y_low, c='blue', alpha=0.7, edgecolor='black', s=60, 
               label=f'1.5mM Ca (n={len(x_low)})')
    plt.scatter(x_high, y_high, c='red', alpha=0.7, edgecolor='black', s=60,
               label=f'4mM Ca (n={len(x_high)})')
    
    # Add regression line and confidence band
    if x_grid is not None:
        plt.plot(x_grid, y_fit, color='black', linewidth=2, label='Pooled regression')
        plt.fill_between(x_grid, y_fit - margin, y_fit + margin, 
                        color='black', alpha=0.15, label='95% CI')
    
    # Format plot
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.title(f'{x_param} vs {y_param}: Calcium Comparison')
    plt.grid(True, alpha=0.3)
    
    # Set reasonable axis limits
    if x_param == 'AMP1':
        plt.xlim(0, max(3, x_pool.max() * 1.1))
    plt.ylim(0, max(3, y_pool.max() * 1.1))
    
    # Add correlation statistics
    stats_text = (f"1.5mM: r={r_low:.2f}, p={p_low:.2g}\n"
                  f"4mM: r={r_high:.2f}, p={p_high:.2g}\n"
                  f"Pooled: r={r_pool:.2f}, p={p_pool:.2g}")
    plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, 
             va='top', fontsize=9, 
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.legend()
    plt.tight_layout()
    
    # Save results
    safe_param = x_param.replace('%', 'pct').replace('/', '_')
    output_file = OUTPUT_DIR / f"{nbr}{safe_param}_vs_ppr2_1_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Save statistics
    stats_file = OUTPUT_DIR / f"{nbr}{safe_param}_correlation_vs_ppr2_1_stats.txt"
    with open(stats_file, 'w') as f:
        f.write(f"Correlation: {x_param} vs {y_param}\n")
        f.write(f"1.5mM Ca: r={r_low:.4f}, p={p_low:.6g}, n={len(x_low)}\n")
        f.write(f"4mM Ca: r={r_high:.4f}, p={p_high:.6g}, n={len(x_high)}\n")
        f.write(f"Pooled: r={r_pool:.4f}, p={p_pool:.6g}, n={len(x_pool)}\n")
    
    print(f"✓ {x_param} vs {y_param}: {stats_text.replace(chr(10), ' | ')}")
    return output_file

# Run both analyses
amp1_output = calcium_scatter_analysis('AMP1','40_')
fail1_output = calcium_scatter_analysis('%Fail1','41_')

print(f"\n✓ Scatter analyses complete:")
print(f"  AMP1 vs PPR2/1: {amp1_output}")
print(f"  %Fail1 vs PPR2/1: {fail1_output}")

### H.8 Trial-Level Amplitude Distributions

Per-trial amplitude histograms are modeled to dissect how success and failure amplitudes diverge under different calcium concentrations, offering a granular view of release variability.


In [ ]:
# Analyze trial-level amplitude distributions using existing trials data
from scipy.stats import norm

def analyze_trial_amplitudes(nbr, fit_gaussian=False):
    """Analyze trial amplitude distributions from PPR_TRIALS_FILENAME."""
    
    # Load trials data using existing path structure
    trials_file = BASE_DIR / PPR_TRIALS_FILENAME
    
    try:
        trials = pd.read_excel(trials_file)
    except FileNotFoundError:
        print(f"Trials file not found: {trials_file}")
        return
    
    # Set column names based on the structure you provided
    trials.columns = ['AMP1', 'status', 'file', 'folder', 'trial']
    
    # Clean data
    trials['AMP1']   = pd.to_numeric(trials['AMP1'], errors='coerce')
    trials['status'] = trials['status'].astype(str).str.lower().str.strip()
    trials['folder'] = trials['folder'].astype(str).str.strip()
    
    # Filter for calcium conditions
    calcium_conditions = {'Theo_4Ca', 'Theo_1_5Ca'}
    trials_filtered = trials[
        trials['folder'].isin(calcium_conditions) &
        trials['status'].isin(['success', 'failure'])
    ].dropna(subset=['AMP1']).copy()
    
    if len(trials_filtered) == 0:
        print("No calcium trial data found")
        available_conditions = trials['folder'].unique()
        print(f"Available conditions: {list(available_conditions)}")
        return
    
    # Create categories
    def categorize_trial(row):
        if row['status'] == 'failure':
            return 'Failures (both Ca)'
        return f"Success {row['folder']}"
    
    trials_filtered['Category'] = trials_filtered.apply(categorize_trial, axis=1)
    
    # Set up plotting
    categories = ['Success Theo_4Ca', 'Success Theo_1_5Ca', 'Failures (both Ca)']
    colors = {'Success Theo_4Ca': 'red', 'Success Theo_1_5Ca': 'blue', 'Failures (both Ca)': 'green'}
    
    # Create bins
    amp_range = trials_filtered['AMP1']
    bins      = np.linspace(amp_range.min(), amp_range.max(), 81)
    bin_width = bins[1] - bins[0]
    
    plt.figure(figsize=(8, 6))
    fit_results = []
    
    for category in categories:
        subset = trials_filtered[trials_filtered['Category'] == category]
        if len(subset) == 0:
            continue
        
        amplitudes = subset['AMP1'].values
        weights = np.ones(len(amplitudes)) * (100.0 / len(amplitudes))
        
        # Plot histogram
        plt.hist(amplitudes, bins=bins, weights=weights, alpha=0.6, 
                color=colors[category], label=f"{category} (n={len(amplitudes)})",
                edgecolor='black', linewidth=0.3)
        
        # Add Gaussian fit if requested
        if fit_gaussian and len(amplitudes) > 1:
            mu, sigma   = norm.fit(amplitudes)
            bin_centers = (bins[:-1] + bins[1:]) / 2
            pdf_scaled  = norm.pdf(bin_centers, mu, sigma) * (bin_width * 100)
            plt.plot(bin_centers, pdf_scaled, color=colors[category], linewidth=2)
            
            # Mark peak
            y_peak = norm.pdf(mu, mu, sigma) * (bin_width * 100)
            plt.text(mu, y_peak * 1.02, f"{mu:.2f}", color='black',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
            
            fit_results.append({
                'Category': category,
                'mu': mu,
                'sigma': sigma,
                'n': len(amplitudes)
            })
    
    # Format plot
    plt.xlabel('AMP1 (Trial Amplitude)')
    plt.ylabel('Proportion (%)')
    title = 'Trial Amplitude Distributions: Calcium Conditions'
    if fit_gaussian:
        title += ' + Gaussian Fits'
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    suffix = "_gaussian_fits" if fit_gaussian else ""
    output_file = OUTPUT_DIR / f"{nbr}_trial_amplitudes_calcium{suffix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Display results
    print(f"Trial amplitude analysis:")
    for category in categories:
        subset = trials_filtered[trials_filtered['Category'] == category]
        if len(subset) > 0:
            mean_amp = subset['AMP1'].mean()
            print(f"  {category}: {len(subset)} trials (mean: {mean_amp:.3f})")
    
    if fit_gaussian and fit_results:
        print("\nGaussian fit parameters:")
        for result in fit_results:
            print(f"  {result['Category']}: μ={result['mu']:.3f}, σ={result['sigma']:.3f}")
    
    print(f"✓ Saved plot to {output_file}")
    return trials_filtered

# Run analyses
trials_basic = analyze_trial_amplitudes('42_', fit_gaussian=False)
trials_fitted = analyze_trial_amplitudes('43_', fit_gaussian=True)

In [ ]:
# Multi-Gaussian Quantal Fitting with optional failure-based noise estimation
# ============================================================================
# Two modes:
#   USE_FAILURE_NOISE = True  -> use failure distribution to estimate noise (σ0)
#   USE_FAILURE_NOISE = False -> use 1.5Ca success as baseline (no failure info)

USE_FAILURE_NOISE = False   # <-- Toggle this flag to switch modes
HIST_BIN_WIDTH = 0.05        # <-- Histogram bin width (in AMP1 units)
FIT_ONSET = True            # <-- If True, estimate μ₁ from onset (rising edge) of 1.5Ca

from scipy.optimize import minimize
from scipy.stats import norm
from dataclasses import dataclass
from typing import Optional
import numpy as np

@dataclass
class QuantalFitResult:
    """Result container for quantal histogram fitting."""
    mu1: float             # quantal amplitude
    sigma1: float          # quantal width
    weights_15: np.ndarray # mixture weights for 1.5Ca condition
    weights_4: np.ndarray  # mixture weights for 4Ca condition
    sigma0: float          # failure/baseline noise
    success: bool
    onset_mu: Optional[float] = None   # onset-derived μ₁ (if FIT_ONSET used)
    onset_sigma: Optional[float] = None

def fit_onset_gaussian(counts: np.ndarray, centers: np.ndarray) -> tuple:
    """
    Fit a Gaussian to the onset (rising edge) of the histogram.
    
    Strategy:
    1. Find the first local minimum after the initial rise (trough between failure/noise and 1Q peak)
    2. Fit a Gaussian to the data from the left edge up to this trough
    3. The fitted Gaussian's mean gives an estimate of μ₁
    
    Returns
    -------
    (mu, sigma, trough_idx) or (None, None, None) if detection fails
    """
    # Smooth the histogram slightly to reduce noise
    from scipy.ndimage import uniform_filter1d
    counts_smooth = uniform_filter1d(counts.astype(float), size=3)
    
    # Find first trough: bin where count decreases then increases again
    # Start from left, find first local minimum after we've seen some counts
    trough_idx = None
    started = False
    for i in range(1, len(counts_smooth) - 1):
        if counts_smooth[i] > 0:
            started = True
        if started and counts_smooth[i] < counts_smooth[i-1] and counts_smooth[i] < counts_smooth[i+1]:
            # Found a local minimum
            trough_idx = i
            break
    centers_4  = (edges_4[:-1] + edges_4[1:]) / 2
    if trough_idx is None or trough_idx < 3:
        # Fallback: use first 30% of the distribution
        trough_idx = max(3, len(counts) // 3)
        print(f"  Warning: No clear trough found, using first {trough_idx} bins")
    
    # Extract onset region (left side up to trough)
    onset_centers = centers[:trough_idx + 1]
    onset_counts = counts[:trough_idx + 1]
    
    if onset_counts.sum() < 5:
        print("  Warning: Too few counts in onset region")
        return None, None, trough_idx
    
    # Fit Gaussian to onset region
    # Use weighted mean and std as initial guess
    weights = onset_counts / (onset_counts.sum() + 1e-12)
    mu_init = np.sum(weights * onset_centers)
    sigma_init = np.sqrt(np.sum(weights * (onset_centers - mu_init)**2))
    sigma_init = max(sigma_init, 0.01)
    
    # Fit by minimizing squared error
    def onset_objective(params):
        mu, sigma, amp = params
        if sigma <= 0 or amp <= 0:
            return 1e12
        model = amp * norm.pdf(onset_centers, mu, sigma)
        return np.sum((onset_counts - model)**2)
    
    from scipy.optimize import minimize
    res = minimize(onset_objective, 
                   [mu_init, sigma_init, onset_counts.max()],
                   method='Nelder-Mead',
                   options={'maxiter': 500})
    
    mu_fit, sigma_fit, _ = res.x
    return mu_fit, abs(sigma_fit), trough_idx

def fit_quantal_histograms(
    counts_15: np.ndarray,
    edges_15: np.ndarray,
    counts_4: np.ndarray,
    edges_4: np.ndarray,
    failure_counts: Optional[np.ndarray] = None,
    failure_edges: Optional[np.ndarray] = None,
    use_failure: bool = True,
    K: int = 3,
    lambda_reg: float = 1e-2,
    fixed_mu1: Optional[float] = None,
    fixed_sigma_1q: Optional[float] = None
) -> QuantalFitResult:
    """
    Joint quantal fitting of two calcium conditions with optional failure constraint.
    
    Parameters
    ----------
    counts_15, edges_15 : histogram of 1.5Ca successes
    counts_4, edges_4   : histogram of 4Ca successes
    failure_counts, failure_edges : histogram of failures (optional)
    use_failure : if True and failure data provided, use failure distribution for σ0
    K : maximum number of quanta
    lambda_reg : L2 regularization on weights
    fixed_mu1 : if provided, fix μ₁ to this value (e.g., from onset fit)
    fixed_sigma_1q : if provided, this is the TOTAL width of 1Q (σ₁ will be derived as sqrt(σ_1q² - σ₀²))
    
    Returns
    -------
    QuantalFitResult with fitted parameters
    """
    # Bin centers
    centers_15 = (edges_15[:-1] + edges_15[1:]) / 2
    centers_4  = (edges_4[:-1] + edges_4[1:]) / 2
    
    # Normalize histograms to probability densities
    dx_15 = edges_15[1] - edges_15[0]
    dx_4  = edges_4[1] - edges_4[0]
    p_15  = counts_15 / (counts_15.sum() * dx_15 + 1e-12)
    p_4   = counts_4 / (counts_4.sum() * dx_4 + 1e-12)
    
    # Estimate initial σ0 from failures or 1.5Ca
    if use_failure and failure_counts is not None and failure_edges is not None:
        failure_centers = (failure_edges[:-1] + failure_edges[1:]) / 2
        dx_f = failure_edges[1] - failure_edges[0]
        p_fail = failure_counts / (failure_counts.sum() * dx_f + 1e-12)
        sigma0_init = np.sqrt(np.sum(p_fail * failure_centers**2 * dx_f) 
                              - (np.sum(p_fail * failure_centers * dx_f))**2)
        sigma0_init = max(sigma0_init, 0.01)
    else:
        # Use 1.5Ca as baseline: estimate from left part of distribution
        sigma0_init = np.std(centers_15[counts_15 > counts_15.max() * 0.1]) * 0.5
        sigma0_init = max(sigma0_init, 0.01)
    
    # Initial guesses (use fixed values if provided)
    mu1_init = fixed_mu1 if fixed_mu1 is not None else np.average(centers_15, weights=counts_15 + 1e-8)
    
    # Track which parameters are fixed
    fix_mu1 = fixed_mu1 is not None
    use_fixed_sigma_1q = fixed_sigma_1q is not None
    
    def mixture_pdf(x, mu1, sigma0, weights, sigma_1q_total=None):
        """
        Compute mixture of Gaussians for quantal model.
        
        If sigma_1q_total is provided, use it directly as 1Q width and derive
        higher quanta widths from it. Otherwise use classical variance-additive model.
        """
        pdf = np.zeros_like(x, dtype=float)
        for k in range(len(weights)):
            if k == 0:
                # Failure component: centered at 0
                pdf += weights[k] * norm.pdf(x, 0, sigma0)
            else:
                mu_k = k * mu1
                if sigma_1q_total is not None:
                    # Use onset-constrained model: 1Q has fixed width, higher quanta scale
                    # sigma_k = sigma_1q * sqrt(k) (variance scales with k)
                    sigma_k = sigma_1q_total * np.sqrt(k)
                else:
                    # Classical model: sigma_k = sqrt(sigma0² + k * sigma1²)
                    # But we need sigma1 passed in - use sigma0 as fallback
                    sigma_k = sigma0 * np.sqrt(1 + k * 0.5)  # rough approximation
                pdf += weights[k] * norm.pdf(x, mu_k, sigma_k)
        return pdf
    
    def objective(params):
        """Negative log-likelihood + L2 regularization on weights."""
        # Unpack params based on what's fixed
        idx = 0
        if fix_mu1:
            mu1 = mu1_init
        else:
            mu1 = params[idx]; idx += 1
        sigma0 = params[idx]; idx += 1
        w15 = params[idx:idx + K + 1]; idx += K + 1
        w4  = params[idx:]
        
        # Softmax to ensure weights sum to 1 and are positive
        w15_norm = np.exp(w15) / np.exp(w15).sum()
        w4_norm  = np.exp(w4) / np.exp(w4).sum()
        
        # Compute model PDFs
        sigma_1q = fixed_sigma_1q if use_fixed_sigma_1q else None
        pdf_15 = mixture_pdf(centers_15, mu1, sigma0, w15_norm, sigma_1q_total=sigma_1q)
        pdf_4  = mixture_pdf(centers_4, mu1, sigma0, w4_norm, sigma_1q_total=sigma_1q)
        
        # Negative log-likelihood (KL-divergence style)
        eps = 1e-12
        nll_15 = -np.sum(p_15 * np.log(pdf_15 + eps)) * dx_15
        nll_4  = -np.sum(p_4 * np.log(pdf_4 + eps)) * dx_4
        
        # L2 regularization on weights to prevent overfitting
        reg = lambda_reg * (np.sum(w15**2) + np.sum(w4**2))
        
        return nll_15 + nll_4 + reg
    
    # Build initial parameter vector and bounds based on what's fixed
    w15_init = np.zeros(K + 1)
    w4_init  = np.zeros(K + 1)
    w15_init[1] = 1.0  # Start with mostly 1-quantum for 1.5Ca
    w4_init[1]  = 1.0  # Start with mostly 1-quantum for 4Ca
    
    x0_parts = []
    bounds = []
    if not fix_mu1:
        x0_parts.append([mu1_init])
        bounds.append((0.001, None))
    x0_parts.append([sigma0_init])
    bounds.append((0.001, None))
    x0_parts.append(w15_init)
    x0_parts.append(w4_init)
    bounds += [(None, None)] * (2 * (K + 1))
    
    x0 = np.concatenate(x0_parts)
    
    # Optimize
    res = minimize(objective, x0, method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 1000, 'ftol': 1e-8})
    
    # Extract results
    idx = 0
    if fix_mu1:
        mu1 = mu1_init
    else:
        mu1 = res.x[idx]; idx += 1
    sigma0 = res.x[idx]; idx += 1
    # sigma1 is derived from fixed_sigma_1q or set to a nominal value
    if use_fixed_sigma_1q:
        sigma1 = fixed_sigma_1q  # Report the 1Q total width as sigma1
    else:
        sigma1 = sigma0 * 0.5  # Nominal value when not using onset
    w15_raw = res.x[idx:idx + K + 1]; idx += K + 1
    w4_raw  = res.x[idx:]
    weights_15 = np.exp(w15_raw) / np.exp(w15_raw).sum()
    weights_4  = np.exp(w4_raw) / np.exp(w4_raw).sum()
    
    return QuantalFitResult(
        mu1=mu1,
        sigma1=sigma1,
        weights_15=weights_15,
        weights_4=weights_4,
        sigma0=sigma0,
        success=res.success
    )

# =============================================================================
# Apply to current dataset
# =============================================================================

# Load trials data
trials_file = BASE_DIR / PPR_TRIALS_FILENAME
trials = pd.read_excel(trials_file)
trials.columns = ['AMP1', 'status', 'file', 'folder', 'trial']
trials['AMP1']   = pd.to_numeric(trials['AMP1'], errors='coerce')
trials['status'] = trials['status'].astype(str).str.lower().str.strip()
trials['folder'] = trials['folder'].astype(str).str.strip()

# Extract data for each condition
success_15 = trials[(trials['folder'] == 'Theo_1_5Ca') & 
                    (trials['status'] == 'success')]['AMP1'].dropna().values
success_4  = trials[(trials['folder'] == 'Theo_4Ca') & 
                    (trials['status'] == 'success')]['AMP1'].dropna().values
failures   = trials[trials['status'] == 'failure']['AMP1'].dropna().values

# Create histograms with common binning (controlled by HIST_BIN_WIDTH)
all_amps = np.concatenate([success_15, success_4, failures])
common_bins = np.arange(all_amps.min(), all_amps.max() + HIST_BIN_WIDTH, HIST_BIN_WIDTH)

counts_15, edges_15 = np.histogram(success_15, bins=common_bins)
counts_4, edges_4   = np.histogram(success_4, bins=common_bins)
failure_counts, failure_edges = np.histogram(failures, bins=common_bins)
centers_15 = (edges_15[:-1] + edges_15[1:]) / 2

# =============================================================================
# Onset fitting: estimate μ₁ from rising edge of 1.5Ca distribution
# =============================================================================
onset_mu, onset_sigma = None, None
if FIT_ONSET:
    print("FIT_ONSET enabled: estimating μ₁ from rising edge of 1.5Ca histogram...")
    onset_mu, onset_sigma, trough_idx = fit_onset_gaussian(counts_15, centers_15)
    if onset_mu is not None:
        print(f"  Onset fit: μ₁ = {onset_mu:.4f}, σ (total width) = {onset_sigma:.4f} (trough at bin {trough_idx})")
        print(f"  Note: Only μ₁ is fixed; σ₁ will be optimized to match the data.")
    else:
        print("  Onset fitting failed, falling back to standard fitting")

# Fit based on mode
if USE_FAILURE_NOISE:
    print("Mode: Using FAILURE distribution to estimate baseline noise (σ0)")
    result = fit_quantal_histograms(
        counts_15, edges_15,
        counts_4, edges_4,
        failure_counts=failure_counts,
        failure_edges=failure_edges,
        use_failure=True,
        K=5,
        lambda_reg=1e-4,
        fixed_mu1=onset_mu if FIT_ONSET else None,
        fixed_sigma_1q=onset_sigma if FIT_ONSET else None
    )
else:
    print("Mode: Using 1.5Ca as baseline (no failure information)")
    result = fit_quantal_histograms(
        counts_15, edges_15,
        counts_4, edges_4,
        failure_counts=None,
        failure_edges=None,
        use_failure=False,
        K=5,
        lambda_reg=1e-4,
        fixed_mu1=onset_mu if FIT_ONSET else None,
        fixed_sigma_1q=onset_sigma if FIT_ONSET else None
    )

# Display results
print(f"\nQuantal Fit Results (success={result.success}):")
print(f"  μ₁ (quantal size):    {result.mu1:.4f}" + (" [FIXED from onset]" if FIT_ONSET and onset_mu else ""))
print(f"  σ₁ (quantal width):   {result.sigma1:.4f}" + (" [FIXED from onset]" if FIT_ONSET and onset_sigma else ""))
print(f"  σ₀ (baseline noise):  {result.sigma0:.4f}")
print(f"\nWeights 1.5Ca: {np.array2string(result.weights_15, precision=3)}")
print(f"Weights 4Ca:   {np.array2string(result.weights_4, precision=3)}")

# =============================================================================
# Overlayed plot: both conditions on the same axes
# =============================================================================
centers_15 = (edges_15[:-1] + edges_15[1:]) / 2
centers_4  = (edges_4[:-1] + edges_4[1:]) / 2
dx = centers_15[1] - centers_15[0]

# Normalize histograms to density
p_15 = counts_15 / (counts_15.sum() * dx + 1e-12)
p_4  = counts_4 / (counts_4.sum() * dx + 1e-12)

# Colors for conditions
color_15 = 'blue'
color_4  = 'red'
color_base = 'black'

fig, ax = plt.subplots(figsize=(10, 6))

# Plot histograms
ax.bar(centers_15, p_15, width=dx * 0.85, alpha=0.4, color=color_15, 
       label='1.5 Ca data', edgecolor=color_15, linewidth=0.5)
ax.bar(centers_4, p_4, width=dx * 0.85, alpha=0.4, color=color_4, 
       label='4 Ca data', edgecolor=color_4, linewidth=0.5)

# If FIT_ONSET was used, highlight the onset region
if FIT_ONSET and onset_mu is not None:
    # Mark the trough location
    _, _, trough_idx = fit_onset_gaussian(counts_15, centers_15)
    ax.axvline(centers_15[trough_idx], color='green', linestyle='--', lw=1.5, 
               alpha=0.7, label=f'Onset trough (bin {trough_idx})')
    # Show the onset Gaussian fit
    x_onset = np.linspace(centers_15.min(), centers_15[trough_idx], 100)
    onset_pdf = norm.pdf(x_onset, onset_mu, onset_sigma)
    onset_pdf_scaled = onset_pdf * p_15.max() / onset_pdf.max() * 0.8
    ax.plot(x_onset, onset_pdf_scaled, 'g-', lw=2, alpha=0.8, label=f'Onset fit (μ={onset_mu:.3f})')

# Fine x-grid for smooth curves
x_fine = np.linspace(min(centers_15.min(), centers_4.min()), 
                     max(centers_15.max(), centers_4.max()), 300)

# Base quantal model (shared parameters, just shape reference in black)
K = len(result.weights_15) - 1
for k in range(K + 1):
    if k == 0:
        mu_k = 0
        sigma_k = result.sigma0
    else:
        mu_k = k * result.mu1
        # Use sigma1 as total 1Q width, scale with sqrt(k)
        sigma_k = result.sigma1 * np.sqrt(k)
    pdf_base_k = norm.pdf(x_fine, mu_k, sigma_k)
    scale_factor = max(p_15.max(), p_4.max()) * 0.3
    ax.plot(x_fine, pdf_base_k * scale_factor, color=color_base, linestyle='-', 
            lw=1.5, alpha=0.6, label=f'Quantal k={k}' if k <= 2 else None)

# Individual Gaussian components for 1.5Ca (dotted, blue)
pdf_total_15 = np.zeros_like(x_fine)
for k in range(K + 1):
    if k == 0:
        mu_k = 0
        sigma_k = result.sigma0
    else:
        mu_k = k * result.mu1
        sigma_k = result.sigma1 * np.sqrt(k)
    pdf_k = result.weights_15[k] * norm.pdf(x_fine, mu_k, sigma_k)
    pdf_total_15 += pdf_k
    ax.plot(x_fine, pdf_k, ':', color=color_15, lw=1.5, alpha=0.8)

# Individual Gaussian components for 4Ca (dotted, red)
pdf_total_4 = np.zeros_like(x_fine)
for k in range(K + 1):
    if k == 0:
        mu_k = 0
        sigma_k = result.sigma0
    else:
        mu_k = k * result.mu1
        sigma_k = result.sigma1 * np.sqrt(k)
    pdf_k = result.weights_4[k] * norm.pdf(x_fine, mu_k, sigma_k)
    pdf_total_4 += pdf_k
    ax.plot(x_fine, pdf_k, ':', color=color_4, lw=1.5, alpha=0.8)

# Total fitted curves (solid, matching condition color)
ax.plot(x_fine, pdf_total_15, '-', color=color_15, lw=2.5, label='1.5 Ca fit')
ax.plot(x_fine, pdf_total_4, '-', color=color_4, lw=2.5, label='4 Ca fit')

# Formatting
ax.set_xlabel('AMP1', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
mode_suffix = "failure_noise" if USE_FAILURE_NOISE else "baseline_1_5Ca"
if FIT_ONSET:
    mode_suffix += "_onset"
ax.set_title(f"Multi-Gaussian Quantal Fit (mode: {mode_suffix})\n"
             f"μ₁={result.mu1:.3f}, σ₁={result.sigma1:.3f}, σ₀={result.sigma0:.3f}", 
             fontsize=11)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# Add annotation for weights
textstr = (f"1.5Ca weights: {np.array2string(result.weights_15, precision=2, separator=', ')}\n"
           f"4Ca weights:   {np.array2string(result.weights_4, precision=2, separator=', ')}")
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=8,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))


plt.tight_layout()
output_file = OUTPUT_DIR / f"quantal_fit_overlay_{mode_suffix}.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"\n✓ Saved plot to {output_file}")
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

# Identify bouton IDs based on histogram bin position for 1.5Ca data
# Bin 9 corresponds to the trough identified in the onset fitting

# Get 1.5Ca trial data (individual trials, not bouton averages)
trials_15_ca = trials[(trials['folder'] == 'Theo_1_5Ca') & 
                      (trials['status'] == 'success')].copy()

# Determine bin edges and bin 9 threshold
bin_9_threshold = common_bins[9]  # Upper edge of bin 9

print(f"Bin 9 threshold (upper edge): {bin_9_threshold:.4f}")
print(f"Total individual trials: {len(trials_15_ca)}")

# Get unique bouton IDs
unique_boutons = trials_15_ca['file'].unique()
print(f"Total unique boutons: {len(unique_boutons)}")

# Categorize boutons based on their trial distribution
boutons_only_below = []    # All trials < threshold
boutons_only_above = []    # All trials >= threshold
boutons_mixed = []         # Trials on both sides

for bouton_id in unique_boutons:
    bouton_trials = trials_15_ca[trials_15_ca['file'] == bouton_id]['AMP1']
    n_below = (bouton_trials < bin_9_threshold).sum()
    n_above = (bouton_trials >= bin_9_threshold).sum()
    
    if n_below > 0 and n_above > 0:
        boutons_mixed.append((bouton_id, n_below, n_above))
    elif n_below > 0:
        boutons_only_below.append((bouton_id, n_below))
    else:
        boutons_only_above.append((bouton_id, n_above))

# Helper function to clean bouton ID (remove "_traces_converted" suffix)
def clean_bouton_id(bid):
    return bid.replace("_traces_converted", "")

# Store all bouton IDs below and above threshold (including mixed based on predominance)
# Clean the IDs by removing "_traces_converted"
all_boutons_below_threshold = [clean_bouton_id(bid) for bid, _ in boutons_only_below]
all_boutons_above_threshold = [clean_bouton_id(bid) for bid, _ in boutons_only_above]
all_boutons_equal_threshold = []  # Boutons with equal trials on both sides

# Dictionary mapping bouton ID to ratio (n_below / n_above)
bouton_ratios = {}

# Add boutons with all trials below (ratio = inf, represented as None or a large value)
for bid, n_below in boutons_only_below:
    cleaned_bid = clean_bouton_id(bid)
    bouton_ratios[cleaned_bid] = {'n_below': n_below, 'n_above': 0, 'ratio': float('inf'), 'category': 'BELOW'}

# Add boutons with all trials above (ratio = 0)
for bid, n_above in boutons_only_above:
    cleaned_bid = clean_bouton_id(bid)
    bouton_ratios[cleaned_bid] = {'n_below': 0, 'n_above': n_above, 'ratio': 0.0, 'category': 'ABOVE'}

# Add mixed boutons to appropriate list based on predominance
for bid, n_below, n_above in boutons_mixed:
    cleaned_bid = clean_bouton_id(bid)
    ratio = n_below / n_above
    if n_below > n_above:
        all_boutons_below_threshold.append(cleaned_bid)
        category = 'BELOW'
    elif n_above > n_below:
        all_boutons_above_threshold.append(cleaned_bid)
        category = 'ABOVE'
    else:
        all_boutons_equal_threshold.append(cleaned_bid)
        category = 'EQUAL'
    bouton_ratios[cleaned_bid] = {'n_below': n_below, 'n_above': n_above, 'ratio': ratio, 'category': category}

# Display results
print(f"\n=== BOUTON CATEGORIZATION BY TRIAL AMPLITUDES ===")

print(f"\nBoutons with ALL trials BELOW bin 9 (n={len(boutons_only_below)}):")
for bid, n in boutons_only_below:
    print(f"  {clean_bouton_id(bid)} ({n} trials)")

print(f"\nBoutons with ALL trials AT OR ABOVE bin 9 (n={len(boutons_only_above)}):")
for bid, n in boutons_only_above:
    print(f"  {clean_bouton_id(bid)} ({n} trials)")

print(f"\nBoutons with MIXED trials (both sides of threshold) (n={len(boutons_mixed)}):")
for bid, n_below, n_above in boutons_mixed:
    ratio_below_above = n_below / n_above
    if n_below > n_above:
        predominance = "BELOW"
    elif n_above > n_below:
        predominance = "ABOVE"
    else:
        predominance = "EQUAL"
    print(f"  {clean_bouton_id(bid)}: {n_below} below, {n_above} above (ratio={ratio_below_above:.2f}) → {predominance}")

print(f"\n=== SUMMARY ===")
print(f"Total boutons below threshold (incl. mixed): {len(all_boutons_below_threshold)}")
print(f"Total boutons above threshold (incl. mixed): {len(all_boutons_above_threshold)}")
print(f"Total boutons equal (mixed 50/50): {len(all_boutons_equal_threshold)}")

print(f"\n=== BOUTON RATIOS DICTIONARY (sample) ===")
for i, (bid, info) in enumerate(bouton_ratios.items()):
    if i < 10:
        ratio_str = f"{info['ratio']:.2f}" if info['ratio'] != float('inf') else "inf"
        print(f"  {bid}: n_below={info['n_below']}, n_above={info['n_above']}, ratio={ratio_str}, category={info['category']}")
print(f"  ... ({len(bouton_ratios)} total entries)")

# Debug: Check ID column name and sample values
print(f"\n=== DEBUG: PCA_Data_WT_Low_Ca columns ===")
print(PCA_Data_WT_Low_Ca.columns.tolist())
print(f"\nSample IDs from PCA_Data_WT_Low_Ca: {PCA_Data_WT_Low_Ca['ID'].head().tolist()}")
print(f"Sample bouton IDs from trials (cleaned): {list(all_boutons_below_threshold[:3])}")

# Extract AMP1, %Fail1, and PPR2/1 for boutons in each group from PCA_Data_WT_Low_Ca
# Match by bouton ID (file column in trials corresponds to ID in PCA data)
below_amp1 = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_below_threshold)]['AMP1'].dropna()
above_amp1 = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_above_threshold)]['AMP1'].dropna()
equal_amp1 = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_equal_threshold)]['AMP1'].dropna()

below_fail1 = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_below_threshold)]['%Fail1'].dropna()
above_fail1 = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_above_threshold)]['%Fail1'].dropna()
equal_fail1 = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_equal_threshold)]['%Fail1'].dropna()

below_ppr = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_below_threshold)]['PPR2/1'].dropna()
above_ppr = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_above_threshold)]['PPR2/1'].dropna()
equal_ppr = PCA_Data_WT_Low_Ca[PCA_Data_WT_Low_Ca['ID'].isin(all_boutons_equal_threshold)]['PPR2/1'].dropna()

print(f"\n=== DATA EXTRACTION RESULTS ===")
print(f"Below AMP1: {len(below_amp1)} values")
print(f"Above AMP1: {len(above_amp1)} values")
print(f"Equal AMP1: {len(equal_amp1)} values")

# If no match found, try matching without path prefix
if len(below_amp1) == 0 and len(above_amp1) == 0:
    print("\n⚠ No matches found. Trying alternative matching...")
    # Extract just the bouton name from the full path if needed
    pca_ids = PCA_Data_WT_Low_Ca['ID'].tolist()
    
    # Try to find common pattern
    for bid in all_boutons_below_threshold[:3]:
        for pca_id in pca_ids[:10]:
            if bid in pca_id or pca_id in bid:
                print(f"  Potential match: trial '{bid}' ~ PCA '{pca_id}'")

# Create boxplots only with non-empty data
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Prepare data and labels for boxplots (only include non-empty groups)
data_amp1 = []
data_fail1 = []
data_ppr = []
labels = []
colors = []

if len(below_amp1) > 0:
    data_amp1.append(below_amp1.values)
    data_fail1.append(below_fail1.values)
    data_ppr.append(below_ppr.values)
    labels.append(f'Below (n={len(below_amp1)})')
    colors.append('blue')

if len(above_amp1) > 0:
    data_amp1.append(above_amp1.values)
    data_fail1.append(above_fail1.values)
    data_ppr.append(above_ppr.values)
    labels.append(f'Above (n={len(above_amp1)})')
    colors.append('red')

if len(equal_amp1) > 0:
    data_amp1.append(equal_amp1.values)
    data_fail1.append(equal_fail1.values)
    data_ppr.append(equal_ppr.values)
    labels.append(f'Equal (n={len(equal_amp1)})')
    colors.append('green')

if len(data_amp1) > 0:
    # AMP1 boxplot
    bp1 = ax1.boxplot(data_amp1, patch_artist=True, labels=labels)
    for i, (box, color) in enumerate(zip(bp1['boxes'], colors)):
        box.set_facecolor(color)
        box.set_alpha(0.6)
    
    # Add strip points
    for i, (vals, color) in enumerate(zip(data_amp1, colors)):
        x_jitter = np.ones(len(vals)) * (i + 1) + np.random.uniform(-0.1, 0.1, len(vals))
        ax1.scatter(x_jitter, vals, c=color, s=30, alpha=0.6, zorder=3, edgecolors='black', linewidths=0.5)
    
    ax1.set_ylabel('AMP1')
    ax1.set_title('AMP1 by Threshold Group')
    ax1.grid(True, alpha=0.3)
    
    # %Fail1 boxplot
    bp2 = ax2.boxplot(data_fail1, patch_artist=True, labels=labels)
    for i, (box, color) in enumerate(zip(bp2['boxes'], colors)):
        box.set_facecolor(color)
        box.set_alpha(0.6)
    
    # Add strip points
    for i, (vals, color) in enumerate(zip(data_fail1, colors)):
        x_jitter = np.ones(len(vals)) * (i + 1) + np.random.uniform(-0.1, 0.1, len(vals))
        ax2.scatter(x_jitter, vals, c=color, s=30, alpha=0.6, zorder=3, edgecolors='black', linewidths=0.5)
    
    ax2.set_ylabel('%Fail1')
    ax2.set_title('%Fail1 by Threshold Group')
    ax2.grid(True, alpha=0.3)
    
    # PPR2/1 boxplot
    bp3 = ax3.boxplot(data_ppr, patch_artist=True, labels=labels)
    for i, (box, color) in enumerate(zip(bp3['boxes'], colors)):
        box.set_facecolor(color)
        box.set_alpha(0.6)
    
    # Add strip points
    for i, (vals, color) in enumerate(zip(data_ppr, colors)):
        x_jitter = np.ones(len(vals)) * (i + 1) + np.random.uniform(-0.1, 0.1, len(vals))
        ax3.scatter(x_jitter, vals, c=color, s=30, alpha=0.6, zorder=3, edgecolors='black', linewidths=0.5)
    
    ax3.set_ylabel('PPR2/1')
    ax3.set_title('PPR2/1 by Threshold Group')
    ax3.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No data to display\nCheck ID matching', ha='center', va='center', transform=ax1.transAxes)
    ax2.text(0.5, 0.5, 'No data to display\nCheck ID matching', ha='center', va='center', transform=ax2.transAxes)
    ax3.text(0.5, 0.5, 'No data to display\nCheck ID matching', ha='center', va='center', transform=ax3.transAxes)

plt.suptitle(f'Bouton Properties by Amplitude Threshold (bin 9 = {bin_9_threshold:.3f})', fontweight='bold')
plt.tight_layout()

output_file = OUTPUT_DIR / "30_bouton_threshold_comparison_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Statistical tests (only if both groups have data)
if len(below_amp1) > 0 and len(above_amp1) > 0:
    amp1_u, amp1_p = mannwhitneyu(below_amp1, above_amp1, alternative='two-sided')
    fail1_u, fail1_p = mannwhitneyu(below_fail1, above_fail1, alternative='two-sided')
    ppr_u, ppr_p = mannwhitneyu(below_ppr, above_ppr, alternative='two-sided')
    
    print(f"\n=== STATISTICAL COMPARISON ===")
    print(f"AMP1 - Below: {below_amp1.mean():.3f} ± {below_amp1.std():.3f}, Above: {above_amp1.mean():.3f} ± {above_amp1.std():.3f}")
    print(f"AMP1 Mann-Whitney U (Below vs Above): p = {amp1_p:.4g}")
    print(f"%Fail1 - Below: {below_fail1.mean():.1f}% ± {below_fail1.std():.1f}%, Above: {above_fail1.mean():.1f}% ± {above_fail1.std():.1f}%")
    print(f"%Fail1 Mann-Whitney U (Below vs Above): p = {fail1_p:.4g}")
    print(f"PPR2/1 - Below: {below_ppr.mean():.3f} ± {below_ppr.std():.3f}, Above: {above_ppr.mean():.3f} ± {above_ppr.std():.3f}")
    print(f"PPR2/1 Mann-Whitney U (Below vs Above): p = {ppr_p:.4g}")
else:
    print("\n⚠ Cannot perform statistical tests - insufficient data in one or both groups")

print(f"\n✓ Saved to {output_file}")


In [ ]:
# Calculate baseline values (first second of recording) for below, above, and equal boutons

# Find time indices for baseline (first second: 0 to 1s)
baseline_mask = COMMON_TIME < 1.0

# Extract traces for each group using NORM_TRACES_DATAFRAME
below_traces = []
above_traces = []
equal_traces = []

# Dictionary to store bouton_id, baseline_std, and group
bouton_baseline_info = {}

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    if row['Condition'] == 'Theo_1_5Ca':
        bouton_id = row['ID']
        trace_values = row['Avg']
        
        # Clean the bouton_id for matching (remove _traces_converted if present)
        clean_id = bouton_id.replace("_traces_converted", "")
        
        if clean_id in all_boutons_below_threshold:
            below_traces.append(trace_values)
            baseline_std = np.nanstd(trace_values[baseline_mask])
            bouton_baseline_info[clean_id] = {'baseline_std': baseline_std, 'group': 'below'}
        elif clean_id in all_boutons_above_threshold:
            above_traces.append(trace_values)
            baseline_std = np.nanstd(trace_values[baseline_mask])
            bouton_baseline_info[clean_id] = {'baseline_std': baseline_std, 'group': 'above'}
        elif clean_id in all_boutons_equal_threshold:
            equal_traces.append(trace_values)
            baseline_std = np.nanstd(trace_values[baseline_mask])
            bouton_baseline_info[clean_id] = {'baseline_std': baseline_std, 'group': 'equal'}

# Calculate per-bouton baseline values for each group
below_baselines = [np.nanmean(trace[baseline_mask]) for trace in below_traces]
above_baselines = [np.nanmean(trace[baseline_mask]) for trace in above_traces]
equal_baselines = [np.nanmean(trace[baseline_mask]) for trace in equal_traces]

# Calculate per-bouton baseline STD for each group
below_baselines_std = [np.nanstd(trace[baseline_mask]) for trace in below_traces]
above_baselines_std = [np.nanstd(trace[baseline_mask]) for trace in above_traces]
equal_baselines_std = [np.nanstd(trace[baseline_mask]) for trace in equal_traces]

print("=== BASELINE VALUES (first second of recording) ===\n")

# Print dictionary summary
print(f"Bouton baseline info dictionary created with {len(bouton_baseline_info)} entries")
print(f"  - Below: {sum(1 for v in bouton_baseline_info.values() if v['group'] == 'below')}")
print(f"  - Above: {sum(1 for v in bouton_baseline_info.values() if v['group'] == 'above')}")
print(f"  - Equal: {sum(1 for v in bouton_baseline_info.values() if v['group'] == 'equal')}")
print()

# Prepare data for boxplot
data_baselines = []
data_baselines_std = []
labels = []
colors = []

if below_baselines:
    data_baselines.append(below_baselines)
    data_baselines_std.append(below_baselines_std)
    labels.append(f'Below (n={len(below_baselines)})')
    colors.append('blue')
    print(f"BELOW threshold (n={len(below_baselines)} boutons):")
    print(f"  Baseline mean: {np.mean(below_baselines):.6f} ± {np.std(below_baselines):.6f}")
    print(f"  Baseline STD mean: {np.mean(below_baselines_std):.6f} ± {np.std(below_baselines_std):.6f}")

if above_baselines:
    data_baselines.append(above_baselines)
    data_baselines_std.append(above_baselines_std)
    labels.append(f'Above (n={len(above_baselines)})')
    colors.append('red')
    print(f"\nABOVE threshold (n={len(above_baselines)} boutons):")
    print(f"  Baseline mean: {np.mean(above_baselines):.6f} ± {np.std(above_baselines):.6f}")
    print(f"  Baseline STD mean: {np.mean(above_baselines_std):.6f} ± {np.std(above_baselines_std):.6f}")

if equal_baselines:
    data_baselines.append(equal_baselines)
    data_baselines_std.append(equal_baselines_std)
    labels.append(f'Equal (n={len(equal_baselines)})')
    colors.append('green')
    print(f"\nEQUAL (50/50) (n={len(equal_baselines)} boutons):")
    print(f"  Baseline mean: {np.mean(equal_baselines):.6f} ± {np.std(equal_baselines):.6f}")
    print(f"  Baseline STD mean: {np.mean(equal_baselines_std):.6f} ± {np.std(equal_baselines_std):.6f}")

# Helper function to get significance stars
def get_significance_stars(p_val):
    if p_val < 0.001:
        return '***'
    elif p_val < 0.01:
        return '**'
    elif p_val < 0.05:
        return '*'
    else:
        return 'ns'

# Create boxplots
if data_baselines:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # --- First boxplot: Baseline Mean ---
    ax1 = axes[0]
    bp1 = ax1.boxplot(data_baselines, patch_artist=True, labels=labels)
    for i, (box, color) in enumerate(zip(bp1['boxes'], colors)):
        box.set_facecolor(color)
        box.set_alpha(0.6)
    
    # Add strip points
    for i, (vals, color) in enumerate(zip(data_baselines, colors)):
        x_jitter = np.ones(len(vals)) * (i + 1) + np.random.uniform(-0.1, 0.1, len(vals))
        ax1.scatter(x_jitter, vals, c=color, s=40, alpha=0.7, zorder=3, edgecolors='black', linewidths=0.5)
    
    ax1.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax1.set_ylabel('Baseline ΔF/F (0-1s)')
    ax1.set_title('Baseline Mean by Amplitude Threshold Group')
    ax1.grid(True, alpha=0.3)
    
    # Statistical test for mean
    if len(below_baselines) > 0 and len(above_baselines) > 0:
        _, p_val = mannwhitneyu(below_baselines, above_baselines, alternative='two-sided')
        stars = get_significance_stars(p_val)
        
        y_max = max(max(below_baselines), max(above_baselines))
        y_range = max(data_baselines[0] + data_baselines[1]) - min(data_baselines[0] + data_baselines[1])
        y_bar = y_max + y_range * 0.1
        
        ax1.plot([1, 1, 2, 2], [y_bar, y_bar + y_range * 0.02, y_bar + y_range * 0.02, y_bar], 
                color='black', linewidth=1.5)
        ax1.text(1.5, y_bar + y_range * 0.03, f'{stars}\np={p_val:.3g}', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax1.set_ylim(top=y_bar + y_range * 0.15)
        
        print(f"\nMann-Whitney U (Below vs Above) - Mean: p = {p_val:.4g} ({stars})")
    
    # --- Second boxplot: Baseline STD ---
    ax2 = axes[1]
    bp2 = ax2.boxplot(data_baselines_std, patch_artist=True, labels=labels)
    for i, (box, color) in enumerate(zip(bp2['boxes'], colors)):
        box.set_facecolor(color)
        box.set_alpha(0.6)
    
    # Add strip points
    for i, (vals, color) in enumerate(zip(data_baselines_std, colors)):
        x_jitter = np.ones(len(vals)) * (i + 1) + np.random.uniform(-0.1, 0.1, len(vals))
        ax2.scatter(x_jitter, vals, c=color, s=40, alpha=0.7, zorder=3, edgecolors='black', linewidths=0.5)
    
    ax2.set_ylabel('Baseline STD ΔF/F (0-1s)')
    ax2.set_title('Baseline STD by Amplitude Threshold Group')
    ax2.grid(True, alpha=0.3)
    
    # Statistical test for STD
    if len(below_baselines_std) > 0 and len(above_baselines_std) > 0:
        _, p_val_std = mannwhitneyu(below_baselines_std, above_baselines_std, alternative='two-sided')
        stars_std = get_significance_stars(p_val_std)
        
        y_max_std = max(max(below_baselines_std), max(above_baselines_std))
        y_range_std = max(data_baselines_std[0] + data_baselines_std[1]) - min(data_baselines_std[0] + data_baselines_std[1])
        y_bar_std = y_max_std + y_range_std * 0.1
        
        ax2.plot([1, 1, 2, 2], [y_bar_std, y_bar_std + y_range_std * 0.02, y_bar_std + y_range_std * 0.02, y_bar_std], 
                color='black', linewidth=1.5)
        ax2.text(1.5, y_bar_std + y_range_std * 0.03, f'{stars_std}\np={p_val_std:.3g}', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax2.set_ylim(top=y_bar_std + y_range_std * 0.15)
        
        print(f"Mann-Whitney U (Below vs Above) - STD: p = {p_val_std:.4g} ({stars_std})")
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "31_baseline_threshold_comparison_boxplot.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Saved to {output_file}")
else:
    print("No baseline data to plot")


In [ ]:
# Scatter plot: Baseline STD vs Below/Above Ratio

# Merge baseline info with ratio info
scatter_data = []
for bouton_id, baseline_info in bouton_baseline_info.items():
    if bouton_id in bouton_ratios:
        ratio_info = bouton_ratios[bouton_id]
        ratio = ratio_info['ratio']
        # Skip infinite ratios (all trials below) for plotting
        if ratio != float('inf') and ratio > 0:
            scatter_data.append({
                'bouton_id': bouton_id,
                'baseline_std': baseline_info['baseline_std'],
                'ratio': ratio,
                'category': ratio_info['category']
            })

# Separate by category for coloring
below_x = [d['ratio'] for d in scatter_data if d['category'] == 'BELOW']
below_y = [d['baseline_std'] for d in scatter_data if d['category'] == 'BELOW']
above_x = [d['ratio'] for d in scatter_data if d['category'] == 'ABOVE']
above_y = [d['baseline_std'] for d in scatter_data if d['category'] == 'ABOVE']
equal_x = [d['ratio'] for d in scatter_data if d['category'] == 'EQUAL']
equal_y = [d['baseline_std'] for d in scatter_data if d['category'] == 'EQUAL']

plt.figure(figsize=(10, 6))

# Plot each category
if below_x:
    plt.scatter(below_x, below_y, c='blue', s=60, alpha=0.7, edgecolors='black', 
                linewidths=0.5, label=f'Below (n={len(below_x)})')
if above_x:
    plt.scatter(above_x, above_y, c='red', s=60, alpha=0.7, edgecolors='black', 
                linewidths=0.5, label=f'Above (n={len(above_x)})')
if equal_x:
    plt.scatter(equal_x, equal_y, c='green', s=60, alpha=0.7, edgecolors='black', 
                linewidths=0.5, label=f'Equal (n={len(equal_x)})')

# Add reference line at ratio = 1
plt.axvline(1, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Ratio = 1')

plt.xlabel('Ratio (n_below / n_above)')
plt.ylabel('Baseline STD (ΔF/F)')
plt.title('Baseline STD vs Trial Amplitude Ratio (Below/Above Threshold)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

output_file = OUTPUT_DIR / "32_baseline_std_vs_ratio_scatter.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Scatter plot: {len(scatter_data)} boutons plotted")
print(f"✓ Saved to {output_file}")

## Chapter I - Stability

Chapter I examines the temporal stability of synaptic properties by comparing boutons imaged before and after a 8min interval.

### I.1 Stability Analysis Configuration

Parameter knobs and helper structures are established to analyze before-versus-after stability experiments. These controls govern how strictly clusters are defined in subsequent comparisons.

You can set the configuration here.


In [ ]:
# Config + tiny helpers (set your "edge" controls here) 

ELLIPSE_ALPHA   = 0.95     # ellipse containment level
ALPHA_EXPANSION = 0.50     # alpha-shape expansion factor (0.0 → off)
KNN_K           = None     # k-NN neighbors (None → auto √N)

import numpy as np
from scipy.stats import chi2

def build_ellipse_models(X, y, n_clusters, alpha=0.95, ridge=1e-6):
    """Mean/cov/inv and chi2 threshold for each cluster."""
    thr = chi2.ppf(alpha, df=2)
    models = []
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts) > 2:
            mu  = pts.mean(axis=0)
            cov = np.cov(pts.T) + np.eye(2)*ridge
            inv = np.linalg.pinv(cov)
            models.append({'cluster': k, 'center': mu, 'cov': cov, 'inv_cov': inv})
    return models, thr

def mahalanobis_sq(x, mu, inv):
    d = x - mu
    return float(d.T @ inv @ d)

def ellipses_containing(x, models, thr):
    return {m['cluster'] for m in models if mahalanobis_sq(x, m['center'], m['inv_cov']) <= thr}

def nearest_ellipse_edge(x, models, thr):
    """Return (cluster_id, euclid_dist_to_edge)."""
    best = (None, np.inf)
    for m in models:
        md2 = mahalanobis_sq(x, m['center'], m['inv_cov'])
        if md2 <= thr:
            return m['cluster'], 0.0
        s = np.sqrt(thr/md2)
        x_proj = m['center'] + s*(x - m['center'])
        dist = float(np.linalg.norm(x - x_proj))
        if dist < best[1]:
            best = (m['cluster'], dist)
    return best


### I.2 Stability Trajectories

This visualization tracks how individual boutons move through PCA space from the baseline to the post-manipulation state, summarizing trajectory lengths and directionality.


In [ ]:
# ==== Cell 1 — Before/After trajectories + summary ====

import matplotlib.pyplot as plt

# Data
before_coords = np.asarray(pca_data['stab_before'])
after_coords  = np.asarray(pca_data['stab_after'])
n_pairs       = int(min(len(before_coords), len(after_coords)))
assert n_pairs > 0, "No paired before/after points."

# Plot
plt.figure(figsize=(8,6))
plt.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.4, label='WT background')
plt.scatter(before_coords[:,0], before_coords[:,1], c='orange', s=60, edgecolors='darkorange', label=f'Before (n={len(before_coords)})')
plt.scatter(after_coords[:,0],  after_coords[:,1],  c='brown',  s=60, edgecolors='darkred',   label=f'After  (n={len(after_coords)})')

# Pair links + mean arrow
moves = []
for i in range(n_pairs):
    plt.plot([before_coords[i,0], after_coords[i,0]],
             [before_coords[i,1], after_coords[i,1]], color='gray', alpha=0.6, lw=1)
    moves.append(float(np.linalg.norm(after_coords[i] - before_coords[i])))

diffs    = after_coords[:n_pairs] - before_coords[:n_pairs]
mean_vec = diffs.mean(axis=0)
center   = np.vstack([before_coords[:n_pairs], after_coords[:n_pairs]]).mean(axis=0)
plt.arrow(center[0], center[1], mean_vec[0], mean_vec[1], color='black',
          width=0.05, head_width=0.25, head_length=0.25, length_includes_head=True, label='Mean trajectory')

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
plt.xlabel(f'PC1 ({pc1_variance:.1%})'); plt.ylabel(f'PC2 ({pc2_variance:.1%})')
plt.title('Stability: Before vs After'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "33_stability_before_after_trajectories.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print("=== STABILITY TRAJECTORY ANALYSIS ===")
print(f"Pairs: {n_pairs}")
print(f"Movement (mean±SD): {np.mean(moves):.3f} ± {np.std(moves):.3f}")
print(f"Mean trajectory |mag|: {np.linalg.norm(mean_vec):.3f}  dir=({mean_vec[0]:.3f}, {mean_vec[1]:.3f})")
print(f"✓ Saved {out}")


### I.3 Stability Movement Histogram

We compile a histogram of bouton displacements to quantify how much synaptic properties drift between the before and after conditions.


In [ ]:
# ==== Cell 2 — Movement distance histogram (auto bins) ====

# Freedman–Diaconis binning with fallback
md      = np.linalg.norm(diffs, axis=1).astype(float)
md      = md[np.isfinite(md)]
q25,q75 = np.percentile(md,[25,75]); iqr=float(q75-q25); n=len(md)
bw      = (2*iqr)/(n**(1/3)) if iqr>0 else 0.0
bins    = max(5, int(np.ceil((md.max()-md.min())/bw))) if bw>0 else max(5, int(np.ceil(np.sqrt(n))))

plt.figure(figsize=(6,4))
plt.hist(md, bins=bins, edgecolor='black', alpha=0.85)
plt.axvline(md.mean(), ls='--', lw=2, label=f'Mean = {md.mean():.2f}')
plt.xlabel('Distance in PCA space'); plt.ylabel('Count'); plt.title('Before→After distances'); plt.legend(); plt.tight_layout()
out = OUTPUT_DIR / "34_stability_movement_distances.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print(f"n={n}  mean={md.mean():.3f}  sd={md.std(ddof=1):.3f}  median={np.median(md):.3f}  min={md.min():.3f}  max={md.max():.3f}  bins={bins}")
print(f"✓ Saved {out}")


### I.4 Stability Trace Evolution

Mean traces with confidence intervals are plotted for before and after recordings, revealing how waveform kinetics change with the stability manipulation.


In [ ]:
# ==== Cell 3 — Mean traces (±SEM) for Stability_Before/_05 vs Stability_After/_05 ====

assert 'NORM_TRACES_DATAFRAME' in locals() and 'COMMON_TIME' in locals()

import pandas as pd

def stack_traces(df, cond):
    rows = df[df['Condition']==cond]
    X    = [np.asarray(r['Avg'], float) for _,r in rows.iterrows()]
    X    = [t for t in X if np.isfinite(t).all() and len(t)==len(COMMON_TIME)]
    return (np.vstack(X) if len(X)>0 else np.empty((0,len(COMMON_TIME)))), len(X)

def mean_sem(X):
    if X.size==0: 
        z = np.zeros(len(COMMON_TIME)); return z,z
    m = np.nanmean(X, axis=0)
    s = np.nanstd(X, axis=0, ddof=1)/np.sqrt(max(1,X.shape[0]))
    return m,s

conds   = ["Stability_Before","Stability_Before_05","Stability_After","Stability_After_05"]
stacked = {c: stack_traces(NORM_TRACES_DATAFRAME,c) for c in conds}
stats   = {c: mean_sem(stacked[c][0]) for c in conds}
counts  = {c: stacked[c][1] for c in conds}

colors = {"Stability_Before":"#1f77b4","Stability_Before_05":"#1f77b4","Stability_After":"#d62728","Stability_After_05":"#d62728"}
styles = {"Stability_Before":('-',2.0),"Stability_Before_05":('--',1.8),"Stability_After":('-',2.0),"Stability_After_05":('--',1.8)}

plt.figure(figsize=(8.5,5.0))
for c in conds:
    mean,sem = stats[c]; ls,lw = styles[c]
    plt.plot(COMMON_TIME, mean, color=colors[c], ls=ls, lw=lw, label=f"{c} (n={counts[c]})")
    plt.fill_between(COMMON_TIME, mean-sem, mean+sem, color=colors[c], alpha=0.15, lw=0)
plt.axhline(0, color='gray', ls=':', lw=1.0)
plt.axvspan(0.5, 2.0, color='gray', alpha=0.08, label='0.5–2.0 s')
plt.xlabel('Time (s)'); plt.ylabel('ΔF/F'); plt.title('Stability conditions: mean traces (±SEM)')
plt.legend(ncol=2, fontsize=9, frameon=True); plt.tight_layout()
out = OUTPUT_DIR / "35_stability_mean_traces_before_after.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print("Counts:", {c:counts[c] for c in conds}); print(f"✓ Saved {out}")


### I.5 Stability Ellipse Boundaries

Elliptical decision boundaries are applied in PCA space to evaluate which boutons remain within the WT tolerance zone after the manipulation, providing a geometric perspective on stability.


In [ ]:
# ===== Cell 4 — Ellipses: PCA with hard edge + tolerance zone, and pie =====
# knobs
ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)

import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# --- build models ---
def _ellipse_models(X, y, n_clusters, ridge=1e-6):
    models=[]
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts)>2:
            mu=pts.mean(axis=0); cov=np.cov(pts.T)+np.eye(2)*ridge; inv=np.linalg.pinv(cov)
            models.append({'cluster':k,'center':mu,'cov':cov,'inv_cov':inv})
    return models
def _md2(x, m): 
    d=x-m['center']; return float(d.T @ m['inv_cov'] @ d)

ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")

# --- plot PCA with hard + tolerance ---
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.35, label='WT')
uniq = np.unique(cluster_assignments)

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3: continue
    m     = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    color = get_cluster_color(cid)
    # tolerance ring: draw tol (filled light), then hard (outline)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  color, 0.10, 0.0)    # tol zone
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, color, 0.00, 2.0)    # hard edge

# overlay pairs
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before  = np.asarray(pca_data['stab_before'])[:n_pairs]
after   = np.asarray(pca_data['stab_after'])[:n_pairs]

# Calculate stability for line colors
def _label_all_for_plot(x, thr):
    """Return all clusters that contain point x (for plotting)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d       = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _label_all_for_plot(before[i], thr_tol)  
    a_clusters = _label_all_for_plot(after[i], thr_tol)   
    is_stable  = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}% (Conservative)')

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "36_ellipses_pca_hard_tol.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability approach (hard-edge only) ---
def _inside_any(x, thr): 
    return any(_md2(x,m) <= thr for m in ellipse_models)

def _label_all(x, thr):
    """Return all clusters that contain point x (conservative approach)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d       = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Get all cluster memberships for each point
b_labs = [_label_all(before[i], thr_tol) for i in range(n_pairs)]
a_labs = [_label_all(after[i], thr_tol) for i in range(n_pairs)]

# Conservative stability: stable if any overlap between before and after cluster sets
stable   = sum(1 for i in range(n_pairs) if len(b_labs[i] & a_labs[i]) > 0)
unstable = n_pairs - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title('Conservative Ellipse Stability (hard-edge)'); plt.tight_layout()
out = OUTPUT_DIR / "37_ellipses_stability_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Ellipses] Conservative: hard α={ELLIPSE_ALPHA:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_labs[i] & a_labs[i]) > 0 and (len(b_labs[i]) > 1 or len(a_labs[i]) > 1):
        overlap_cases.append((i, b_labs[i], a_labs[i], b_labs[i] & a_labs[i]))

if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with ellipse overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")


### I.6 Stability Alpha Shapes

Alpha-shape contours supply a non-parametric boundary around WT clusters, allowing us to test whether post-manipulation boutons exit the original manifold.


In [ ]:
# ===== Cell — Alpha-shapes: PCA with hard polygon + tolerance zone, and pie =====
# knobs
ALPHA_EXPANSION = 2      # tolerance zone: expanded by sqrt(1+exp)
ALPHA_KNN_Q     = 0.2       # alpha heuristic quantile (0.8 for very small n)

import numpy as np, matplotlib.pyplot as plt, alphashape
from shapely.affinity import scale as shp_scale
from shapely.geometry import MultiPoint, Polygon as ShapelyPolygon, MultiPolygon, Point

def _alpha_for(pts):
    n=len(pts); d2=np.sum((pts[:,None,:]-pts[None,:,:])**2, axis=2); np.fill_diagonal(d2, np.inf)
    kth=np.partition(d2,1,axis=1)[:,1]; base=np.sqrt(kth)
    return float(1.5*np.quantile(base, ALPHA_KNN_Q if n>=10 else 0.8))

def _make_alpha_shapes(X,y,expansion):
    s = float(np.sqrt(1.0+expansion)) if expansion>0 else 1.0
    res={}
    for cid in np.unique(y):
        pts = X[y==cid]
        if len(pts)<3: res[cid]=None; continue
        a=_alpha_for(pts)
        poly = alphashape.alphashape([tuple(r) for r in pts], a)
        if poly is None or getattr(poly,'is_empty',True): poly = MultiPoint([tuple(r) for r in pts]).convex_hull
        res[cid]={'hard':poly, 'tol': (shp_scale(poly, xfact=s, yfact=s, origin='centroid') if expansion>0 else poly)}
    return res

shapes = _make_alpha_shapes(pca_coordinates, cluster_assignments, ALPHA_EXPANSION)
# --- PCA: draw tol (light fill) + hard (outline) ---
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.35, label='WT')
for cid,sh in shapes.items():
    if not sh: continue
    for tag, (fa, ls, lw) in dict(tol=(0.10,'-',0.0), hard=(0.00,'-',2.0)).items():
        g = sh[tag]
        geoms=[g] if g.geom_type=='Polygon' else (list(g.geoms) if g.geom_type=='MultiPolygon' else [])
        for gg in geoms:
            X,Y = np.array(gg.exterior.coords).T
            if fa>0: ax.fill(X,Y,color=get_cluster_color(cid),alpha=fa)
            ax.plot(X,Y,color=get_cluster_color(cid),ls=ls,lw=lw)

# pairs with conservative coloring
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before = np.asarray(pca_data['stab_before'])[:n_pairs]
after  = np.asarray(pca_data['stab_after'])[:n_pairs]

# Conservative approach: find ALL clusters that contain each point
def _in_shape_all(x, use_tol=False, paired_point=None, paired_clusters=None):
    """Return set of all clusters that contain point x (conservative approach)"""
    p = Point(float(x[0]), float(x[1]))
    clusters = set()
    
    # Check containment in all shapes
    for cid, sh in shapes.items():
        if sh is None: continue
        g = sh['tol'] if use_tol else sh['hard']
        if g.geom_type == 'Polygon':
            if g.contains(p):
                clusters.add(cid)
        elif g.geom_type == 'MultiPolygon':
            if any(gg.contains(p) for gg in g.geoms):
                clusters.add(cid)
    
    # If not inside any shape, assign to nearest boundary
    if not clusters:
        distances = []
        for cid, sh in shapes.items():
            if sh is None: continue
            g = sh['tol'] if use_tol else sh['hard']
            dist = p.distance(g)
            distances.append((cid, dist))
        
        if distances:
            # If paired point is inside shapes, prefer those clusters when distances are close
            if paired_clusters:
                # Find distances to paired clusters
                paired_dists = [(cid, d) for cid, d in distances if cid in paired_clusters]
                if paired_dists:
                    min_paired_dist = min(paired_dists, key=lambda t: t[1])[1]
                    overall_min_dist = min(distances, key=lambda t: t[1])[1]
                    # If paired cluster is within 20% of nearest, prefer it
                    if min_paired_dist <= overall_min_dist * 1.2:
                        nearest_cid = min(paired_dists, key=lambda t: t[1])[0]
                        clusters = {nearest_cid}
                        return clusters
            
            # Otherwise use nearest
            nearest_cid = min(distances, key=lambda t: t[1])[0]
            clusters = {nearest_cid}
    
    return clusters

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _in_shape_all(before[i], use_tol=True)
    a_clusters = _in_shape_all(after[i], use_tol=True)
    
    # Check stability
    is_stable = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Before')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Alpha-shapes: hard + tol (exp={ALPHA_EXPANSION:.2f}) (Conservative)')

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()

out = OUTPUT_DIR / "38_alphashapes_pca_hard_tol.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability (tolerance boundary) ---
b_clusters_all = [_in_shape_all(before[i], use_tol=True) for i in range(n_pairs)]
a_clusters_all = [_in_shape_all(after[i], use_tol=True) for i in range(n_pairs)]

# Conservative stability: stable if any overlap between before and after cluster sets
stable = sum(1 for i in range(n_pairs) if len(b_clusters_all[i] & a_clusters_all[i]) > 0)
unstable = n_pairs - stable

plt.figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title(f'Conservative Alpha-shape Stability (exp={ALPHA_EXPANSION:.2f})'); plt.tight_layout()
out = OUTPUT_DIR / "39_alphashapes_stability_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Alpha] Conservative: exp={ALPHA_EXPANSION:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_clusters_all[i] & a_clusters_all[i]) > 0 and (len(b_clusters_all[i]) > 1 or len(a_clusters_all[i]) > 1):
        overlap_cases.append((i, b_clusters_all[i], a_clusters_all[i], b_clusters_all[i] & a_clusters_all[i]))

# After the stability calculation, add this diagnostic
print("\nDiagnostic for UNSTABLE pairs (red lines):")
unstable_indices = [i for i in range(n_pairs) if len(b_clusters_all[i] & a_clusters_all[i]) == 0]

for idx in unstable_indices[:10]:  # Show first 10 unstable pairs
    b_pos = before[idx]
    a_pos = after[idx]
    b_clusters = b_clusters_all[idx]
    a_clusters = a_clusters_all[idx]
    
    print(f"\nPair {idx}: UNSTABLE")
    print(f"  Before {b_pos}: assigned to clusters={b_clusters}")
    print(f"  After  {a_pos}: assigned to clusters={a_clusters}")
    
    # Show distances to all shape boundaries
    for cid, sh in shapes.items():
        if sh is None: continue
        g = sh['tol']
        p_before = Point(float(b_pos[0]), float(b_pos[1]))
        p_after = Point(float(a_pos[0]), float(a_pos[1]))
        b_dist = p_before.distance(g)
        a_dist = p_after.distance(g)
        print(f"    Cluster {cid}: Before dist={b_dist:.3f}, After dist={a_dist:.3f}")

# After creating shapes, verify tolerance expansion
print("\nVerifying tolerance zone sizes:")
for cid, sh in shapes.items():
    if sh is None: continue
    hard_area = sh['hard'].area
    tol_area = sh['tol'].area
    expansion_ratio = np.sqrt(tol_area / hard_area)
    print(f"Cluster {cid}: area ratio = {expansion_ratio:.3f} (expected: {np.sqrt(1+ALPHA_EXPANSION):.3f})")
if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with alpha-shape overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")



### I.7 Stability Bootstrap Analysis

Bootstrap resampling compares observed bouton shifts to random expectations and visualizes representative random pairs, strengthening conclusions about genuine remodeling.


In [ ]:
# ===== Cell 4 — Ellipses: PCA with hard edge + tolerance zone, and pie =====
# knobs
ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)
N_BOOTSTRAP   = 2000      # bootstrap iterations

# --- Bootstrap stability analysis ---
n_pairs = 26
rng = np.random.default_rng(852)
rng_viz = np.random.default_rng(852)

# --- build models ---
ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")



def _label_all(x, thr, models):
    """Return all clusters that contain point x"""
    inside = [m['cluster'] for m in models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    d = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

def compute_stability(before, after, thr, models):
    """Compute number of stable pairs"""
    stable = 0
    for i in range(len(before)):
        b_clusters = _label_all(before[i], thr, models)
        a_clusters = _label_all(after[i], thr, models)
        if len(b_clusters & a_clusters) > 0:
            stable += 1
    return stable

# Bootstrap
bootstrap_stabilities = []
for b in range(N_BOOTSTRAP):
    # Sample random pairs with replacement
    indices = rng.choice(len(pca_coordinates), size=(n_pairs, 2), replace=True)
    before = pca_coordinates[indices[:, 0]]
    after = pca_coordinates[indices[:, 1]]
    stable = compute_stability(before, after, thr_tol, ellipse_models)
    bootstrap_stabilities.append(stable / n_pairs * 100)

bootstrap_stabilities = np.array(bootstrap_stabilities)
mean_stability = np.mean(bootstrap_stabilities)
ci_low = np.percentile(bootstrap_stabilities, 2.5)
ci_high = np.percentile(bootstrap_stabilities, 97.5)

print(f"\nBootstrap Results ({N_BOOTSTRAP} iterations):")
print(f"  Mean stability: {mean_stability:.1f}%")
print(f"  95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]")
print(f"  Std: {np.std(bootstrap_stabilities):.1f}%")

####
# --- Generate one set of pairs for visualization ---
####
indices_viz = rng_viz.choice(len(pca_coordinates), size=(n_pairs, 2), replace=False)
before = pca_coordinates[indices_viz[:, 0]]
after = pca_coordinates[indices_viz[:, 1]]

# --- plot PCA with hard + tolerance ---
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(pca_coordinates[:,0], pca_coordinates[:,1], c=get_cluster_colors(cluster_assignments), s=20, alpha=0.35, label='WT')
uniq = np.unique(cluster_assignments)


# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _label_all(before[i], thr_tol, ellipse_models)
    a_clusters = _label_all(after[i], thr_tol, ellipse_models)
    is_stable  = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

ax.scatter(before[:,0], before[:,1], c='orange', s=60, edgecolors='k', lw=0.5, label='Random A')
ax.scatter(after[:,0],  after[:,1],  c='brown',  s=60, marker='s', edgecolors='k', lw=0.5, label='Random B')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

pc1,pc2 = PCA_RESULTS['explained_variance']; ax.set_xlabel(f'PC1 ({pc1:.1%})'); ax.set_ylabel(f'PC2 ({pc2:.1%})')
ax.set_title(f'Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}% (Random Pairs)')

handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
ax.legend(handles=handles); ax.grid(alpha=0.3); plt.tight_layout()
out = OUTPUT_DIR / "36_ellipses_pca_hard_tol_random.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Bootstrap distribution histogram ---
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(bootstrap_stabilities, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(mean_stability, color='red', linestyle='--', lw=2, label=f'Mean: {mean_stability:.1f}%')
ax.axvline(ci_low, color='orange', linestyle=':', lw=2, label=f'95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]')
ax.axvline(ci_high, color='orange', linestyle=':', lw=2)
ax.set_xlabel('Stability (%)')
ax.set_ylabel('Frequency')
ax.set_title(f'Bootstrap Distribution of Stability ({N_BOOTSTRAP} iterations, {n_pairs} pairs each)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
out = OUTPUT_DIR / "37_ellipses_stability_bootstrap.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Pie chart with bootstrap CI ---
stable_pct = mean_stability
unstable_pct = 100 - mean_stability

plt.figure(figsize=(6.3,5.8))
plt.pie([stable_pct, unstable_pct], 
        labels=[f"Stable ({stable_pct:.1f}%)", f"Unstable ({unstable_pct:.1f}%)"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title(f'Random Pairs Stability (Bootstrap: {N_BOOTSTRAP} iter)\n95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]')
plt.tight_layout()
out = OUTPUT_DIR / "38_ellipses_stability_pie_bootstrap.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print(f"\n[Ellipses Bootstrap] α={ELLIPSE_ALPHA:.2f}, tol={ELLIPSE_TOL*100:.0f}%")
print(f"  Mean stability: {mean_stability:.1f}% (95% CI: [{ci_low:.1f}%, {ci_high:.1f}%])")


In [ ]:
# Pie plots side by side for ellipse stability and bootstrap results

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.8))

# Pie plot 1: Ellipse stability (conservative, hard-edge)
ax1.pie([stable, unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50', '#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
ax1.set_title('Conservative Ellipse Stability (hard-edge)')

# Pie plot 2: Bootstrap results
ax2.pie([stable_pct, unstable_pct], labels=[f"Stable ({stable_pct:.1f}%)", f"Unstable ({unstable_pct:.1f}%)"],
        colors=['#4CAF50', '#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
ax2.set_title('Bootstrap Results')

# Equal aspect ratio for both
ax1.set_aspect('equal')
ax2.set_aspect('equal')

plt.tight_layout()
out = OUTPUT_DIR / "46_stability_ellipse_bootstrap_pies.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()

print(f"[Ellipses] Conservative: hard α={ELLIPSE_ALPHA:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)")
print(f"✓ Saved {out}")

### I.8 Stability Plasticity Profiles

Paired-pulse and amplitude profiles are recalculated for before and after datasets with statistical annotations to detect systematic shifts in short-term plasticity.


In [ ]:
# Compare PPR profiles before vs after treatment
from scipy.stats import ttest_rel

def plot_stability_ppr_profile():
    """Plot PPR profile with statistical testing."""
    
    # PPR analysis
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_Stability_Before.columns]
    pulse_numbers = list(range(1, len(ppr_cols) + 2))
    
    # Calculate PPR profiles
    ppr_before = [1.0] + PCA_Data_Stability_Before[ppr_cols].mean().tolist()
    ppr_before_sem = [0.0] + PCA_Data_Stability_Before[ppr_cols].sem().tolist()
    ppr_after = [1.0] + PCA_Data_Stability_After[ppr_cols].mean().tolist()
    ppr_after_sem = [0.0] + PCA_Data_Stability_After[ppr_cols].sem().tolist()
    
    # Create plot
    plt.figure(figsize=(8, 6))
    
    # PPR plot
    plt.plot(pulse_numbers, ppr_before, marker='o', color='orange', linewidth=2, 
             label=f'Before (n={len(PCA_Data_Stability_Before)})')
    plt.fill_between(pulse_numbers, np.array(ppr_before) - np.array(ppr_before_sem),
                     np.array(ppr_before) + np.array(ppr_before_sem), color='orange', alpha=0.2)
    plt.plot(pulse_numbers, ppr_after, marker='s', color='brown', linewidth=2, 
             label=f'After (n={len(PCA_Data_Stability_After)})')
    plt.fill_between(pulse_numbers, np.array(ppr_after) - np.array(ppr_after_sem),
                     np.array(ppr_after) + np.array(ppr_after_sem), color='brown', alpha=0.2)
    plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
    plt.xlabel('Pulse Number')
    plt.ylabel('PPR (A_n/A_1)')
    plt.title('PPR Profile: Before vs After')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(pulse_numbers)
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / "47_stability_ppr_profile.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Statistical testing for PPR
    print("=== PPR STATISTICAL ANALYSIS ===")
    for i, col in enumerate(ppr_cols, start=2):
        t_stat, p_val = ttest_rel(PCA_Data_Stability_Before[col], PCA_Data_Stability_After[col], nan_policy='omit')
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        print(f"Pulse {i}: t={t_stat:.3f}, p={p_val:.3e} ({sig})")
    
    return output_file

plot_output = plot_stability_ppr_profile()
print(f"✓ Saved PPR profile to {plot_output}")


### I.9 Stability Parameter Comparisons

Detailed paired statistical tests are run for amplitudes, failure rates, and key ratios, with significance markers highlighting which parameters change reliably.


In [ ]:
from scipy.stats import wilcoxon

# Detailed statistical comparisons for key parameters
def plot_stability_comparisons():
    """Create paired boxplots with statistical tests."""
    
    def add_significance_bar(ax, p_value, positions=[0, 1]):
        """Add significance bar above boxplot."""
        y_max = ax.get_ylim()[1]
        y_min = ax.get_ylim()[0]
        y_sig = y_max + 0.05 * (y_max - y_min)
        
        # Significance stars
        if p_value < 0.001:
            sig_text = '***'
        elif p_value < 0.01:
            sig_text = '**'
        elif p_value < 0.05:
            sig_text = '*'
        else:
            sig_text = 'ns'
        
        # Draw bar and text
        ax.plot(positions, [y_sig, y_sig], color='black', linewidth=1.5)
        ax.text(np.mean(positions), y_sig + 0.01 * (y_max - y_min), sig_text,
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # AMP1 comparison
    amp1_data = pd.DataFrame({
        'AMP1': pd.concat([PCA_Data_Stability_Before['AMP1'], PCA_Data_Stability_After['AMP1']]),
        'Condition': ['Before'] * len(PCA_Data_Stability_Before) + ['After'] * len(PCA_Data_Stability_After)
    })
    
    sns.boxplot(data=amp1_data, x='Condition', y='AMP1', hue='Condition', ax=axes[0], palette=['orange', 'brown'], legend=False)
    sns.stripplot(data=amp1_data, x='Condition', y='AMP1', ax=axes[0], color='black', size=3, alpha=0.6)
    
    # Add connecting lines for paired data
    n_pairs = min(len(PCA_Data_Stability_Before), len(PCA_Data_Stability_After))
    for i in range(n_pairs):
        axes[0].plot([0, 1], [PCA_Data_Stability_Before['AMP1'].iloc[i], PCA_Data_Stability_After['AMP1'].iloc[i]],
                     color='gray', alpha=0.4, linewidth=1)
    
    t_stat_amp1, p_val_amp1 = ttest_rel(PCA_Data_Stability_Before['AMP1'], PCA_Data_Stability_After['AMP1'])
    add_significance_bar(axes[0], p_val_amp1)
    axes[0].set_title('AMP1: Before vs After')
    
    # PPR2/1 comparison
    ppr_data = pd.DataFrame({
        'PPR2/1': pd.concat([PCA_Data_Stability_Before['PPR2/1'], PCA_Data_Stability_After['PPR2/1']]),
        'Condition': ['Before'] * len(PCA_Data_Stability_Before) + ['After'] * len(PCA_Data_Stability_After)
    })
    
    sns.boxplot(data=ppr_data, x='Condition', y='PPR2/1', hue='Condition', ax=axes[1], palette=['orange', 'brown'], legend=False)
    sns.stripplot(data=ppr_data, x='Condition', y='PPR2/1', ax=axes[1], color='black', size=3, alpha=0.6)
    
    for i in range(n_pairs):
        axes[1].plot([0, 1], [PCA_Data_Stability_Before['PPR2/1'].iloc[i], PCA_Data_Stability_After['PPR2/1'].iloc[i]],
                     color='gray', alpha=0.4, linewidth=1)
    
    t_stat_ppr, p_val_ppr = ttest_rel(PCA_Data_Stability_Before['PPR2/1'], PCA_Data_Stability_After['PPR2/1'])
    add_significance_bar(axes[1], p_val_ppr)
    axes[1].set_title('PPR2/1: Before vs After')
    
    # Baseline fluorescence (F0) analysis
    # Calculate baseline from first 0.5s of traces
    # Extract traces from RAW_TRACES_DF for Stability_Before and Stability_After conditions
    stab_before_df = RAW_TRACES_DF[RAW_TRACES_DF['Condition'].isin(['Stability_Before', 'Stability_Before_05'])]
    stab_after_df = RAW_TRACES_DF[RAW_TRACES_DF['Condition'].isin(['Stability_After', 'Stability_After_05'])]
    
    stab_before_traces = stab_before_df['Avg'].tolist()
    stab_after_traces = stab_after_df['Avg'].tolist()
    
    baseline_before = [np.nanmean(trace[:int(0.5 * len(COMMON_TIME))]) for trace in stab_before_traces]
    baseline_after = [np.nanmean(trace[:int(0.5 * len(COMMON_TIME))]) for trace in stab_after_traces]
    print(baseline_before, baseline_after)
    f0_data = pd.DataFrame({
        'F0': baseline_before + baseline_after,
        'Condition': ['Before'] * len(baseline_before) + ['After'] * len(baseline_after)
    })
    
    sns.boxplot(data=f0_data, x='Condition', y='F0', hue='Condition', ax=axes[2], palette=['orange', 'brown'], legend=False)
    sns.stripplot(data=f0_data, x='Condition', y='F0', ax=axes[2], color='black', size=3, alpha=0.6)
    
    # Connect paired points
    for i in range(min(len(baseline_before), len(baseline_after))):
        axes[2].plot([0, 1], [baseline_before[i], baseline_after[i]],
                        color='gray', alpha=0.4, linewidth=1)
    
    # Wilcoxon signed-rank test for baseline
    stat_f0, p_val_f0 = wilcoxon(baseline_before, baseline_after)
    add_significance_bar(axes[2], p_val_f0)
    axes[2].set_title('Baseline F0: Before vs After')
    axes[2].set_ylabel('F0 (a.u.)')

    
    plt.tight_layout()
    output_file = OUTPUT_DIR / "48_stability_statistical_comparisons.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print results
    print("=== STABILITY STATISTICAL RESULTS ===")
    print(f"AMP1: t={t_stat_amp1:.3f}, p={p_val_amp1:.4g}")
    print(f"PPR2/1: t={t_stat_ppr:.3f}, p={p_val_ppr:.4g}")
    print(f"Baseline F0 (Wilcoxon): W={stat_f0:.3f}, p={p_val_f0:.4g}")
    
    return output_file, baseline_after, baseline_before

comparison_output = plot_stability_comparisons()
print(f"✓ Saved comparisons to {comparison_output}")

# Additional plot: Distribution of baseline values
fig2, ax2 = plt.subplots(figsize=(8, 5))

# Calculate the difference (After - Before)
baseline_before = comparison_output[2]
baseline_after = comparison_output[1]
baseline_diff = [after - before for after, before in zip(baseline_after, baseline_before)]

sns.histplot(baseline_diff, kde=True, color='purple', alpha=0.7, ax=ax2)

# Add vertical line at 0
ax2.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='No change')

ax2.set_xlabel('ΔBaseline F0 (After - Before)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Baseline Difference (After - Before)')
ax2.legend()

plt.tight_layout()
distribution_output = OUTPUT_DIR / "49_baseline_difference_distribution.pdf"
plt.savefig(distribution_output, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved baseline difference distribution to {distribution_output}")

### I.10 Stability Synthesis

A textual summary consolidates sample sizes, significant metrics, and interpretation from the stability analysis, ensuring the narrative captures the most impactful findings.


In [ ]:
# Final stability analysis summary
def stability_summary():
    """Generate comprehensive stability analysis summary."""
    
    print("=" * 60)
    print("COMPREHENSIVE STABILITY ANALYSIS SUMMARY")
    print("=" * 60)
    
    # Sample sizes
    n_before = len(PCA_Data_Stability_Before)
    n_after = len(PCA_Data_Stability_After)
    n_pairs = min(n_before, n_after)
    
    print(f"\nSample sizes:")
    print(f"  Before: {n_before} boutons")
    print(f"  After: {n_after} boutons")
    print(f"  Paired: {n_pairs} boutons")
    
    # Key parameter changes
    print(f"\nKey parameter changes (Before → After):")
    
    # AMP1
    amp1_before_mean = PCA_Data_Stability_Before['AMP1'].mean()
    amp1_after_mean = PCA_Data_Stability_After['AMP1'].mean()
    amp1_change = ((amp1_after_mean - amp1_before_mean) / amp1_before_mean) * 100
    print(f"  AMP1: {amp1_before_mean:.3f} → {amp1_after_mean:.3f} ({amp1_change:+.1f}%)")
    
    # PPR2/1
    ppr_before_mean = PCA_Data_Stability_Before['PPR2/1'].mean()
    ppr_after_mean = PCA_Data_Stability_After['PPR2/1'].mean()
    ppr_change = ((ppr_after_mean - ppr_before_mean) / ppr_before_mean) * 100
    print(f"  PPR2/1: {ppr_before_mean:.3f} → {ppr_after_mean:.3f} ({ppr_change:+.1f}%)")
    
    # PCA movement analysis (if available)
    if 'movement_distances' in locals():
        print(f"\nPCA space movement:")
        print(f"  Mean distance: {np.mean(movement_distances):.3f} ± {np.std(movement_distances):.3f}")
        print(f"  Max distance: {np.max(movement_distances):.3f}")
        print(f"  Boutons with large movement (>mean): {np.sum(movement_distances > np.mean(movement_distances))}/{len(movement_distances)}")
    
    # Cluster stability (if available)
    if 'stable_pairs' in locals() and 'unstable_pairs' in locals():
        total_analyzed = stable_pairs + unstable_pairs
        stability_pct = (stable_pairs / total_analyzed) * 100
        print(f"\nCluster assignment stability:")
        print(f"  Stable pairs: {stable_pairs}/{total_analyzed} ({stability_pct:.1f}%)")
        print(f"  Unstable pairs: {unstable_pairs}/{total_analyzed} ({100-stability_pct:.1f}%)")
    
    print(f"\n" + "=" * 60)
    print("Analysis complete - all figures saved to OUTPUT_DIR")
    print("=" * 60)

# Run summary
stability_summary()

## Chapter J - 50Hz experiments with low and high calcium

### J.1 50Hz Trajectories

In [ ]:
# Analyze theo concentration effects on bouton properties in PCA space

def extract_base_name(bouton_id):
    """Extract base name (date + linescan + bouton) for pairing across conditions.
    
    Examples: 
        '20210721_linescan1_50Hz_10pulses_4mMCa_bouton1_traces_converted' 
            -> '20210721_linescan1_bouton1'
        '20191017_linescan2_20Hz_10pulses_2.5mMCa_bouton2_traces_converted'
            -> '20191017_linescan2_bouton2'
    """
    import re
    bid = str(bouton_id)
    # Remove _traces_converted suffix
    bid = re.sub(r'_traces_converted$', '', bid)
    # Remove Hz frequency patterns (e.g. _20Hz_, _50Hz_)
    bid = re.sub(r'_\d+Hz_?', '_', bid, flags=re.IGNORECASE)
    # Remove pulse count patterns (e.g. _10pulses_)
    bid = re.sub(r'_\d+pulses_?', '_', bid, flags=re.IGNORECASE)
    # Remove calcium concentration patterns (e.g. _4mMCa_, _1.5mMCa_, _2.5mMCa_, _1_5Ca, _4Ca)
    bid = re.sub(r'_\d+\.?\d*mMCa_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+[_.]?\d*Ca_?', '_', bid, flags=re.IGNORECASE)
    # Remove trailing underscores  
    bid = re.sub(r'_+$', '', bid)
    # Collapse multiple underscores
    bid = re.sub(r'_+', '_', bid)
    return bid

def find_valid_pairs(ids_1, ids_2, coords_1, coords_2):
    """Find valid pairs between two sets of boutons based on matching base names.
    
    Returns tuple of (valid_indices_1, valid_indices_2, base_names).
    """
    base_1 = {i: extract_base_name(bid) for i, bid in enumerate(ids_1)}
    base_2 = {i: extract_base_name(bid) for i, bid in enumerate(ids_2)}
    
    # Build lookup for second set
    lookup_2 = {}
    for i, base in base_2.items():
        if base not in lookup_2:
            lookup_2[base] = i
    
    valid_1, valid_2, names = [], [], []
    for i1, base in base_1.items():
        if base in lookup_2:
            valid_1.append(i1)
            valid_2.append(lookup_2[base])
            names.append(base)
    
    return valid_1, valid_2, names

def plot_theo_50Hz_trajectories():
    """Plot how theo concentration changes affect PCA positioning at 50Hz.
    
    Note: Only 1.5mM and 4mM can be paired (when base IDs match).
          2.5mM at 50Hz is ALWAYS unpaired and shown without connections.
    """
    
    plt.figure(figsize=(12, 8))
    
    # Background: WT pooled (standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.3, s=30, label='WT pooled')
    
    # Get projected data from pca_data dictionary
    theo_1_5_coords = pca_data['50Hz_1_5Ca']
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    theo_4_coords   = pca_data['50Hz_4Ca']
    
    # Get IDs for pairing
    theo_1_5_ids = PCA_Data_50Hz_1_5_Ca['ID'].values
    theo_4_ids   = PCA_Data_50Hz_4_Ca['ID'].values
    # Note: theo_2_5_ids not used for pairing (always unpaired)

    # theo 1.5mM @ 50Hz - blue triangles pointing down
    plt.scatter(theo_1_5_coords[:, 0], theo_1_5_coords[:, 1], 
               marker='v', s=60, c='blue', alpha=0.8, edgecolors='darkblue', linewidth=0.5,
               label=f'Theo 1.5mM 50Hz (n={len(theo_1_5_coords)})')
    
    # theo 2.5mM @ 50Hz - orange diamonds (unpaired)
    plt.scatter(theo_2_5_coords[:, 0], theo_2_5_coords[:, 1], 
               marker='D', s=60, c='orange', alpha=0.8, edgecolors='darkorange', linewidth=0.5,
               label=f'Theo 2.5mM 50Hz (n={len(theo_2_5_coords)}) [unpaired]')
    
    # theo 4mM @ 50Hz - red triangles pointing up  
    plt.scatter(theo_4_coords[:, 0], theo_4_coords[:, 1], 
               marker='^', s=60, c='red', alpha=0.8, edgecolors='darkred', linewidth=0.5,
               label=f'Theo 4mM 50Hz (n={len(theo_4_coords)})')
    
    # Calculate centroids
    center_pooled   = np.mean(pca_coordinates, axis=0)
    center_theo_1_5 = np.mean(theo_1_5_coords, axis=0)
    center_theo_2_5 = np.mean(theo_2_5_coords, axis=0)
    center_theo_4   = np.mean(theo_4_coords, axis=0)
    
    # Plot centroids
    plt.scatter(center_pooled[0], center_pooled[1], marker='X', s=200, c='gray', 
               edgecolor='black', linewidth=2, label='WT pooled centroid', zorder=5)
    plt.scatter(center_theo_1_5[0], center_theo_1_5[1], marker='X', s=200, c='blue', 
               edgecolor='black', linewidth=2, label='1.5mM centroid', zorder=5)
    plt.scatter(center_theo_2_5[0], center_theo_2_5[1], marker='X', s=200, c='orange', 
               edgecolor='black', linewidth=2, label='2.5mM centroid', zorder=5)
    plt.scatter(center_theo_4[0], center_theo_4[1], marker='X', s=200, c='red', 
               edgecolor='black', linewidth=2, label='4mM centroid', zorder=5)
    
    # Draw arrows from WT pooled centroid to theo condition centroids
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_1_5[0] - center_pooled[0], center_theo_1_5[1] - center_pooled[1],
              color='blue', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_2_5[0] - center_pooled[0], center_theo_2_5[1] - center_pooled[1],
              color='orange', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_4[0] - center_pooled[0], center_theo_4[1] - center_pooled[1],
              color='red', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    
    # Annotate centroid coordinates
    plt.text(center_pooled[0], center_pooled[1]-0.3, 
             f'WT pooled\n({center_pooled[0]:.2f},{center_pooled[1]:.2f})', 
             color='gray', fontsize=9, ha='center', va='top', fontweight='bold')
    plt.text(center_theo_1_5[0], center_theo_1_5[1]-0.3, 
             f'1.5mM\n({center_theo_1_5[0]:.2f},{center_theo_1_5[1]:.2f})', 
             color='blue', fontsize=9, ha='center', va='top', fontweight='bold')
    plt.text(center_theo_2_5[0], center_theo_2_5[1]+0.3, 
             f'2.5mM\n({center_theo_2_5[0]:.2f},{center_theo_2_5[1]:.2f})', 
             color='darkorange', fontsize=9, ha='center', va='bottom', fontweight='bold')
    plt.text(center_theo_4[0], center_theo_4[1]+0.3, 
             f'4mM\n({center_theo_4[0]:.2f},{center_theo_4[1]:.2f})', 
             color='red', fontsize=9, ha='center', va='bottom', fontweight='bold')
    
    # Connect ONLY valid paired boutons between 1.5mM and 4mM based on matching base IDs
    # (2.5mM is ALWAYS unpaired, so no connections drawn for it)
    valid_1_5_idx, valid_4_idx, pair_names = find_valid_pairs(
        theo_1_5_ids, theo_4_ids, theo_1_5_coords, theo_4_coords
    )
    
    n_pairs = len(valid_1_5_idx)
    if n_pairs > 0:
        for i1, i4 in zip(valid_1_5_idx, valid_4_idx):
            # Connect 1.5 → 4 mM only (no 2.5 since it's unpaired)
            plt.plot([theo_1_5_coords[i1, 0], theo_4_coords[i4, 0]],
                     [theo_1_5_coords[i1, 1], theo_4_coords[i4, 1]],
                     color='gray', alpha=0.4, linewidth=1.0, linestyle='--')
        print(f"Connected {n_pairs} valid 1.5mM ↔ 4mM bouton pairs (matched by base ID)")
    else:
        print("No valid pairs found between 1.5mM and 4mM conditions")
    
    print(f"Note: 2.5mM at 50Hz ({len(theo_2_5_coords)} boutons) is ALWAYS unpaired")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('Theo Concentration Effects on Bouton Properties (50Hz)\n[1.5↔4mM paired, 2.5mM unpaired]')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "theo_50Hz_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, n_pairs, valid_1_5_idx, valid_4_idx

def calculate_theo_50Hz_movements(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, 
                                   n_pairs, valid_1_5_idx, valid_4_idx):
    """Calculate movement statistics for theo concentration changes at 50Hz."""
    
    # Get coordinates
    theo_1_5_coords = pca_data['50Hz_1_5Ca']
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    theo_4_coords   = pca_data['50Hz_4Ca']

    # Distances from WT pooled centroid to each theo level centroid
    dist_to_1_5 = np.linalg.norm(center_theo_1_5 - center_pooled)
    dist_to_2_5 = np.linalg.norm(center_theo_2_5 - center_pooled)
    dist_to_4   = np.linalg.norm(center_theo_4 - center_pooled)
    
    # Movement from standard to each theo concentration
    movements_to_1_5  = []
    n_1_5_comparisons = min(len(pca_coordinates), len(theo_1_5_coords))
    for i in range(n_1_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_1_5_coords[i])
        movements_to_1_5.append(dist)
    
    movements_to_2_5  = []
    n_2_5_comparisons = min(len(pca_coordinates), len(theo_2_5_coords))
    for i in range(n_2_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_2_5_coords[i])
        movements_to_2_5.append(dist)
    
    movements_to_4  = []
    n_4_comparisons = min(len(pca_coordinates), len(theo_4_coords))
    for i in range(n_4_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_4_coords[i])
        movements_to_4.append(dist)
    
    # Movement between theo concentrations (ONLY valid paired boutons: 1.5 ↔ 4)
    movements_1_5_to_4 = []
    for i1, i4 in zip(valid_1_5_idx, valid_4_idx):
        dist = np.linalg.norm(theo_1_5_coords[i1] - theo_4_coords[i4])
        movements_1_5_to_4.append(dist)
    
    return {
        'centroid_distances': {
            '1.5mM': dist_to_1_5,
            '2.5mM': dist_to_2_5,
            '4mM': dist_to_4
        },
        'individual_movements': {
            'to_1.5': movements_to_1_5,
            'to_2.5': movements_to_2_5,
            'to_4': movements_to_4,
            '1.5_to_4': movements_1_5_to_4
        }
    }

# Run analysis
center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, n_pairs, valid_1_5_idx, valid_4_idx = plot_theo_50Hz_trajectories()
movement_stats = calculate_theo_50Hz_movements(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, 
                                                n_pairs, valid_1_5_idx, valid_4_idx)

# Display results
print(f"\n=== THEO CONCENTRATION ANALYSIS (50Hz) ===")
print(f"Centroid coordinates:")
print(f"  WT pooled:      ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  Theo 1.5mM:     ({center_theo_1_5[0]:.3f}, {center_theo_1_5[1]:.3f}) - distance from WT: {movement_stats['centroid_distances']['1.5mM']:.3f}")
print(f"  Theo 2.5mM:     ({center_theo_2_5[0]:.3f}, {center_theo_2_5[1]:.3f}) - distance from WT: {movement_stats['centroid_distances']['2.5mM']:.3f} [UNPAIRED]")
print(f"  Theo 4mM:       ({center_theo_4[0]:.3f}, {center_theo_4[1]:.3f}) - distance from WT: {movement_stats['centroid_distances']['4mM']:.3f}")

print(f"\nIndividual bouton movements from WT pooled:")
if movement_stats['individual_movements']['to_1.5']:
    moves = movement_stats['individual_movements']['to_1.5']
    print(f"WT → Theo 1.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

if movement_stats['individual_movements']['to_2.5']:
    moves = movement_stats['individual_movements']['to_2.5']
    print(f"WT → Theo 2.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f} [UNPAIRED]")

if movement_stats['individual_movements']['to_4']:
    moves = movement_stats['individual_movements']['to_4']
    print(f"WT → Theo 4mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

print(f"\nPairwise movements (valid pairs only):")
if movement_stats['individual_movements']['1.5_to_4']:
    moves = movement_stats['individual_movements']['1.5_to_4']
    print(f"1.5mM ↔ 4mM (n={len(moves)} valid pairs): {np.mean(moves):.3f} ± {np.std(moves):.3f}")
else:
    print("No valid 1.5mM ↔ 4mM pairs found")

print(f"\n✓ Theo 50Hz trajectory analysis complete")
print(f"✓ Saved to {OUTPUT_DIR / 'theo_50Hz_concentration_pca_trajectories.pdf'}")

In [ ]:
# Analyze theo concentration effects on bouton properties in PCA space

def extract_base_name(bouton_id):
    """Extract base name (date + linescan + bouton) for pairing across conditions.
    
    Examples: 
        '20210721_linescan1_50Hz_10pulses_4mMCa_bouton1_traces_converted' 
            -> '20210721_linescan1_bouton1'
        '20191017_linescan2_20Hz_10pulses_2.5mMCa_bouton2_traces_converted'
            -> '20191017_linescan2_bouton2'
    """
    import re
    bid = str(bouton_id)
    # Remove _traces_converted suffix
    bid = re.sub(r'_traces_converted$', '', bid)
    # Remove Hz frequency patterns (e.g. _20Hz_, _50Hz_)
    bid = re.sub(r'_\d+Hz_?', '_', bid, flags=re.IGNORECASE)
    # Remove pulse count patterns (e.g. _10pulses_)
    bid = re.sub(r'_\d+pulses_?', '_', bid, flags=re.IGNORECASE)
    # Remove calcium concentration patterns (e.g. _4mMCa_, _1.5mMCa_, _2.5mMCa_, _1_5Ca, _4Ca)
    bid = re.sub(r'_\d+\.?\d*mMCa_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+[_.]?\d*Ca_?', '_', bid, flags=re.IGNORECASE)
    # Remove trailing underscores  
    bid = re.sub(r'_+$', '', bid)
    # Collapse multiple underscores
    bid = re.sub(r'_+', '_', bid)
    return bid

def find_valid_pairs(ids_1, ids_2, coords_1, coords_2):
    """Find valid pairs between two sets of boutons based on matching base names.
    
    Returns tuple of (valid_indices_1, valid_indices_2, base_names).
    """
    base_1 = {i: extract_base_name(bid) for i, bid in enumerate(ids_1)}
    base_2 = {i: extract_base_name(bid) for i, bid in enumerate(ids_2)}
    
    # Build lookup for second set
    lookup_2 = {}
    for i, base in base_2.items():
        if base not in lookup_2:
            lookup_2[base] = i
    
    valid_1, valid_2, names = [], [], []
    for i1, base in base_1.items():
        if base in lookup_2:
            valid_1.append(i1)
            valid_2.append(lookup_2[base])
            names.append(base)
    
    return valid_1, valid_2, names

def plot_theo_50Hz_trajectories():
    """Plot how theo concentration changes affect PCA positioning at 50Hz.
    
    Note: Only 1.5mM and 4mM can be paired (when base IDs match).
          2.5mM at 50Hz is ALWAYS unpaired and shown without connections.
    """
    
    plt.figure(figsize=(14, 8))
    
    # Background: WT pooled (standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.3, s=30, label='WT pooled')
    
    # Get projected data from pca_data dictionary
    theo_1_5_coords = pca_data['50Hz_1_5Ca']
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    theo_4_coords   = pca_data['50Hz_4Ca']
    
    # Get 20Hz data for centroids
    theo_1_5_20Hz_coords = pca_data['WT_1_5Ca']
    theo_4_20Hz_coords   = pca_data['WT_4Ca']
    
    # Get IDs for pairing
    theo_1_5_ids = PCA_Data_50Hz_1_5_Ca['ID'].values
    theo_4_ids   = PCA_Data_50Hz_4_Ca['ID'].values
    # Note: theo_2_5_ids not used for pairing (always unpaired)

    # theo 1.5mM @ 50Hz - blue triangles pointing down
    plt.scatter(theo_1_5_coords[:, 0], theo_1_5_coords[:, 1], 
               marker='v', s=60, c='blue', alpha=0.8, edgecolors='darkblue', linewidth=0.5,
               label=f'Theo 1.5mM 50Hz (n={len(theo_1_5_coords)})')
    
    # theo 2.5mM @ 50Hz - orange diamonds (unpaired)
    plt.scatter(theo_2_5_coords[:, 0], theo_2_5_coords[:, 1], 
               marker='D', s=60, c='orange', alpha=0.8, edgecolors='darkorange', linewidth=0.5,
               label=f'Theo 2.5mM 50Hz (n={len(theo_2_5_coords)}) [unpaired]')
    
    # theo 4mM @ 50Hz - red triangles pointing up  
    plt.scatter(theo_4_coords[:, 0], theo_4_coords[:, 1], 
               marker='^', s=60, c='red', alpha=0.8, edgecolors='darkred', linewidth=0.5,
               label=f'Theo 4mM 50Hz (n={len(theo_4_coords)})')
    
    # Calculate centroids for 50Hz
    center_pooled   = np.mean(pca_coordinates, axis=0)
    center_theo_1_5 = np.mean(theo_1_5_coords, axis=0)
    center_theo_2_5 = np.mean(theo_2_5_coords, axis=0)
    center_theo_4   = np.mean(theo_4_coords, axis=0)
    
    # Calculate centroids for 20Hz
    center_theo_1_5_20Hz = np.mean(theo_1_5_20Hz_coords, axis=0)
    center_theo_4_20Hz   = np.mean(theo_4_20Hz_coords, axis=0)
    
    # Plot 50Hz centroids
    plt.scatter(center_pooled[0], center_pooled[1], marker='s', s=150, c='yellow', 
               edgecolor='darkorange', linewidth=2, label='WT pooled centroid', zorder=5)
    plt.scatter(center_theo_1_5[0], center_theo_1_5[1], marker='X', s=200, c='blue', 
               edgecolor='black', linewidth=2, label='1.5mM 50Hz centroid', zorder=5)
    plt.scatter(center_theo_2_5[0], center_theo_2_5[1], marker='X', s=200, c='orange', 
               edgecolor='black', linewidth=2, label='2.5mM 50Hz centroid', zorder=5)
    plt.scatter(center_theo_4[0], center_theo_4[1], marker='X', s=200, c='red', 
               edgecolor='black', linewidth=2, label='4mM 50Hz centroid', zorder=5)
    
    # Plot 20Hz centroids (different markers - squares)
    plt.scatter(center_theo_1_5_20Hz[0], center_theo_1_5_20Hz[1], marker='s', s=150, c='lightblue', 
               edgecolor='darkblue', linewidth=2, label=f'1.5mM 20Hz centroid (n={len(theo_1_5_20Hz_coords)})', zorder=5)
    plt.scatter(center_theo_4_20Hz[0], center_theo_4_20Hz[1], marker='s', s=150, c='lightcoral', 
               edgecolor='darkred', linewidth=2, label=f'4mM 20Hz centroid (n={len(theo_4_20Hz_coords)})', zorder=5)
    
    # === ARROWS BETWEEN CENTROIDS ===
    arrow_props = dict(arrowstyle='->', color='black', lw=2, mutation_scale=15)
    
    # Arrow: WT pooled → Theo 2.5mM 50Hz
    plt.annotate('', xy=(center_theo_2_5[0], center_theo_2_5[1]), 
                 xytext=(center_pooled[0], center_pooled[1]),
                 arrowprops=dict(arrowstyle='->', color='darkorange', lw=2.5, mutation_scale=20),
                 zorder=6)
    
    # Arrow: 1.5mM 20Hz → 1.5mM 50Hz
    plt.annotate('', xy=(center_theo_1_5[0], center_theo_1_5[1]), 
                 xytext=(center_theo_1_5_20Hz[0], center_theo_1_5_20Hz[1]),
                 arrowprops=dict(arrowstyle='->', color='blue', lw=2.5, mutation_scale=20),
                 zorder=6)
    
    # Arrow: 4mM 20Hz → 4mM 50Hz
    plt.annotate('', xy=(center_theo_4[0], center_theo_4[1]), 
                 xytext=(center_theo_4_20Hz[0], center_theo_4_20Hz[1]),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2.5, mutation_scale=20),
                 zorder=6)
    
    # Annotate 50Hz centroid coordinates
    plt.text(center_pooled[0], center_pooled[1]-0.3, 
             f'WT pooled\n({center_pooled[0]:.2f},{center_pooled[1]:.2f})', 
             color='darkorange', fontsize=9, ha='center', va='top', fontweight='bold')
    plt.text(center_theo_1_5[0], center_theo_1_5[1]-0.3, 
             f'1.5mM 50Hz\n({center_theo_1_5[0]:.2f},{center_theo_1_5[1]:.2f})', 
             color='blue', fontsize=9, ha='center', va='top', fontweight='bold')
    plt.text(center_theo_2_5[0], center_theo_2_5[1]+0.3, 
             f'2.5mM 50Hz\n({center_theo_2_5[0]:.2f},{center_theo_2_5[1]:.2f})', 
             color='darkorange', fontsize=9, ha='center', va='bottom', fontweight='bold')
    plt.text(center_theo_4[0], center_theo_4[1]+0.3, 
             f'4mM 50Hz\n({center_theo_4[0]:.2f},{center_theo_4[1]:.2f})', 
             color='red', fontsize=9, ha='center', va='bottom', fontweight='bold')
    
    # Annotate 20Hz centroid coordinates
    plt.text(center_theo_1_5_20Hz[0]+0.3, center_theo_1_5_20Hz[1], 
             f'1.5mM 20Hz\n({center_theo_1_5_20Hz[0]:.2f},{center_theo_1_5_20Hz[1]:.2f})', 
             color='darkblue', fontsize=8, ha='left', va='center', fontweight='bold')
    plt.text(center_theo_4_20Hz[0]+0.3, center_theo_4_20Hz[1], 
             f'4mM 20Hz\n({center_theo_4_20Hz[0]:.2f},{center_theo_4_20Hz[1]:.2f})', 
             color='darkred', fontsize=8, ha='left', va='center', fontweight='bold')
    
    # Connect ONLY valid paired boutons between 1.5mM and 4mM based on matching base IDs
    # (2.5mM is ALWAYS unpaired, so no connections drawn for it)
    valid_1_5_idx, valid_4_idx, pair_names = find_valid_pairs(
        theo_1_5_ids, theo_4_ids, theo_1_5_coords, theo_4_coords
    )
    
    n_pairs = len(valid_1_5_idx)
    if n_pairs > 0:
        for i1, i4 in zip(valid_1_5_idx, valid_4_idx):
            # Connect 1.5 → 4 mM only (no 2.5 since it's unpaired)
            plt.plot([theo_1_5_coords[i1, 0], theo_4_coords[i4, 0]],
                     [theo_1_5_coords[i1, 1], theo_4_coords[i4, 1]],
                     color='gray', alpha=0.4, linewidth=1.0, linestyle='--')
        print(f"Connected {n_pairs} valid 1.5mM ↔ 4mM bouton pairs (matched by base ID)")
    else:
        print("No valid pairs found between 1.5mM and 4mM conditions")
    
    print(f"Note: 2.5mM at 50Hz ({len(theo_2_5_coords)} boutons) is ALWAYS unpaired")
    
    # Calculate and print 20Hz centroid distance
    dist_20Hz_centroids = np.linalg.norm(center_theo_4_20Hz - center_theo_1_5_20Hz)
    print(f"\n20Hz centroid distance (1.5mM → 4mM): {dist_20Hz_centroids:.3f}")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('Theo Concentration Effects on Bouton Properties (50Hz + 20Hz centroids)\n[1.5↔4mM paired, 2.5mM unpaired]')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "Fig5_theo_50Hz_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return (center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, 
            center_theo_1_5_20Hz, center_theo_4_20Hz, n_pairs, valid_1_5_idx, valid_4_idx)

def calculate_theo_50Hz_movements(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, 
                                   center_theo_1_5_20Hz, center_theo_4_20Hz,
                                   n_pairs, valid_1_5_idx, valid_4_idx):
    """Calculate movement statistics for theo concentration changes at 50Hz."""
    
    # Get coordinates
    theo_1_5_coords = pca_data['50Hz_1_5Ca']
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    theo_4_coords   = pca_data['50Hz_4Ca']

    # Distances from WT pooled centroid to each theo level centroid
    dist_to_1_5 = np.linalg.norm(center_theo_1_5 - center_pooled)
    dist_to_2_5 = np.linalg.norm(center_theo_2_5 - center_pooled)
    dist_to_4   = np.linalg.norm(center_theo_4 - center_pooled)
    
    # Distance between 20Hz centroids
    dist_20Hz_1_5_to_4 = np.linalg.norm(center_theo_4_20Hz - center_theo_1_5_20Hz)
    
    # Movement from standard to each theo concentration
    movements_to_1_5  = []
    n_1_5_comparisons = min(len(pca_coordinates), len(theo_1_5_coords))
    for i in range(n_1_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_1_5_coords[i])
        movements_to_1_5.append(dist)
    
    movements_to_2_5  = []
    n_2_5_comparisons = min(len(pca_coordinates), len(theo_2_5_coords))
    for i in range(n_2_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_2_5_coords[i])
        movements_to_2_5.append(dist)
    
    movements_to_4  = []
    n_4_comparisons = min(len(pca_coordinates), len(theo_4_coords))
    for i in range(n_4_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_4_coords[i])
        movements_to_4.append(dist)
    
    # Movement between theo concentrations (ONLY valid paired boutons: 1.5 ↔ 4)
    movements_1_5_to_4 = []
    for i1, i4 in zip(valid_1_5_idx, valid_4_idx):
        dist = np.linalg.norm(theo_1_5_coords[i1] - theo_4_coords[i4])
        movements_1_5_to_4.append(dist)
    
    return {
        'centroid_distances': {
            '1.5mM': dist_to_1_5,
            '2.5mM': dist_to_2_5,
            '4mM': dist_to_4,
            '20Hz_1.5_to_4': dist_20Hz_1_5_to_4
        },
        'individual_movements': {
            'to_1.5': movements_to_1_5,
            'to_2.5': movements_to_2_5,
            'to_4': movements_to_4,
            '1.5_to_4': movements_1_5_to_4
        }
    }

# Run analysis
(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, 
 center_theo_1_5_20Hz, center_theo_4_20Hz, n_pairs, valid_1_5_idx, valid_4_idx) = plot_theo_50Hz_trajectories()
movement_stats = calculate_theo_50Hz_movements(center_pooled, center_theo_1_5, center_theo_2_5, center_theo_4, 
                                                center_theo_1_5_20Hz, center_theo_4_20Hz,
                                                n_pairs, valid_1_5_idx, valid_4_idx)

# Display results
print(f"\n=== THEO CONCENTRATION ANALYSIS (50Hz + 20Hz) ===")
print(f"50Hz Centroid coordinates:")
print(f"  WT pooled:      ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  Theo 1.5mM:     ({center_theo_1_5[0]:.3f}, {center_theo_1_5[1]:.3f}) - distance from WT: {movement_stats['centroid_distances']['1.5mM']:.3f}")
print(f"  Theo 2.5mM:     ({center_theo_2_5[0]:.3f}, {center_theo_2_5[1]:.3f}) - distance from WT: {movement_stats['centroid_distances']['2.5mM']:.3f} [UNPAIRED]")
print(f"  Theo 4mM:       ({center_theo_4[0]:.3f}, {center_theo_4[1]:.3f}) - distance from WT: {movement_stats['centroid_distances']['4mM']:.3f}")

print(f"\n20Hz Centroid coordinates:")
print(f"  Theo 1.5mM:     ({center_theo_1_5_20Hz[0]:.3f}, {center_theo_1_5_20Hz[1]:.3f})")
print(f"  Theo 4mM:       ({center_theo_4_20Hz[0]:.3f}, {center_theo_4_20Hz[1]:.3f})")
print(f"  Distance 1.5→4: {movement_stats['centroid_distances']['20Hz_1.5_to_4']:.3f}")

print(f"\nIndividual bouton movements from WT pooled:")
if movement_stats['individual_movements']['to_1.5']:
    moves = movement_stats['individual_movements']['to_1.5']
    print(f"WT → Theo 1.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

if movement_stats['individual_movements']['to_2.5']:
    moves = movement_stats['individual_movements']['to_2.5']
    print(f"WT → Theo 2.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f} [UNPAIRED]")

if movement_stats['individual_movements']['to_4']:
    moves = movement_stats['individual_movements']['to_4']
    print(f"WT → Theo 4mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f}")

print(f"\nPairwise movements (valid pairs only):")
if movement_stats['individual_movements']['1.5_to_4']:
    moves = movement_stats['individual_movements']['1.5_to_4']
    print(f"1.5mM ↔ 4mM (n={len(moves)} valid pairs): {np.mean(moves):.3f} ± {np.std(moves):.3f}")
else:
    print("No valid 1.5mM ↔ 4mM pairs found")

print(f"\n✓ Theo 50Hz + 20Hz trajectory analysis complete")
print(f"✓ Saved to {OUTPUT_DIR / 'theo_50Hz_concentration_pca_trajectories.pdf'}")


In [ ]:
# Analyze theo concentration effects on bouton properties in PCA space

def extract_base_name(bouton_id):
    """Extract base name (date + linescan + bouton) for pairing across conditions.
    
    Examples: 
        '20210721_linescan1_50Hz_10pulses_4mMCa_bouton1_traces_converted' 
            -> '20210721_linescan1_bouton1'
        '20191017_linescan2_20Hz_10pulses_2.5mMCa_bouton2_traces_converted'
            -> '20191017_linescan2_bouton2'
    """
    import re
    bid = str(bouton_id)
    # Remove _traces_converted suffix
    bid = re.sub(r'_traces_converted$', '', bid)
    # Remove Hz frequency patterns (e.g. _20Hz_, _50Hz_)
    bid = re.sub(r'_\d+Hz_?', '_', bid, flags=re.IGNORECASE)
    # Remove pulse count patterns (e.g. _10pulses_)
    bid = re.sub(r'_\d+pulses_?', '_', bid, flags=re.IGNORECASE)
    # Remove calcium concentration patterns (e.g. _4mMCa_, _1.5mMCa_, _2.5mMCa_, _1_5Ca, _4Ca)
    bid = re.sub(r'_\d+\.?\d*mMCa_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+[_.]?\d*Ca_?', '_', bid, flags=re.IGNORECASE)
    # Remove trailing underscores  
    bid = re.sub(r'_+$', '', bid)
    # Collapse multiple underscores
    bid = re.sub(r'_+', '_', bid)
    return bid

def plot_theo_50Hz_trajectories():
    """Plot how theo concentration changes affect PCA positioning at 50Hz.
    
    Note: Only 2.5mM at 50Hz is shown (unpaired).
    """
    
    plt.figure(figsize=(12, 8))
    
    # Background: WT pooled (standard condition)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.3, s=30, label='WT pooled')
    
    # Get projected data from pca_data dictionary (only 2.5mM)
    theo_2_5_coords = pca_data['50Hz_2_5Ca']
    
    # theo 2.5mM @ 50Hz - orange diamonds (unpaired)
    plt.scatter(theo_2_5_coords[:, 0], theo_2_5_coords[:, 1], 
               marker='D', s=60, c='orange', alpha=0.8, edgecolors='darkorange', linewidth=0.5,
               label=f'2.5mM 50Hz (n={len(theo_2_5_coords)}) [unpaired]')
    
    # Calculate centroids
    center_pooled   = np.mean(pca_coordinates, axis=0)
    center_theo_2_5 = np.mean(theo_2_5_coords, axis=0)
    
    # Plot centroids
    plt.scatter(center_theo_2_5[0], center_theo_2_5[1], marker='X', s=200, c='orange', 
               edgecolor='black', linewidth=2, label='2.5mM centroid', zorder=5)
    
    # Draw arrow from pooled centroid to theo 2.5mM centroid
    plt.arrow(center_pooled[0], center_pooled[1],
              center_theo_2_5[0] - center_pooled[0], center_theo_2_5[1] - center_pooled[1],
              color='black', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.7)
    
    # Annotate centroid coordinates
    plt.text(center_theo_2_5[0], center_theo_2_5[1]+0.3, 
             f'2.5mM\n({center_theo_2_5[0]:.2f},{center_theo_2_5[1]:.2f})', 
             color='darkorange', fontsize=9, ha='center', va='bottom', fontweight='bold')
    
    print(f"Note: Only 2.5mM at 50Hz ({len(theo_2_5_coords)} boutons) is shown (unpaired)")
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title('Effects on Bouton Properties (50Hz)\n[2.5mM unpaired only]')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "theo_50Hz_concentration_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_theo_2_5

def calculate_theo_50Hz_movements(center_pooled, center_theo_2_5):
    """Calculate movement statistics for theo concentration changes at 50Hz."""
    
    # Get coordinates (only 2.5mM)
    theo_2_5_coords = pca_data['50Hz_2_5Ca']

    # Distance from standard condition to 2.5mM
    dist_to_2_5 = np.linalg.norm(center_theo_2_5 - center_pooled)
    
    # Movement from standard to 2.5mM theo concentration
    movements_to_2_5  = []
    n_2_5_comparisons = min(len(pca_coordinates), len(theo_2_5_coords))
    for i in range(n_2_5_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - theo_2_5_coords[i])
        movements_to_2_5.append(dist)
    
    return {
        'centroid_distances': {
            '2.5mM': dist_to_2_5
        },
        'individual_movements': {
            'to_2.5': movements_to_2_5
        }
    }

# Run analysis
center_pooled, center_theo_2_5 = plot_theo_50Hz_trajectories()
movement_stats = calculate_theo_50Hz_movements(center_pooled, center_theo_2_5)

# Display results
print(f"\n=== THEO CONCENTRATION ANALYSIS (50Hz) ===")
print(f"Centroid coordinates:")
print(f"  WT pooled:      ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  Theo 2.5mM:     ({center_theo_2_5[0]:.3f}, {center_theo_2_5[1]:.3f}) - distance: {movement_stats['centroid_distances']['2.5mM']:.3f} [UNPAIRED]")

print(f"\nIndividual bouton movements from WT pooled:")
if movement_stats['individual_movements']['to_2.5']:
    moves = movement_stats['individual_movements']['to_2.5']
    print(f"WT → Theo 2.5mM (n={len(moves)}): {np.mean(moves):.3f} ± {np.std(moves):.3f} [UNPAIRED]")

print(f"\n✓ Theo 50Hz trajectory analysis complete (2.5mM only)")
print(f"✓ Saved to {OUTPUT_DIR / 'theo_50Hz_concentration_pca_trajectories.pdf'}")


In [ ]:
from scipy.stats import mannwhitneyu

# PPR profile comparison for clusters 1 and 3 at both 20Hz and 50Hz

# Get 50Hz data with cluster assignments
pca_50hz_coords = pca_data['50Hz_2_5Ca']

# Assign clusters to 50Hz data based on nearest cluster center
def assign_to_nearest_cluster(coords, cluster_centers):
    """Assign each point to its nearest cluster center."""
    assignments = []
    for point in coords:
        distances = [np.linalg.norm(point - center) for center in cluster_centers]
        assignments.append(np.argmin(distances) + 1)  # +1 because clusters are 1-indexed
    return np.array(assignments)

# Calculate cluster centers from WT pooled data
cluster_centers = []
for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_points = pca_coordinates[cluster_assignments == cluster_id]
    if len(cluster_points) > 0:
        cluster_centers.append(cluster_points.mean(axis=0))

# Assign 50Hz data to clusters
hz50_cluster_assignments = assign_to_nearest_cluster(pca_50hz_coords, cluster_centers)

# Filter for clusters 1 and 3 at 50Hz
cluster_1_mask_50 = hz50_cluster_assignments == 1
cluster_3_mask_50 = hz50_cluster_assignments == 3

cluster_1_data_50 = PCA_Data_50Hz_2_5_Ca[cluster_1_mask_50]
cluster_3_data_50 = PCA_Data_50Hz_2_5_Ca[cluster_3_mask_50]

# Filter for clusters 1 and 3 at 20Hz (from WT pooled data)
cluster_1_data_20 = PCA_Data_WT_Pooled_clustered[cluster_assignments == 1]
cluster_3_data_20 = PCA_Data_WT_Pooled_clustered[cluster_assignments == 3]

print(f"20Hz data (WT pooled):")
print(f"  Cluster 1: {len(cluster_1_data_20)} boutons")
print(f"  Cluster 3: {len(cluster_3_data_20)} boutons")
print(f"\n50Hz data assigned to clusters:")
print(f"  Cluster 1: {len(cluster_1_data_50)} boutons")
print(f"  Cluster 3: {len(cluster_3_data_50)} boutons")

# PPR columns
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in cluster_1_data_20.columns]
pulse_numbers = list(range(1, len(ppr_cols) + 2))

# Calculate PPR profiles for each cluster
def get_ppr_profile(data, ppr_cols):
    """Calculate PPR profile with mean and SEM."""
    means = [1.0] + data[ppr_cols].mean().tolist()
    sems = [0.0] + data[ppr_cols].sem().tolist()
    return means, sems

cluster_1_means_20, cluster_1_sems_20 = get_ppr_profile(cluster_1_data_20, ppr_cols)
cluster_3_means_20, cluster_3_sems_20 = get_ppr_profile(cluster_3_data_20, ppr_cols)
cluster_1_means_50, cluster_1_sems_50 = get_ppr_profile(cluster_1_data_50, ppr_cols)
cluster_3_means_50, cluster_3_sems_50 = get_ppr_profile(cluster_3_data_50, ppr_cols)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Get cluster colors
color_1 = get_cluster_color(1)
color_3 = get_cluster_color(3)

# Left panel: 20Hz
ax1.plot(pulse_numbers, cluster_1_means_20, marker='o', color=color_1, linewidth=2.5,
         label=f'Cluster 1 @ 20Hz (n={len(cluster_1_data_20)})', markersize=8)
ax1.fill_between(pulse_numbers, 
                 np.array(cluster_1_means_20) - np.array(cluster_1_sems_20),
                 np.array(cluster_1_means_20) + np.array(cluster_1_sems_20),
                 color=color_1, alpha=0.2)

ax1.plot(pulse_numbers, cluster_3_means_20, marker='s', color=color_3, linewidth=2.5,
         label=f'Cluster 3 @ 20Hz (n={len(cluster_3_data_20)})', markersize=8)
ax1.fill_between(pulse_numbers, 
                 np.array(cluster_3_means_20) - np.array(cluster_3_sems_20),
                 np.array(cluster_3_means_20) + np.array(cluster_3_sems_20),
                 color=color_3, alpha=0.2)

ax1.axhline(1, color='gray', linestyle='dotted', linewidth=2)
ax1.set_xlabel('Pulse Number', fontsize=12)
ax1.set_ylabel('PPR (A_n/A_1)', fontsize=12)
ax1.set_title('PPR Profiles at 20Hz: Cluster 1 vs Cluster 3', fontsize=14, fontweight='bold')
ax1.set_xticks(pulse_numbers)
ax1.legend(fontsize=11, frameon=True)
ax1.grid(True, alpha=0.3)

# Right panel: 50Hz
ax2.plot(pulse_numbers, cluster_1_means_50, marker='o', color=color_1, linewidth=2.5,
         label=f'Cluster 1 @ 50Hz (n={len(cluster_1_data_50)})', markersize=8)
ax2.fill_between(pulse_numbers, 
                 np.array(cluster_1_means_50) - np.array(cluster_1_sems_50),
                 np.array(cluster_1_means_50) + np.array(cluster_1_sems_50),
                 color=color_1, alpha=0.2)

ax2.plot(pulse_numbers, cluster_3_means_50, marker='s', color=color_3, linewidth=2.5,
         label=f'Cluster 3 @ 50Hz (n={len(cluster_3_data_50)})', markersize=8)
ax2.fill_between(pulse_numbers, 
                 np.array(cluster_3_means_50) - np.array(cluster_3_sems_50),
                 np.array(cluster_3_means_50) + np.array(cluster_3_sems_50),
                 color=color_3, alpha=0.2)

ax2.axhline(1, color='gray', linestyle='dotted', linewidth=2)
ax2.set_xlabel('Pulse Number', fontsize=12)
ax2.set_ylabel('PPR (A_n/A_1)', fontsize=12)
ax2.set_title('PPR Profiles at 50Hz: Cluster 1 vs Cluster 3', fontsize=14, fontweight='bold')
ax2.set_xticks(pulse_numbers)
ax2.legend(fontsize=11, frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save figure
output_file = OUTPUT_DIR / "51_ppr_20Hz_50Hz_cluster1_vs_cluster3.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print(f"\n=== PPR PROFILE COMPARISON ===")
print(f"\n20Hz - Cluster 1 (n={len(cluster_1_data_20)}):")
print(f"  PPR2/1: {cluster_1_data_20['PPR2/1'].mean():.3f} ± {cluster_1_data_20['PPR2/1'].std():.3f}")
print(f"  PPR10/1: {cluster_1_data_20['PPR10/1'].mean():.3f} ± {cluster_1_data_20['PPR10/1'].std():.3f}")

print(f"\n20Hz - Cluster 3 (n={len(cluster_3_data_20)}):")
print(f"  PPR2/1: {cluster_3_data_20['PPR2/1'].mean():.3f} ± {cluster_3_data_20['PPR2/1'].std():.3f}")
print(f"  PPR10/1: {cluster_3_data_20['PPR10/1'].mean():.3f} ± {cluster_3_data_20['PPR10/1'].std():.3f}")

print(f"\n50Hz - Cluster 1 (n={len(cluster_1_data_50)}):")
print(f"  PPR2/1: {cluster_1_data_50['PPR2/1'].mean():.3f} ± {cluster_1_data_50['PPR2/1'].std():.3f}")
print(f"  PPR10/1: {cluster_1_data_50['PPR10/1'].mean():.3f} ± {cluster_1_data_50['PPR10/1'].std():.3f}")

print(f"\n50Hz - Cluster 3 (n={len(cluster_3_data_50)}):")
print(f"  PPR2/1: {cluster_3_data_50['PPR2/1'].mean():.3f} ± {cluster_3_data_50['PPR2/1'].std():.3f}")
print(f"  PPR10/1: {cluster_3_data_50['PPR10/1'].mean():.3f} ± {cluster_3_data_50['PPR10/1'].std():.3f}")

# Statistical comparison (20Hz)
ppr2_u_20, ppr2_p_20 = mannwhitneyu(cluster_1_data_20['PPR2/1'].dropna(), 
                                     cluster_3_data_20['PPR2/1'].dropna(), 
                                     alternative='two-sided')
ppr10_u_20, ppr10_p_20 = mannwhitneyu(cluster_1_data_20['PPR10/1'].dropna(), 
                                       cluster_3_data_20['PPR10/1'].dropna(), 
                                       alternative='two-sided')

# Statistical comparison (50Hz)
ppr2_u_50, ppr2_p_50 = mannwhitneyu(cluster_1_data_50['PPR2/1'].dropna(), 
                                     cluster_3_data_50['PPR2/1'].dropna(), 
                                     alternative='two-sided')
ppr10_u_50, ppr10_p_50 = mannwhitneyu(cluster_1_data_50['PPR10/1'].dropna(), 
                                       cluster_3_data_50['PPR10/1'].dropna(), 
                                       alternative='two-sided')

print(f"\nStatistical tests (Mann-Whitney U) - 20Hz:")
print(f"  PPR2/1: U={ppr2_u_20:.1f}, p={ppr2_p_20:.4g}")
print(f"  PPR10/1: U={ppr10_u_20:.1f}, p={ppr10_p_20:.4g}")

print(f"\nStatistical tests (Mann-Whitney U) - 50Hz:")
print(f"  PPR2/1: U={ppr2_u_50:.1f}, p={ppr2_p_50:.4g}")
print(f"  PPR10/1: U={ppr10_u_50:.1f}, p={ppr10_p_50:.4g}")

print(f"\n✓ Saved to {output_file}")

In [ ]:
# Show 20Hz to 50Hz evolution for valid 2.5mM pairs
# This plot exclusively shows paired boutons from 2.5mM 20Hz → 2.5mM 50Hz

def plot_2_5mM_20Hz_to_50Hz_evolution():
    """Plot PCA evolution from 2.5mM @ 20Hz to 2.5mM @ 50Hz for valid pairs only.
    
    Valid pairs have the same base name (date + linescan + bouton number).
    """
    
    # Get 20Hz 2.5mM data (this is the WT pooled 2.5mM, same as standard condition)
    # The standard WT pooled is 2.5mM Ca at 20Hz by default
    # We need to find boutons that exist in both 20Hz and 50Hz at 2.5mM
    
    # Get IDs for 50Hz 2.5mM
    hz50_2_5_ids = PCA_Data_50Hz_2_5_Ca['ID'].values
    hz50_2_5_coords = pca_data['50Hz_2_5Ca']
    
    # Get IDs for 20Hz (WT pooled, which is 2.5mM at 20Hz)
    hz20_ids = PCA_Data_WT_Pooled['ID'].values
    hz20_coords = pca_coordinates  # Already computed from WT pooled
    
    # Find valid pairs between 20Hz and 50Hz at 2.5mM using base name matching
    valid_20_idx, valid_50_idx, pair_names = find_valid_pairs(
        hz20_ids, hz50_2_5_ids, hz20_coords, hz50_2_5_coords
    )
    
    n_pairs = len(valid_20_idx)
    
    if n_pairs == 0:
        print("No valid 2.5mM 20Hz ↔ 50Hz pairs found")
        print(f"  20Hz (WT pooled): {len(hz20_ids)} boutons")
        print(f"  50Hz 2.5mM: {len(hz50_2_5_ids)} boutons")
        print("\nSample IDs from 20Hz:")
        for bid in hz20_ids[:5]:
            print(f"  {bid} → base: {extract_base_name(bid)}")
        print("\nSample IDs from 50Hz 2.5mM:")
        for bid in hz50_2_5_ids[:5]:
            print(f"  {bid} → base: {extract_base_name(bid)}")
        return None
    
    # Extract coordinates for valid pairs
    hz20_valid = np.array([hz20_coords[i] for i in valid_20_idx])
    hz50_valid = np.array([hz50_2_5_coords[i] for i in valid_50_idx])
    
    # Create plot
    plt.figure(figsize=(12, 8))
    
    # Background: All WT pooled points (faded)
    plt.scatter(pca_coordinates[:, 0], pca_coordinates[:, 1], 
               c=get_cluster_colors(cluster_assignments), alpha=0.2, s=20, label='WT pooled (all)')
    
    # Background: All 50Hz 2.5mM points (faded)
    plt.scatter(hz50_2_5_coords[:, 0], hz50_2_5_coords[:, 1],
               c='orange', alpha=0.2, s=20, marker='D')
    
    # Highlight: Valid paired 20Hz points (green circles)
    plt.scatter(hz20_valid[:, 0], hz20_valid[:, 1],
               marker='o', s=80, c='green', alpha=0.9, edgecolors='darkgreen', linewidth=1,
               label=f'20Hz 2.5mM (n={n_pairs} paired)', zorder=4)
    
    # Highlight: Valid paired 50Hz points (purple squares)
    plt.scatter(hz50_valid[:, 0], hz50_valid[:, 1],
               marker='s', s=80, c='purple', alpha=0.9, edgecolors='darkviolet', linewidth=1,
               label=f'50Hz 2.5mM (n={n_pairs} paired)', zorder=4)
    
    # Draw connecting lines for each pair
    for i in range(n_pairs):
        plt.plot([hz20_valid[i, 0], hz50_valid[i, 0]],
                 [hz20_valid[i, 1], hz50_valid[i, 1]],
                 color='gray', alpha=0.5, linewidth=1.2, linestyle='-')
    
    # Calculate and plot centroids
    center_20 = np.mean(hz20_valid, axis=0)
    center_50 = np.mean(hz50_valid, axis=0)
    
    plt.scatter(center_20[0], center_20[1], marker='X', s=250, c='green',
               edgecolor='black', linewidth=2, label='20Hz centroid', zorder=6)
    plt.scatter(center_50[0], center_50[1], marker='X', s=250, c='purple',
               edgecolor='black', linewidth=2, label='50Hz centroid', zorder=6)
    
    # Draw arrow from 20Hz to 50Hz centroid
    plt.arrow(center_20[0], center_20[1],
              center_50[0] - center_20[0], center_50[1] - center_20[1],
              color='black', width=0.03, head_width=0.2, length_includes_head=True, 
              alpha=0.8, zorder=5)
    
    # Annotate centroids
    plt.text(center_20[0]-0.2, center_20[1]+0.3,
             f'20Hz\n({center_20[0]:.2f},{center_20[1]:.2f})',
             color='darkgreen', fontsize=10, ha='right', va='bottom', fontweight='bold')
    plt.text(center_50[0]+0.2, center_50[1]-0.3,
             f'50Hz\n({center_50[0]:.2f},{center_50[1]:.2f})',
             color='darkviolet', fontsize=10, ha='left', va='top', fontweight='bold')
    
    # Calculate statistics
    centroid_distance = np.linalg.norm(center_50 - center_20)
    individual_distances = [np.linalg.norm(hz50_valid[i] - hz20_valid[i]) for i in range(n_pairs)]
    mean_distance = np.mean(individual_distances)
    std_distance = np.std(individual_distances)
    
    # Format plot
    pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
    plt.xlabel(f'PC1 ({pc1_variance:.1%} variance)')
    plt.ylabel(f'PC2 ({pc2_variance:.1%} variance)')
    plt.title(f'2.5mM Ca: 20Hz → 50Hz Evolution\n{n_pairs} valid paired boutons')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "52_2_5mM_20Hz_to_50Hz_pca_evolution.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print(f"\n=== 2.5mM Ca: 20Hz → 50Hz EVOLUTION ===")
    print(f"Valid pairs found: {n_pairs}")
    print(f"\nCentroid movement:")
    print(f"  20Hz centroid: ({center_20[0]:.3f}, {center_20[1]:.3f})")
    print(f"  50Hz centroid: ({center_50[0]:.3f}, {center_50[1]:.3f})")
    print(f"  Centroid distance: {centroid_distance:.3f}")
    print(f"\nIndividual bouton movements:")
    print(f"  Mean: {mean_distance:.3f} ± {std_distance:.3f}")
    print(f"  Min: {min(individual_distances):.3f}, Max: {max(individual_distances):.3f}")
    print(f"\n✓ Saved to {output_file}")
    
    return {
        'n_pairs': n_pairs,
        'pair_names': pair_names,
        'centroid_20': center_20,
        'centroid_50': center_50,
        'centroid_distance': centroid_distance,
        'individual_distances': individual_distances
    }

# Run the 20Hz → 50Hz evolution analysis for 2.5mM
evolution_stats = plot_2_5mM_20Hz_to_50Hz_evolution()

### J.2 Cluster Composition Comparison

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Compare cluster distributions between 20Hz WT and 50Hz for each calcium concentration
# Three separate figures: 1.5mM Ca, 2.5mM Ca, 4mM Ca


# Get WT cluster counts (20Hz baseline)
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
all_clusters = sorted(wt_cluster_counts.index)

# Train kNN classifier on WT pooled data
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(pca_coordinates, cluster_assignments)

# Project 50Hz conditions and assign to clusters
hz50_1_5_assignments = knn.predict(pca_data['50Hz_1_5Ca'])
hz50_2_5_assignments = knn.predict(pca_data['50Hz_2_5Ca'])
hz50_4_assignments   = knn.predict(pca_data['50Hz_4Ca'])

hz50_1_5_counts = pd.Series(hz50_1_5_assignments).value_counts()
hz50_2_5_counts = pd.Series(hz50_2_5_assignments).value_counts()
hz50_4_counts   = pd.Series(hz50_4_assignments).value_counts()

#Project 20Hz calcium condition data and assign to clusters
hz20_1_5_assignments = knn.predict(pca_data['WT_1_5Ca'])
hz20_4_assignments   = knn.predict(pca_data['WT_4Ca'])

hz20_1_5_counts = pd.Series(hz20_1_5_assignments).value_counts()
hz20_4_counts   = pd.Series(hz20_4_assignments).value_counts()

# Ensure all clusters represented
hz50_1_5_complete = pd.Series([hz50_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_2_5_complete = pd.Series([hz50_2_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_4_complete   = pd.Series([hz50_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

hz20_1_5_complete = pd.Series([hz20_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz20_4_complete   = pd.Series([hz20_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)

hz50_1_5_percentages = 100 * hz50_1_5_complete / len(hz50_1_5_assignments)
hz50_2_5_percentages = 100 * hz50_2_5_complete / len(hz50_2_5_assignments)
hz50_4_percentages   = 100 * hz50_4_complete / len(hz50_4_assignments)

hz20_1_5_percentages = 100 * hz20_1_5_complete / len(hz20_1_5_assignments)
hz20_4_percentages   = 100 * hz20_4_complete / len(hz20_4_assignments)

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

def plot_cluster_comparison(wt_pct, hz50_pct, n_wt, n_50, ca_conc, output_name):
    """Plot stacked bar comparison for a single calcium concentration."""
    fig, ax = plt.subplots(figsize=(6, 6))
    bar_width = 0.5
    
    conditions = [
        (0, wt_pct, n_wt, f'20Hz {ca_conc}\n(n={n_wt})'),
        (1, hz50_pct, n_50, f'50Hz {ca_conc}\n(n={n_50})')
    ]
    
    for pos, percentages, n_samples, label in conditions:
        bottom = 0
        for cluster_id in all_clusters:
            color = get_cluster_color(cluster_id)
            pct = percentages.iloc[cluster_id - 1] if cluster_id <= len(percentages) else 0
            
            ax.bar(pos, pct, bar_width, bottom=bottom, color=color,
                   alpha=0.9, label=f'Cluster {cluster_id}' if pos == 0 else None)
            
            if pct > 5:
                ax.text(pos, bottom + pct/2, f"{pct:.1f}%", ha='center', va='center',
                        color=get_text_color(color), fontsize=9, fontweight='bold')
            
            bottom += pct
    
    ax.set_ylabel('Percentage (%)', fontsize=11)
    ax.set_title(f'Cluster Distribution: 20Hz vs 50Hz @ {ca_conc}', fontsize=12, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels([label for _, _, _, label in conditions])
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
    
    handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
    ax.legend(handles, [f'Cluster {c}' for c in all_clusters],
              bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    
    plt.tight_layout()
    output_file = OUTPUT_DIR / output_name
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved to {output_file}")

# Plot three figures
plot_cluster_comparison(hz20_1_5_percentages, hz50_1_5_percentages, 
                        len(hz20_1_5_assignments), len(hz50_1_5_assignments),
                        '1.5mM Ca', 'fig5_cluster_distribution_1_5mM_Ca.pdf')

plot_cluster_comparison(wt_percentages, hz50_2_5_percentages,
                        len(cluster_assignments), len(hz50_2_5_assignments),
                        '2.5mM Ca', 'fig5_cluster_distribution_2_5mM_Ca.pdf')
plot_cluster_comparison(hz20_4_percentages, hz50_4_percentages,
                        len(hz20_4_assignments), len(hz50_4_assignments),
                        '4mM Ca', 'fig5_cluster_distribution_4mM_Ca.pdf')

print("\n=== CLUSTER DISTRIBUTION SUMMARY ===")
for ca, pct, n in [('1.5mM', hz50_1_5_percentages, len(hz50_1_5_assignments)),
                   ('2.5mM', hz50_2_5_percentages, len(hz50_2_5_assignments)),
                   ('4mM', hz50_4_percentages, len(hz50_4_assignments))]:
    print(f"\n50Hz @ {ca} (n={n}):")
    for c in all_clusters:
        print(f"  Cluster {c}: {pct.iloc[c-1]:.1f}%")


### J.3 PPR profiles

In [ ]:
# Compare PPR profiles across WT and 50Hz conditions at different calcium concentrations
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns]
pulse_numbers = list(range(1, len(ppr_cols) + 2))

# Calculate PPR profiles for each condition
def get_ppr_profile(data, condition_name):
    means = [1.0] + data[ppr_cols].mean().tolist()
    sems = [0.0] + data[ppr_cols].sem().tolist()
    return means, sems

# Get profiles for each condition
means_wt, sems_wt = get_ppr_profile(PCA_Data_WT_Pooled, 'WT 2.5mM Ca')
means_50hz_1_5, sems_50hz_1_5 = get_ppr_profile(PCA_Data_50Hz_1_5_Ca, '50Hz 1.5mM Ca')  # Fixed: added underscore
means_50hz_2_5, sems_50hz_2_5 = get_ppr_profile(PCA_Data_50Hz_2_5_Ca, '50Hz 2.5mM Ca')  # Fixed: added underscore
means_50hz_4, sems_50hz_4 = get_ppr_profile(PCA_Data_50Hz_4_Ca, '50Hz 4mM Ca')  # Fixed: added underscore


# Plot PPR profiles
plt.figure(figsize=(10, 6))

# WT pooled (black - reference)
plt.plot(pulse_numbers, means_wt, marker='o', color='black', linewidth=2,
         label=f'WT 2.5mM Ca (n={len(PCA_Data_WT_Pooled)})')
plt.fill_between(pulse_numbers, np.array(means_wt) - np.array(sems_wt),
                 np.array(means_wt) + np.array(sems_wt), color='black', alpha=0.15)

# 50Hz 1.5mM Ca (blue)
plt.plot(pulse_numbers, means_50hz_1_5, marker='v', color='blue', linewidth=2,
         label=f'50Hz 1.5mM Ca (n={len(PCA_Data_50Hz_1_5_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_50hz_1_5) - np.array(sems_50hz_1_5),
                 np.array(means_50hz_1_5) + np.array(sems_50hz_1_5), color='blue', alpha=0.2)

# 50Hz 2.5mM Ca (orange)
plt.plot(pulse_numbers, means_50hz_2_5, marker='D', color='orange', linewidth=2,
         label=f'50Hz 2.5mM Ca (n={len(PCA_Data_50Hz_2_5_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_50hz_2_5) - np.array(sems_50hz_2_5),
                 np.array(means_50hz_2_5) + np.array(sems_50hz_2_5), color='orange', alpha=0.2)

# 50Hz 4mM Ca (red)
plt.plot(pulse_numbers, means_50hz_4, marker='^', color='red', linewidth=2,
         label=f'50Hz 4mM Ca (n={len(PCA_Data_50Hz_4_Ca)})')
plt.fill_between(pulse_numbers, np.array(means_50hz_4) - np.array(sems_50hz_4),
                 np.array(means_50hz_4) + np.array(sems_50hz_4), color='red', alpha=0.2)

# Format plot
plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)
plt.xlabel('Pulse Number', fontsize=11)
plt.ylim(0.9,2.5)
plt.ylabel('PPR (A_n/A_1)', fontsize=11)
plt.title('PPR Profiles: WT vs 50Hz Conditions at Different Calcium Concentrations', fontsize=12, fontweight='bold')
plt.xticks(pulse_numbers)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save figure and data
output_fig = OUTPUT_DIR / "53_ppr_profiles_wt_vs_50hz_all_calcium.pdf"
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

# Save numerical data
output_data = OUTPUT_DIR / "fig5_ppr_profiles_wt_vs_50hz_data.txt"
with open(output_data, "w") as f:
    f.write("Pulse\tWT_Mean\tWT_SEM\t50Hz_1.5mM_Mean\t50Hz_1.5mM_SEM\t50Hz_2.5mM_Mean\t50Hz_2.5mM_SEM\t50Hz_4mM_Mean\t50Hz_4mM_SEM\n")
    for i, pulse in enumerate(pulse_numbers):
        f.write(f"{pulse}\t{means_wt[i]:.4f}\t{sems_wt[i]:.4f}\t"
                f"{means_50hz_1_5[i]:.4f}\t{sems_50hz_1_5[i]:.4f}\t"
                f"{means_50hz_2_5[i]:.4f}\t{sems_50hz_2_5[i]:.4f}\t"
                f"{means_50hz_4[i]:.4f}\t{sems_50hz_4[i]:.4f}\n")

# Print summary statistics
print("=== PPR PROFILE COMPARISON: WT vs 50Hz CONDITIONS ===")
print(f"\nWT 2.5mM Ca (n={len(PCA_Data_WT_Pooled)}):")
print(f"  PPR2/1: {means_wt[1]:.3f} ± {sems_wt[1]:.3f}")
print(f"  PPR10/1: {means_wt[-1]:.3f} ± {sems_wt[-1]:.3f}")

print(f"\n50Hz 1.5mM Ca (n={len(PCA_Data_50Hz_1_5_Ca)}):")
print(f"  PPR2/1: {means_50hz_1_5[1]:.3f} ± {sems_50hz_1_5[1]:.3f}")
print(f"  PPR10/1: {means_50hz_1_5[-1]:.3f} ± {sems_50hz_1_5[-1]:.3f}")

print(f"\n50Hz 2.5mM Ca (n={len(PCA_Data_50Hz_2_5_Ca)}):")
print(f"  PPR2/1: {means_50hz_2_5[1]:.3f} ± {sems_50hz_2_5[1]:.3f}")
print(f"  PPR10/1: {means_50hz_2_5[-1]:.3f} ± {sems_50hz_2_5[-1]:.3f}")

print(f"\n50Hz 4mM Ca (n={len(PCA_Data_50Hz_4_Ca)}):")
print(f"  PPR2/1: {means_50hz_4[1]:.3f} ± {sems_50hz_4[1]:.3f}")
print(f"  PPR10/1: {means_50hz_4[-1]:.3f} ± {sems_50hz_4[-1]:.3f}")

print(f"\n✓ Saved PPR profiles to {output_fig}")
print(f"✓ Saved numerical data to {output_data}")

### J.4 50Hz traces

In [ ]:
# Plot mean traces for all three 50Hz conditions on the same graph

# Extract traces for each 50Hz condition
hz50_1_5_traces = []
hz50_2_5_traces = []
hz50_4_traces = []

for _, row in NORM_TRACES_DATAFRAME.iterrows():
    if row['Condition'] == 'Theo_1_5_50Hz':
        hz50_1_5_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_2_5_50Hz':
        hz50_2_5_traces.append(row['Avg'])
    elif row['Condition'] == 'Theo_4_50Hz':
        hz50_4_traces.append(row['Avg'])

# Check if we have traces
if not (hz50_1_5_traces and hz50_2_5_traces and hz50_4_traces):
    print("50Hz trace conditions not found in resampled data")
    available_conditions = NORM_TRACES_DATAFRAME['Condition'].unique()
    print(f"Available conditions: {list(available_conditions)}")
else:
    # Calculate means and SEMs for each condition
    hz50_1_5_mean = np.nanmean(hz50_1_5_traces, axis=0)
    hz50_1_5_sem = np.nanstd(hz50_1_5_traces, axis=0, ddof=1) / np.sqrt(len(hz50_1_5_traces))
    
    hz50_2_5_mean = np.nanmean(hz50_2_5_traces, axis=0)
    hz50_2_5_sem = np.nanstd(hz50_2_5_traces, axis=0, ddof=1) / np.sqrt(len(hz50_2_5_traces))
    
    hz50_4_mean = np.nanmean(hz50_4_traces, axis=0)
    hz50_4_sem = np.nanstd(hz50_4_traces, axis=0, ddof=1) / np.sqrt(len(hz50_4_traces))
    
    # Create plot with all three conditions
    plt.figure(figsize=(12, 6))
    
    # 1.5mM Ca (blue)
    plt.plot(COMMON_TIME, hz50_1_5_mean, color='blue', linewidth=2, 
             label=f'50Hz 1.5mM Ca (n={len(hz50_1_5_traces)})')
    plt.fill_between(COMMON_TIME, hz50_1_5_mean - hz50_1_5_sem, 
                     hz50_1_5_mean + hz50_1_5_sem, color='blue', alpha=0.25)
    
    # 2.5mM Ca (orange)
    plt.plot(COMMON_TIME, hz50_2_5_mean, color='orange', linewidth=2,
             label=f'50Hz 2.5mM Ca (n={len(hz50_2_5_traces)})')
    plt.fill_between(COMMON_TIME, hz50_2_5_mean - hz50_2_5_sem, 
                     hz50_2_5_mean + hz50_2_5_sem, color='orange', alpha=0.25)
    
    # 4mM Ca (red)
    plt.plot(COMMON_TIME, hz50_4_mean, color='red', linewidth=2,
             label=f'50Hz 4mM Ca (n={len(hz50_4_traces)})')
    plt.fill_between(COMMON_TIME, hz50_4_mean - hz50_4_sem, 
                     hz50_4_mean + hz50_4_sem, color='red', alpha=0.25)
    
    # Add stimulus markers (10 pulses at 50Hz = every 20ms starting at 1.0s)
    stim_times = [1.0 + 0.02*i for i in range(10)]  # 50Hz = 20ms intervals
    for stim_time in stim_times:
        if stim_time <= 2.0:
            plt.axvline(stim_time, color='gray', linestyle='--', alpha=0.4, linewidth=1)
    
    # Formatting
    plt.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    plt.xlim(0.8, 2.0)
    plt.xlabel('Time (s)', fontsize=11)
    plt.ylabel('ΔF/F', fontsize=11)
    plt.xlim(0.898,1.4)
    plt.title('Mean Traces: 50Hz Stimulation at Different Calcium Concentrations', 
              fontsize=12, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "Fig5_50hz_mean_traces_all_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print(f"=== 50Hz MEAN TRACE COMPARISON ===")
    print(f"1.5mM Ca: n={len(hz50_1_5_traces)} traces")
    print(f"  Peak response: {hz50_1_5_mean.max():.3f} ΔF/F at t={COMMON_TIME[np.argmax(hz50_1_5_mean)]:.2f}s")
    
    print(f"\n2.5mM Ca: n={len(hz50_2_5_traces)} traces")
    print(f"  Peak response: {hz50_2_5_mean.max():.3f} ΔF/F at t={COMMON_TIME[np.argmax(hz50_2_5_mean)]:.2f}s")
    
    print(f"\n4mM Ca: n={len(hz50_4_traces)} traces")
    print(f"  Peak response: {hz50_4_mean.max():.3f} ΔF/F at t={COMMON_TIME[np.argmax(hz50_4_mean)]:.2f}s")
    
    print(f"\n✓ Saved mean traces to {output_file}")

### J.5 Boxplot


In [ ]:
from scipy.stats import mannwhitneyu

# Boxplot comparing AMP1 between 20Hz and 50Hz at 2.5mM Ca

# Extract data
amp1_20hz = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50hz = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()

# Create figure
fig, ax = plt.subplots(figsize=(6, 6))

# Prepare data for boxplot
data = [amp1_20hz.values, amp1_50hz.values]
positions = [0, 1]
colors = ['steelblue', 'orange']
labels = [f'AMP1 20Hz\n(n={len(amp1_20hz)})', f'AMP1 50Hz\n(n={len(amp1_50hz)})']

# Create boxplots
bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add strip points
for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)

# Statistical test
_, p_amp1 = mannwhitneyu(amp1_20hz, amp1_50hz, alternative='two-sided')

def get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# Add significance bar
y_max = max([d.max() for d in data]) * 1.1
ax.plot([0, 1], [y_max, y_max], 'k-', lw=1.5)
ax.text(0.5, y_max * 1.02, f'{get_stars(p_amp1)} (p={p_amp1:.2g})', ha='center', fontsize=10)

# Formatting
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Amplitude (ΔF/F)', fontsize=11)
ax.set_title('AMP1 Comparison: 20Hz vs 50Hz at 2.5mM Ca', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(top=y_max * 1.15)

plt.tight_layout()
output_file = OUTPUT_DIR / "Fig5_amp1_20Hz_vs_50Hz_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== AMP1 COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"AMP1 20Hz: {amp1_20hz.mean():.3f} ± {amp1_20hz.std():.3f} (n={len(amp1_20hz)})")
print(f"AMP1 50Hz: {amp1_50hz.mean():.3f} ± {amp1_50hz.std():.3f} (n={len(amp1_50hz)})")
print(f"  Mann-Whitney p = {p_amp1:.4g}")
print(f"\n✓ Saved to {output_file}")


In [ ]:
from scipy.stats import mannwhitneyu

# Boxplot comparing AMP1 (20Hz+50Hz) vs AMP2 (20Hz) vs AMP2 (50Hz) at 2.5mM Ca

# Extract data
amp1_20hz = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50hz = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()
amp1_combined = pd.concat([amp1_20hz, amp1_50hz], ignore_index=True)

amp2_20hz = PCA_Data_WT_Pooled['AMP2'].dropna()
amp2_50hz = PCA_Data_50Hz_2_5_Ca['AMP2'].dropna()

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Prepare data for boxplot
data = [amp1_combined.values, amp2_20hz.values, amp2_50hz.values]
positions = [0, 1, 2]
colors = ['gray', 'steelblue', 'orange']
labels = [f'AMP1\n20Hz+50Hz\n(n={len(amp1_combined)})', 
          f'AMP2\n20Hz\n(n={len(amp2_20hz)})', 
          f'AMP2\n50Hz\n(n={len(amp2_50hz)})']

# Create boxplots
bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add strip points
for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=20, alpha=0.4, edgecolors='black', linewidths=0.3)

# Statistical tests
_, p_amp1_vs_amp2_20 = mannwhitneyu(amp1_combined, amp2_20hz, alternative='two-sided')
_, p_amp1_vs_amp2_50 = mannwhitneyu(amp1_combined, amp2_50hz, alternative='two-sided')
_, p_amp2_20_vs_50 = mannwhitneyu(amp2_20hz, amp2_50hz, alternative='two-sided')

def get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# Formatting
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Amplitude (ΔF/F)', fontsize=11)
ax.set_title('AMP1 (pooled) vs AMP2 at 2.5mM Ca', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
output_file = OUTPUT_DIR / "Fig5_amp1_pooled_vs_amp2_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== AMPLITUDE COMPARISON @ 2.5mM Ca ===")
print(f"AMP1 (20Hz+50Hz): {amp1_combined.mean():.3f} ± {amp1_combined.std():.3f} (n={len(amp1_combined)})")
print(f"AMP2 20Hz: {amp2_20hz.mean():.3f} ± {amp2_20hz.std():.3f} (n={len(amp2_20hz)})")
print(f"AMP2 50Hz: {amp2_50hz.mean():.3f} ± {amp2_50hz.std():.3f} (n={len(amp2_50hz)})")
print(f"\nStatistics:")
print(f"  AMP1 vs AMP2 20Hz: p = {p_amp1_vs_amp2_20:.4g} ({get_stars(p_amp1_vs_amp2_20)})")
print(f"  AMP1 vs AMP2 50Hz: p = {p_amp1_vs_amp2_50:.4g} ({get_stars(p_amp1_vs_amp2_50)})")
print(f"  AMP2 20Hz vs 50Hz: p = {p_amp2_20_vs_50:.4g} ({get_stars(p_amp2_20_vs_50)})")
print(f"\n✓ Saved to {output_file}")

In [ ]:
from scipy.stats import mannwhitneyu

# Boxplot comparing %Fail1 between 20Hz and 50Hz at 2.5mM Ca

# Extract data
fail1_20hz = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_50hz = PCA_Data_50Hz_2_5_Ca['%Fail1'].dropna()

# Create figure
fig, ax = plt.subplots(figsize=(6, 6))

# Prepare data for boxplot
data = [fail1_20hz.values, fail1_50hz.values]
positions = [0, 1]
colors = ['steelblue', 'orange']
labels = [f'%Fail1 20Hz\n(n={len(fail1_20hz)})', f'%Fail1 50Hz\n(n={len(fail1_50hz)})']

# Create boxplots
bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add strip points
for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)

# Statistical test
_, p_fail1 = mannwhitneyu(fail1_20hz, fail1_50hz, alternative='two-sided')

def get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# Add significance bar
y_max = max([d.max() for d in data]) * 1.1
ax.plot([0, 1], [y_max, y_max], 'k-', lw=1.5)
ax.text(0.5, y_max * 1.02, f'{get_stars(p_fail1)} (p={p_fail1:.2g})', ha='center', fontsize=10)

# Formatting
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Failure Rate (%)', fontsize=11)
ax.set_title('%Fail1 Comparison: 20Hz vs 50Hz at 2.5mM Ca', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(top=y_max * 1.15)

plt.tight_layout()
output_file = OUTPUT_DIR / "Fig5_fail1_20Hz_vs_50Hz_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== %Fail1 COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"%Fail1 20Hz: {fail1_20hz.mean():.1f}% ± {fail1_20hz.std():.1f}% (n={len(fail1_20hz)})")
print(f"%Fail1 50Hz: {fail1_50hz.mean():.1f}% ± {fail1_50hz.std():.1f}% (n={len(fail1_50hz)})")
print(f"  Mann-Whitney p = {p_fail1:.4g}")
print(f"\n✓ Saved to {output_file}")

In [ ]:
from scipy.stats import mannwhitneyu

# Boxplot comparing %Fail1 (pooled 20Hz+50Hz) vs %Fail2 (20Hz) vs %Fail2 (50Hz) at 2.5mM Ca

# Extract data
fail1_20hz = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_50hz = PCA_Data_50Hz_2_5_Ca['%Fail1'].dropna()
fail1_combined = pd.concat([fail1_20hz, fail1_50hz], ignore_index=True)

fail2_20hz = PCA_Data_WT_Pooled['%Fail2'].dropna()
fail2_50hz = PCA_Data_50Hz_2_5_Ca['%Fail2'].dropna()

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Prepare data for boxplot
data = [fail1_combined.values, fail2_20hz.values, fail2_50hz.values]
positions = [0, 1, 2]
colors = ['gray', 'steelblue', 'orange']
labels = [f'%Fail1\n20Hz+50Hz\n(n={len(fail1_combined)})', 
          f'%Fail2\n20Hz\n(n={len(fail2_20hz)})', 
          f'%Fail2\n50Hz\n(n={len(fail2_50hz)})']

# Create boxplots
bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add strip points
for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=20, alpha=0.4, edgecolors='black', linewidths=0.3)

# Statistical tests
_, p_fail1_vs_fail2_20 = mannwhitneyu(fail1_combined, fail2_20hz, alternative='two-sided')
_, p_fail1_vs_fail2_50 = mannwhitneyu(fail1_combined, fail2_50hz, alternative='two-sided')
_, p_fail2_20_vs_50 = mannwhitneyu(fail2_20hz, fail2_50hz, alternative='two-sided')

def get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# Formatting
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Failure Rate (%)', fontsize=11)
ax.set_title('%Fail1 (pooled) vs %Fail2 at 2.5mM Ca', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
output_file = OUTPUT_DIR / "Fig5_fail1_pooled_vs_fail2_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== FAILURE RATE COMPARISON @ 2.5mM Ca ===")
print(f"%Fail1 (20Hz+50Hz): {fail1_combined.mean():.1f}% ± {fail1_combined.std():.1f}% (n={len(fail1_combined)})")
print(f"%Fail2 20Hz: {fail2_20hz.mean():.1f}% ± {fail2_20hz.std():.1f}% (n={len(fail2_20hz)})")
print(f"%Fail2 50Hz: {fail2_50hz.mean():.1f}% ± {fail2_50hz.std():.1f}% (n={len(fail2_50hz)})")
print(f"\nStatistics:")
print(f"  %Fail1 vs %Fail2 20Hz: p = {p_fail1_vs_fail2_20:.4g} ({get_stars(p_fail1_vs_fail2_20)})")
print(f"  %Fail1 vs %Fail2 50Hz: p = {p_fail1_vs_fail2_50:.4g} ({get_stars(p_fail1_vs_fail2_50)})")
print(f"  %Fail2 20Hz vs 50Hz: p = {p_fail2_20_vs_50:.4g} ({get_stars(p_fail2_20_vs_50)})")
print(f"\n✓ Saved to {output_file}")

In [ ]:
# Extract PPR2/1 and PPR3/1 data for 20Hz WT and 50Hz WT at 2.5mM Ca
ppr2_1_20hz = PCA_Data_WT_Pooled['PPR2/1'].dropna()
ppr2_1_50hz = PCA_Data_50Hz_2_5_Ca['PPR2/1'].dropna()

ppr3_1_20hz = PCA_Data_WT_Pooled['PPR3/1'].dropna()
ppr3_1_50hz = PCA_Data_50Hz_2_5_Ca['PPR3/1'].dropna()

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

def plot_boxplot(ax, data_20hz, data_50hz, title, ylabel):
    """Helper function to plot boxplot on given axis."""
    data = [data_20hz.values, data_50hz.values]
    positions = [0, 1]
    colors = ['steelblue', 'orange']
    labels = [f'{ylabel} 20Hz\n(n={len(data_20hz)})', 
              f'{ylabel} 50Hz\n(n={len(data_50hz)})']
    
    bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add strip points
    for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
        x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
        ax.scatter(x_jitter, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)
    
    # Statistical test
    _, p_val = mannwhitneyu(data_20hz, data_50hz, alternative='two-sided')
    
    def get_stars(p):
        return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    
    # Add significance bar
    y_max = max([d.max() for d in data]) * 1.1
    ax.plot([0, 1], [y_max, y_max], 'k-', lw=1.5)
    ax.text(0.5, y_max * 1.02, f'{get_stars(p_val)} (p={p_val:.2g})', ha='center', fontsize=10)
    
    # Formatting
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('Ratio', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(top=y_max * 1.15)
    
    return p_val

# Plot PPR2/1
p_ppr2_1 = plot_boxplot(ax1, ppr2_1_20hz, ppr2_1_50hz, 
                        'PPR2/1 Comparison: 20Hz vs 50Hz @ 2.5mM Ca', 'PPR2/1')

# Plot PPR3/1
p_ppr3_1 = plot_boxplot(ax2, ppr3_1_20hz, ppr3_1_50hz, 
                        'PPR3/1 Comparison: 20Hz vs 50Hz @ 2.5mM Ca', 'PPR3/1')

plt.tight_layout()
output_file = OUTPUT_DIR / "Fig5_ppr2_1_ppr3_1_20Hz_vs_50Hz_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== PPR RATIO COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"\nPPR2/1:")
print(f"  20Hz: {ppr2_1_20hz.mean():.3f} ± {ppr2_1_20hz.std():.3f} (n={len(ppr2_1_20hz)})")
print(f"  50Hz: {ppr2_1_50hz.mean():.3f} ± {ppr2_1_50hz.std():.3f} (n={len(ppr2_1_50hz)})")
print(f"  Mann-Whitney p = {p_ppr2_1:.4g}")

print(f"\nPPR3/1:")
print(f"  20Hz: {ppr3_1_20hz.mean():.3f} ± {ppr3_1_20hz.std():.3f} (n={len(ppr3_1_20hz)})")
print(f"  50Hz: {ppr3_1_50hz.mean():.3f} ± {ppr3_1_50hz.std():.3f} (n={len(ppr3_1_50hz)})")
print(f"  Mann-Whitney p = {p_ppr3_1:.4g}")

print(f"\n✓ Saved to {output_file}")